<a href="https://colab.research.google.com/github/Jay-Youssef/recipe/blob/main/PET_Recipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title rPET__Recipe v3.0 (Standard / RecyClass / A1-only)_Special_FA_Weight

from datetime import datetime
import itertools, math, re, io

import numpy as np
import pandas as pd

from IPython.display import display, clear_output
from google.colab import files
import ipywidgets as widgets

# --- Excel writer dependency (robust in Colab) ---
try:
    import xlsxwriter  # noqa: F401
except Exception:
    !pip install -q xlsxwriter
    import xlsxwriter  # noqa: F401


# =========================================================
# UI widgets (ONLY Date, Product, Mode)
# =========================================================
date_picker = widgets.Text(
    value=datetime.today().date().isoformat(),
    description="Recipe Date:",
    style={"description_width": "initial"},
)

product_dd = widgets.Dropdown(
    options=[("C100", "C100"), ("C102", "C102"), ("C103", "C103")],
    value="C100",
    description="Product:",
    style={"description_width": "initial"},
)

mode_dd = widgets.Dropdown(
    options=[("Standard", "STANDARD"), ("RecyClass", "RECYCLASS"), ("A1 only", "A1_ONLY")],
    value="STANDARD",
    description="Production Mode:",
    style={"description_width": "initial"},
)


# =========================================================
# Product + Mode specs (single source of truth)
# =========================================================
PRODUCT_SPECS = {
    # product: (LB_target, LB_tol, b*_min, b*_max)
    "C100": (10.0, 3.0, -3.0,  0.0),
    "C103": ( 5.0, 3.0,  0.0,  1.0),
    "C102": (15.0, 3.0, -5.0, -3.0),
}

MODE_LSTAR_RANGE = {
    "STANDARD":  (68.0, 72.0),
    "RECYCLASS": (67.0, 70.0),
    "A1_ONLY":   (66.0, 69.0),
}

RESERVE_BC_SHEET_NAME_DESIRED = "Reserve und BC Buchungskontrolle"
EXCEL_SHEET_NAME_LIMIT = 31
RESERVE_BC_SHEET_NAME = RESERVE_BC_SHEET_NAME_DESIRED[:EXCEL_SHEET_NAME_LIMIT]

# L* -> LT mapping anchor:
# L*=72 -> LT=1.80, L*=68 -> LT=2.30 (linear)
def lt_from_Lstar(L):
    L = float(L)
    return 1.8 + (72.0 - L) * 0.125

def get_recipe_specs(product, mode):
    lb_target, lb_tol, bmin, bmax = PRODUCT_SPECS[product]

    Lmin, Lmax = MODE_LSTAR_RANGE[mode]
    lt_min = lt_from_Lstar(Lmax)
    lt_max = lt_from_Lstar(Lmin)
    lt_target = 0.5 * (lt_min + lt_max)
    lt_tol    = 0.5 * (lt_max - lt_min)

    return {
        "lb_target": float(lb_target),
        "lb_tol": float(lb_tol),
        "b_range": (float(bmin), float(bmax)),
        "L_range": (float(Lmin), float(Lmax)),
        "lt_target": float(lt_target),
        "lt_tol": float(lt_tol),
    }

# =========================================================
# Global constants / tunables
# =========================================================
MIN_TOTAL_SOFT = 4
MIN_TOTAL_HARD = 4
MAX_TOTAL      = 6.7

MIN_MIXES = 50
MIXES_PREF = 40
MAX_NONZERO_BINS = 5
EPS = 1e-9

# Solver lanes:
#   clear
#   clb_low        (2 .. <25)
#   clb_high       (25 .. <50)
#   lightgrey
#   highlight_use
STEP_CLEAR    = 0.5
STEP_CLB_LOW  = 0.5
STEP_CLB_HIGH = 0.5
STEP_LG       = 0.5
STEP_HL       = 0.050

VISIBLE_BB_LIMIT = 200.0

BD_WARN_LOW  = 220.0
BD_WARN_HIGH = 270.0

# ======= Lightblue / Darkblue physics tunables =======
DARKBLUE_FACTOR_BASE = 2.5
DARKBLUE_FACTOR_HL   = 2.5

# ======= FA BB tonnage correction for recipe table =======
FA_BB_TONNAGE       = 0.70
NORMAL_BB_TONNAGE   = 1.00
FA_LANE_THRESHOLD   = 0.50

# Caps for candidate enumeration (BB units)
UPPER_CAP = {
    "clear":          200.0,
    "clb_low":        200.0,
    "clb_high":       200.0,
    "lightgrey":      200.0,
    "highlight_use":  200.0,
}

# Artikels to exclude early
EXCLUDE_ARTIKELS = {
    1100138, 1101019, 1000047, 1000048,
    1000057, 1100031, 1001049
}

# Solver search grid bound
MAX_GUESS_PER_GROUP = 20

# ======= LG (Lightgrey) / darkness tunables =======
DARK_LT_THRESHOLD = 3.0
DARK_TEXT_TOKENS  = {"03", "04", "dunkel", "dark", "lg", "lightgrey", "grau"}

# ======= Legacy-style anchor tunables =======
ANCHOR_WIDTH_CLB_LOW  = 10.0
ANCHOR_WIDTH_CLB_HIGH = 10.0
ANCHOR_WIDTH_LG       = 14.0

CLB_LOW_MIN   = 2.0
CLB_LOW_MAX   = 25.0
CLB_LOW_STEP  = 2.0

CLB_HIGH_MIN  = 25.0
CLB_HIGH_MAX  = 50.0
CLB_HIGH_STEP = 2.0

# Mirroring the legacy LG style as a low-LB 10-wide band with stepped starts.
LG_MIN   = 0.0
LG_MAX   = 25.0
LG_STEP  = 2.0

# =========================================================
# RecyClass / A1 allowlists (single source of truth)
# =========================================================
RECYCLASS_REF = pd.DataFrame([
    {"Artikel": 1000018, "Recyclass_label": "Ohne"},
    {"Artikel": 1000020, "Recyclass_label": "Ohne"},
    {"Artikel": 1000049, "Recyclass_label": "Ohne"},
    {"Artikel": 1000127, "Recyclass_label": "RecyClass"},
    {"Artikel": 1000126, "Recyclass_label": "RecyClass"},
    {"Artikel": 1000125, "Recyclass_label": "RecyClass"},
    {"Artikel": 1000118, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1000119, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1000120, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1001049, "Recyclass_label": "Ohne"},
    {"Artikel": 1000047, "Recyclass_label": "Ohne"},
    {"Artikel": 1000021, "Recyclass_label": "Ohne"},
    {"Artikel": 1000022, "Recyclass_label": "Ohne"},
    {"Artikel": 1000130, "Recyclass_label": "Ohne"},
    {"Artikel": 1000132, "Recyclass_label": "Ohne"},
    {"Artikel": 1000122, "Recyclass_label": "Ohne"},
    {"Artikel": 1100101, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1100094, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1100106, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1101003, "Recyclass_label": "Ohne"},
])

RECYCLASS_REF = RECYCLASS_REF.copy()
RECYCLASS_REF["Artikel"] = pd.to_numeric(RECYCLASS_REF["Artikel"], errors="coerce").astype("Int64")
RECYCLASS_REF["Recyclass_label"] = RECYCLASS_REF["Recyclass_label"].astype(str).str.strip()

A1_ALLOWED = set(
    RECYCLASS_REF.loc[RECYCLASS_REF["Recyclass_label"].eq("RecyClass A1"), "Artikel"]
    .dropna().astype(int).tolist()
)

RECY_ALLOWED = set(
    RECYCLASS_REF.loc[RECYCLASS_REF["Recyclass_label"].isin(["RecyClass", "RecyClass A1"]), "Artikel"]
    .dropna().astype(int).tolist()
)

# =========================================================
# Runtime active specs
# =========================================================
PRODUCTION_MODE = "STANDARD"
PRODUCT_CODE    = "C100"
ACTIVE_SPECS    = get_recipe_specs(PRODUCT_CODE, PRODUCTION_MODE)

target_lb = float(ACTIVE_SPECS["lb_target"])
LB_TOL    = float(ACTIVE_SPECS["lb_tol"])
lt_target = float(ACTIVE_SPECS["lt_target"])
LT_TOL    = float(ACTIVE_SPECS["lt_tol"])

def apply_active_specs_from_widgets():
    global PRODUCTION_MODE, PRODUCT_CODE, ACTIVE_SPECS
    global target_lb, LB_TOL, lt_target, LT_TOL

    PRODUCT_CODE    = str(product_dd.value)
    PRODUCTION_MODE = str(mode_dd.value)

    ACTIVE_SPECS = get_recipe_specs(PRODUCT_CODE, PRODUCTION_MODE)

    target_lb = float(ACTIVE_SPECS["lb_target"])
    LB_TOL    = float(ACTIVE_SPECS["lb_tol"])
    lt_target = float(ACTIVE_SPECS["lt_target"])
    LT_TOL    = float(ACTIVE_SPECS["lt_tol"])

# =========================================================
# Utilities
# =========================================================
def find_ranges(packets):
    out = []
    grouped = {}
    pat = re.compile(r"^(.*-)(\d+)$")

    for pkt in packets:
        s = str(pkt)
        m = pat.match(s)
        if not m:
            grouped.setdefault(s, []).append(None)
            continue
        p, n = m.groups()
        grouped.setdefault(p, []).append(int(n))

    for p, nums in grouped.items():
        if nums == [None]:
            out.append(p.rstrip('-'))
            continue

        numeric_suffixes = [x for x in nums if isinstance(x, int)]
        if not numeric_suffixes:
            out.append(p.rstrip('-'))
            continue

        numeric_suffixes = sorted(set(numeric_suffixes))
        prefix_base = p.rstrip('-')
        start = end = numeric_suffixes[0]

        for x in numeric_suffixes[1:]:
            if x == end + 1:
                end = x
            else:
                if start == end:
                    out.append(f"{p}{start}")
                else:
                    out.append(f"{prefix_base} ({start} bis {end})")
                start = end = x

        if start == end:
            out.append(f"{p}{start}")
        else:
            out.append(f"{prefix_base} ({start} bis {end})")

    return out

def _to_numeric(series):
    s = series.astype(str).str.replace(',', '.', regex=False)
    s = s.str.replace(r'[^\d\.\-]+', '', regex=True)
    return pd.to_numeric(s, errors='coerce')

def compute_result_lb(picks, lb_means):
    tot = sum(picks.values())
    if tot <= EPS:
        return float('nan')
    return sum(picks.get(g, 0.0) * lb_means.get(g, 0.0) for g in picks) / tot

def compute_result_lt(picks, lt_means):
    tot = sum(picks.values())
    if tot <= EPS:
        return float('nan')

    acc, w = 0.0, 0.0
    for g, v in picks.items():
        lt = lt_means.get(g, None)
        if lt is None or (isinstance(lt, float) and np.isnan(lt)):
            continue
        acc += v * lt
        w += v
    return acc / w if w > 0 else float('nan')

def parse_bulk_density(x):
    if x is None:
        return np.nan

    s = str(x).strip()
    s = s.replace(",", ".")
    s = re.sub(r"[^0-9.\-eE+]", "", s)

    try:
        v = float(s)
    except Exception:
        return np.nan

    if np.isnan(v):
        return np.nan

    if v < 5.0:
        return v * 1000.0
    return v

def compute_result_bd(picks, bd_means):
    tot = sum(picks.values())
    if tot <= EPS:
        return float("nan")

    acc, w = 0.0, 0.0
    for g, v in picks.items():
        bd = bd_means.get(g, None)
        if bd is None or (isinstance(bd, float) and np.isnan(bd)):
            continue
        acc += v * bd
        w += v
    return acc / w if w > 0 else float("nan")

def compute_full_mixes(picks, stock_caps):
    limits = []
    for g, v in picks.items():
        if v <= EPS:
            continue
        avail = stock_caps.get(g, 0.0)
        if avail <= EPS:
            return 0
        limits.append(math.floor(avail / v))
    return int(min(limits)) if limits else 0

def bb_to_display(amount_bb, group_key):
    if amount_bb is None or amount_bb <= EPS:
        return None

    if group_key == 'highlight_use':
        kg = round(amount_bb * 1000)
        return f"{kg} kg"

    x2 = amount_bb * 2.0
    if abs(x2 - round(x2)) < 1e-9:
        if abs(amount_bb - round(amount_bb)) < 1e-9:
            if int(round(amount_bb)) == 1:
                return "1 BB"
            return f"{int(round(amount_bb))} BB"
        return f"{amount_bb:.1f} BB"

    kg = round(amount_bb * 1000)
    return f"{kg} kg"

# =========================================================
# Legacy anchor helpers
# =========================================================
def _build_legacy_anchor_weights(df_lane):
    if df_lane.empty:
        return df_lane.copy()

    tmp = df_lane.copy()

    tmp["__w_mass"] = pd.to_numeric(tmp["bb"], errors="coerce").fillna(0.0).astype(float)
    if float(tmp["__w_mass"].sum()) <= EPS:
        tmp["__w_mass"] = pd.to_numeric(tmp["Menge"], errors="coerce").fillna(0.0).astype(float)

    tmp["__w_lt"] = pd.to_numeric(tmp["Menge"], errors="coerce").fillna(0.0).astype(float)
    if float(tmp["__w_lt"].sum()) <= EPS:
        tmp["__w_lt"] = tmp["__w_mass"]

    tmp["lb_pct"] = pd.to_numeric(tmp["lb_pct"], errors="coerce")
    tmp["lt_num"] = pd.to_numeric(tmp.get("lt_num", np.nan), errors="coerce")
    return tmp

def _scan_legacy_anchor_window(df_lane, start_min, end_max, width, step):
    if df_lane.empty:
        return None

    tmp = _build_legacy_anchor_weights(df_lane)
    best = None

    starts = []
    s = float(start_min)
    while s + float(width) <= float(end_max) + 1e-9:
        starts.append(round(s, 6))
        s += float(step)

    for start in starts:
        end = round(start + float(width), 6)
        mask = tmp["lb_pct"].ge(float(start)) & tmp["lb_pct"].lt(float(end))
        cand = tmp.loc[mask].copy()
        if cand.empty:
            continue

        massW = float(pd.to_numeric(cand["__w_mass"], errors="coerce").fillna(0.0).sum())
        if massW <= EPS:
            continue

        lt_vals = pd.to_numeric(cand["lt_num"], errors="coerce")
        lt_w = pd.to_numeric(cand["__w_lt"], errors="coerce").fillna(0.0).astype(float)
        lt_mask = lt_vals.notna() & lt_w.gt(EPS)

        if bool(lt_mask.any()):
            lt_mean = float((lt_vals[lt_mask].astype(float) * lt_w[lt_mask]).sum() / max(float(lt_w[lt_mask].sum()), EPS))
            lt_err = abs(float(lt_mean) - float(lt_target))
        else:
            lt_err = float("inf")

        cand_key = (massW, -lt_err, -start, -end)
        if best is None or cand_key > best["cmp"]:
            best = {
                "start": float(start),
                "end": float(end),
                "massW": float(massW),
                "lt_err": float(lt_err),
                "cmp": cand_key,
            }

    return None if best is None else {k: v for k, v in best.items() if k != "cmp"}

def _choose_clb_low_anchor_window(df_clb_low):
    return _scan_legacy_anchor_window(
        df_clb_low,
        start_min=CLB_LOW_MIN,
        end_max=CLB_LOW_MAX,
        width=ANCHOR_WIDTH_CLB_LOW,
        step=CLB_LOW_STEP,
    )

def _choose_clb_high_anchor_window(df_clb_high):
    return _scan_legacy_anchor_window(
        df_clb_high,
        start_min=CLB_HIGH_MIN,
        end_max=CLB_HIGH_MAX,
        width=ANCHOR_WIDTH_CLB_HIGH,
        step=CLB_HIGH_STEP,
    )

def _choose_lg_anchor_window(df_lg):
    return _scan_legacy_anchor_window(
        df_lg,
        start_min=LG_MIN,
        end_max=LG_MAX,
        width=ANCHOR_WIDTH_LG,
        step=LG_STEP,
    )

def _append_reason_col(series, reason):
    s = series.fillna("").astype(str)
    return np.where(s.eq(""), str(reason), s + "|" + str(reason))

# =========================================================
# Stock stats
# =========================================================
def compute_group_stats_for_solver(df):
    grp = df.groupby("group", as_index=False).agg(
        bb=("bb", "sum"),
        wlb=("lb_pct", lambda s: (df.loc[s.index, "bb"] * s).sum() / max(df.loc[s.index, "bb"].sum(), 1e-9)),
        wbd=("AV_BulkDensity",
             lambda s: (df.loc[s.index, "Menge"] * df.loc[s.index, "AV_BulkDensity"]).sum() /
                       max(df.loc[s.index, "Menge"].sum(), 1e-9)
             if not df.loc[s.index, "AV_BulkDensity"].isna().all() else np.nan),
        wlt=("AV_Lightness",
             lambda s: (df.loc[s.index, "Menge"] * df.loc[s.index, "AV_Lightness"]).sum() /
                       max(df.loc[s.index, "Menge"].sum(), 1e-9)
             if not df.loc[s.index, "AV_Lightness"].isna().all() else np.nan),
    )

    present = {r["group"]: float(r["bb"]) for _, r in grp.iterrows() if r["bb"] > EPS}
    lb_means = {r["group"]: float(r["wlb"]) for _, r in grp.iterrows()}
    bd_means = {r["group"]: (None if pd.isna(r["wbd"]) else float(r["wbd"])) for _, r in grp.iterrows()}
    lt_means = {r["group"]: (None if pd.isna(r["wlt"]) else float(r["wlt"])) for _, r in grp.iterrows()}

    base_no_hl = sum(v for g, v in present.items() if g != "highlight_use")
    shares_no_hl = {g: (present[g] / base_no_hl if base_no_hl > 0 else 0.0)
                    for g in present if g != "highlight_use"}

    return (present, lb_means, shares_no_hl, bd_means, lt_means)

def _lane_bb_tonnage_map(df_eff):
    out = {}
    for g in ["clear", "clb_low", "clb_high", "lightgrey"]:
        lane = df_eff[df_eff["group"].eq(g)].copy()
        if lane.empty:
            out[g] = NORMAL_BB_TONNAGE
            continue

        bb_all = pd.to_numeric(lane["bb"], errors="coerce").fillna(0.0).astype(float)
        fa_mask = lane["__is_A1_by_FA"].fillna(False).astype(bool) if "__is_A1_by_FA" in lane.columns else pd.Series(False, index=lane.index)

        bb_total = float(bb_all.sum())
        bb_fa = float(bb_all[fa_mask].sum())
        fa_share = (bb_fa / bb_total) if bb_total > EPS else 0.0

        out[g] = FA_BB_TONNAGE if fa_share >= FA_LANE_THRESHOLD else NORMAL_BB_TONNAGE

    out["highlight_use"] = 1.0
    return out

# =========================================================
# Dynamic stock assessment & min batch size logic
# =========================================================
def get_effective_min_total(stock_caps, demand_mixes_floor=10):
    return MIN_TOTAL_HARD

# =========================================================
# Candidate enumeration helpers
# =========================================================
def generate_grid_values(group_name, step, cap_limit, cap_stock, max_guess):
    vals = [0.0]
    k = 1
    while True:
        v = round(k * step, 6)
        if v > cap_limit + 1e-9:
            break
        if v > cap_stock + 1e-9:
            break
        vals.append(v)
        k += 1
        if len(vals) >= max_guess + 1:
            break
    return vals

def enumerate_candidates_base(groups_present, uppers, max_guess_per_group, min_total_dynamic):
    keys_core = [
        g for g in ["clear", "clb_low", "clb_high", "lightgrey", "highlight_use"]
        if groups_present.get(g, 0.0) > 0.0
    ]
    if not keys_core:
        return []

    step_map = {
        "clear": STEP_CLEAR,
        "clb_low": STEP_CLB_LOW,
        "clb_high": STEP_CLB_HIGH,
        "lightgrey": STEP_LG,
        "highlight_use": STEP_HL,
    }
    grids = {
        g: generate_grid_values(
            g,
            step_map[g],
            uppers.get(g, float("inf")),
            groups_present.get(g, 0.0),
            max_guess_per_group
        )
        for g in keys_core
    }

    out = []
    for tpl in itertools.product(*[grids[g] for g in keys_core]):
        pick = dict(zip(keys_core, tpl))
        total_bb = sum(pick.values())

        if total_bb < min_total_dynamic - 1e-9 or total_bb > MAX_TOTAL + 1e-9:
            continue

        nz_nonhl = sum(1 for gg, vv in pick.items() if gg != "highlight_use" and vv > EPS)
        if nz_nonhl > MAX_NONZERO_BINS:
            continue

        out.append(pick)

    return out

# =========================================================
# Highlight tuning / feasibility
# =========================================================
def nudge_highlight_for_lb(target_lb_local, pick, lb_means, caps, max_iter=200):
    cur = dict(pick)
    cur.setdefault("highlight_use", 0.0)

    for _ in range(max_iter):
        lb_now = compute_result_lb(cur, lb_means)
        if np.isnan(lb_now):
            break

        diff = lb_now - float(target_lb_local)
        if abs(diff) <= float(LB_TOL) + 1e-9:
            break

        if diff > 0:
            if cur["highlight_use"] > EPS:
                cur["highlight_use"] = round(max(0.0, cur["highlight_use"] - STEP_HL), 6)
            else:
                break
        else:
            new_hl = cur["highlight_use"] + STEP_HL
            if new_hl <= caps.get("highlight_use", float("inf")) + 1e-9:
                cur["highlight_use"] = round(new_hl, 6)
            else:
                break

        total_now = sum(cur.values())
        if total_now > MAX_TOTAL + 1e-9:
            for g_try in ["clear", "clb_low", "clb_high", "lightgrey"]:
                if cur.get(g_try, 0.0) >= 0.5 - 1e-9:
                    cur[g_try] = round(cur[g_try] - 0.5, 6)
                    if cur[g_try] < EPS:
                        cur[g_try] = 0.0
                    break

    return cur

def feasible_sequence(base_pick):
    tuned = nudge_highlight_for_lb(target_lb, base_pick, group_lb_means, stock_caps)

    total_bb = sum(tuned.values())
    if total_bb < effective_min_total - 1e-9 or total_bb > MAX_TOTAL + 1e-9:
        return None

    nz_nonhl = sum(1 for gg, vv in tuned.items() if gg != "highlight_use" and vv > EPS)
    if nz_nonhl > MAX_NONZERO_BINS:
        return None

    lb_final = compute_result_lb(tuned, group_lb_means)
    if np.isnan(lb_final) or abs(lb_final - float(target_lb)) > float(LB_TOL) + 1e-9:
        return None

    lt_final = compute_result_lt(tuned, group_lt_means)
    if lt_final is None or (isinstance(lt_final, float) and np.isnan(lt_final)):
        return None
    if abs(float(lt_final) - float(lt_target)) > float(LT_TOL) + 1e-9:
        return None

    bd_final = compute_result_bd(tuned, group_bd_means)

    mixes_final = compute_full_mixes(tuned, stock_caps)
    if int(mixes_final) < int(MIN_MIXES):
        return None

    return {
        "picks": tuned,
        "mixes": mixes_final,
        "lb": lb_final,
        "bd": bd_final,
        "lt": float(lt_final),
        "total_bb": total_bb
    }

def build_scored_candidates_with_priority():
    base_list = enumerate_candidates_base(groups_present, upper_cap_local, MAX_GUESS_PER_GROUP, effective_min_total)

    try:
        base_no_clear = sum(1 for p in base_list if float(p.get("clear", 0.0)) <= EPS)
        base_with_clear = sum(1 for p in base_list if float(p.get("clear", 0.0)) > EPS)
        print(
            "DEBUG build_scored:",
            f"base_total={len(base_list)}",
            f"base_no_clear={base_no_clear}",
            f"base_with_clear={base_with_clear}"
        )
    except Exception:
        pass

    scored = []
    for base in base_list:
        cand = feasible_sequence(base)
        if cand is not None:
            scored.append(cand)

    try:
        scored_no_clear = sum(1 for r in scored if float(r.get("picks", {}).get("clear", 0.0)) <= EPS)
        scored_with_clear = sum(1 for r in scored if float(r.get("picks", {}).get("clear", 0.0)) > EPS)
        print(
            "DEBUG build_scored:",
            f"feasible_total={len(scored)}",
            f"feasible_no_clear={scored_no_clear}",
            f"feasible_with_clear={scored_with_clear}"
        )
    except Exception:
        pass

    return scored

# =========================================================
# Choose best candidate
# =========================================================
def choose_best_candidate(scored):
    if not scored:
        return None

    df = pd.DataFrame(scored).copy()
    df["lb_err"] = (df["lb"] - float(target_lb)).abs()
    df["lt_err"] = (df["lt"] - float(lt_target)).abs()

    df = df[df["lb_err"] <= float(LB_TOL) + 1e-9].copy()
    if df.empty:
        return None

    df = df[df["lt"].notna()].copy()
    if df.empty:
        return None
    df = df[df["lt_err"] <= float(LT_TOL) + 1e-9].copy()
    if df.empty:
        return None

    df = df[df["mixes"].astype(int) >= int(MIN_MIXES)].copy()
    if df.empty:
        return None

    df["total_bb"] = df["total_bb"].astype(float)
    df = df[(df["total_bb"] >= MIN_TOTAL_SOFT - 1e-9) & (df["total_bb"] <= MAX_TOTAL + 1e-9)].copy()
    if df.empty:
        return None
    try:
        df["uses_clear"] = df["picks"].apply(lambda p: float(p.get("clear", 0.0)) > EPS)
        print(
            "DEBUG choose_best_candidate:",
            f"candidates={len(df)}",
            f"no_clear={int((~df['uses_clear']).sum())}",
            f"with_clear={int(df['uses_clear'].sum())}"
        )
    except Exception:
        pass

    NONHL_LANES = ["clear", "clb_low", "clb_high", "lightgrey"]

    df_gate_base = df.copy()
    df = df_gate_base

    def _is_integer_bb(v):
        v = float(v)
        if v <= EPS:
            return True
        return abs(v - round(v)) < 1e-9

    def _all_nonhl_integer(picks):
        for g in NONHL_LANES:
            if not _is_integer_bb(picks.get(g, 0.0)):
                return False
        return True

    def _half_count_nonhl(picks):
        c = 0
        for g in NONHL_LANES:
            v = float(picks.get(g, 0.0))
            if v <= EPS:
                continue
            if abs(v - round(v)) < 1e-9:
                continue
            if abs(v * 2.0 - round(v * 2.0)) < 1e-9:
                c += 1
            else:
                c += 10
        return c

    df["all_integer"] = df["picks"].apply(_all_nonhl_integer)
    df["half_count"] = df["picks"].apply(_half_count_nonhl)

    stock_clear = float(stock_shares_no_hl_all.get("clear", 0.0))
    stock_clb_low = float(stock_shares_no_hl_all.get("clb_low", 0.0))
    stock_clb_high = float(stock_shares_no_hl_all.get("clb_high", 0.0))
    stock_lg = float(stock_shares_no_hl_all.get("lightgrey", 0.0))

    def stock_balance_penalty(picks):
        non_hl_total = float(
            picks.get("clear", 0.0) +
            picks.get("clb_low", 0.0) +
            picks.get("clb_high", 0.0) +
            picks.get("lightgrey", 0.0)
        )
        if non_hl_total <= EPS:
            return 1e6
        cand_clear = float(picks.get("clear", 0.0)) / non_hl_total
        cand_clb_low = float(picks.get("clb_low", 0.0)) / non_hl_total
        cand_clb_high = float(picks.get("clb_high", 0.0)) / non_hl_total
        cand_lg = float(picks.get("lightgrey", 0.0)) / non_hl_total
        return (
            abs(cand_clear - stock_clear) +
            abs(cand_clb_low - stock_clb_low) +
            abs(cand_clb_high - stock_clb_high) +
            abs(cand_lg - stock_lg)
        )

    def bottleneck_penalty(mixes_val):
        m = float(mixes_val)
        if m >= float(MIXES_PREF):
            return 0.0
        return ((float(MIXES_PREF) - m) / max(float(MIXES_PREF), 1.0)) ** 2

    def _is_half_step(v):
        return (v > EPS) and (abs(v * 2.0 - round(v * 2.0)) < 1e-9) and (abs(v - round(v)) > 1e-9)

    def fractional_penalty(picks):
        if PRODUCTION_MODE == "A1_ONLY":
            return 0.0

        half_count = 0
        half_in_lg = 0

        for g in ["clear", "clb_low", "clb_high", "lightgrey"]:
            v = float(picks.get(g, 0.0))
            if v <= EPS:
                continue

            if abs(v - round(v)) < 1e-9:
                continue

            if _is_half_step(v):
                half_count += 1
                if g == "lightgrey":
                    half_in_lg += 1
            else:
                return 1e6

        base = 500.0 * max(0, half_count - 1) + 50.0 * half_count
        base += 800.0 * half_in_lg
        return base

    df["fractional_penalty"] = df["picks"].apply(fractional_penalty)
    df["stock_balance_penalty"] = df["picks"].apply(stock_balance_penalty)
    df["bottleneck_penalty"] = df["mixes"].apply(bottleneck_penalty)

    df_sorted = df.sort_values(
        [
            "fractional_penalty",
            "stock_balance_penalty",
            "lb_err",
            "lt_err",
            "all_integer",
            "half_count",
            "total_bb",
            "mixes",
            "bottleneck_penalty",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            True,
            False,
            False,
            True,
        ],
        kind="stable"
    )
    if df_sorted.empty:
        return None

    pick = df_sorted.iloc[0]
    return {
        "picks": dict(pick["picks"]),
        "result_lb": float(pick["lb"]),
        "result_bd": (float(pick["bd"]) if not (pd.isna(pick["bd"])) else float("nan")),
        "result_lt": float(pick["lt"]),
        "mixes": int(pick["mixes"]),
        "single_lane": True,
        "lt_target": float(lt_target),
        "lt_tol": float(LT_TOL),
    }

# =========================================================
# Final display table helpers (late-stage only)
# =========================================================
def prepare_final_recipe_display(candidate_recipe_display):
    cols = [
        "Material_Vis", "Big Bags", "Percentage", "Artikel", "Lagerplatz",
        "AV_LB%", "AV_Lightness", "Quantity", "Quality", "Charge/ Paket", "Pick_Seq"
    ]

    if candidate_recipe_display is None or candidate_recipe_display.empty:
        out = pd.DataFrame(columns=cols + ["LB_is_overflow", "Raw_Charge", "__shown_packet_count"])
        return out

    cr = candidate_recipe_display.copy()

    if "Pick_Seq" not in cr.columns:
        cr["Pick_Seq"] = cr.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1

    if "Quantity" not in cr.columns:
        cr["Quantity"] = cr.get("Tonnage_Allocated", cr.get("Menge", 0.0))

    if "Raw_Charge" not in cr.columns:
        cr["Raw_Charge"] = [[] for _ in range(len(cr))]
    else:
        def _coerce_raw_charge(x):
            if isinstance(x, (list, tuple, set)):
                return [str(v).strip() for v in x if str(v).strip() not in ("", "nan", "NaN", "None")]
            if x is None or (isinstance(x, float) and np.isnan(x)):
                return []
            s = str(x).strip()
            if s in ("", "nan", "NaN", "None"):
                return []
            return [s]
        cr["Raw_Charge"] = cr["Raw_Charge"].apply(_coerce_raw_charge)

    for c in cols:
        if c not in cr.columns:
            cr[c] = 0.0 if c in ("Big Bags", "AV_LB%", "AV_Lightness", "Quantity", "Pick_Seq", "Percentage") else ""
        else:
            if pd.api.types.is_categorical_dtype(cr[c]):
                cr[c] = cr[c].astype(str)

    df_merge = cr[cols].reset_index(drop=True).copy()
    df_merge["LB_is_overflow"] = cr["LB_is_overflow"].values if "LB_is_overflow" in cr.columns else False
    df_merge["Raw_Charge"] = cr["Raw_Charge"].values

    def _reorder_block(block):
        overflow = block[block["LB_is_overflow"]].copy()
        core = block[~block["LB_is_overflow"]].copy()
        mat_name = str(block["Material_Vis"].iloc[0]) if "Material_Vis" in block.columns and len(block) > 0 else ""
        desc_order = (mat_name == "Clearlightblue High")
        if core.empty:
            return block.sort_values("AV_LB%", ascending=(not desc_order)) if "AV_LB%" in block.columns else block
        if "AV_LB%" in core.columns:
            core = core.sort_values("AV_LB%", ascending=(not desc_order))
        return pd.concat([overflow, core], axis=0)

    df_merge = (
        df_merge.groupby("Material_Vis", group_keys=False, sort=False, observed=False)
                .apply(_reorder_block)
                .reset_index(drop=True)
    )

    material_order = ["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"]

    if pd.api.types.is_categorical_dtype(df_merge["Material_Vis"]):
        df_merge["__mat_order"] = df_merge["Material_Vis"].cat.codes
    else:
        df_merge["__mat_order"] = df_merge["Material_Vis"].map({m: i for i, m in enumerate(material_order)}).fillna(999).astype(int)

    df_merge["__row_order"] = np.arange(len(df_merge), dtype=int)
    df_merge = (
        df_merge.sort_values(["__mat_order", "__row_order"], kind="stable")
                .drop(columns=["__mat_order", "__row_order"])
                .reset_index(drop=True)
    )

    df_merge["Pick_Seq"] = df_merge.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1
    df_merge["__shown_packet_count"] = df_merge["Raw_Charge"].apply(lambda x: len(x) if isinstance(x, (list, tuple, set)) else 0)

    return df_merge

def split_final_display_for_reserve_bc(df_final_display, shown_amount_threshold=5):
    if df_final_display is None or df_final_display.empty:
        empty = pd.DataFrame(columns=(df_final_display.columns.tolist() if df_final_display is not None else []))
        return empty.copy(), empty.copy()

    work = df_final_display.copy()
    shown_amt = pd.to_numeric(work.get("__shown_packet_count", 0), errors="coerce").fillna(0.0).astype(float)

    reserve_mask = shown_amt.gt(0.0) & shown_amt.lt(float(shown_amount_threshold))
    candidate_recipe_display_main = work.loc[~reserve_mask].copy().reset_index(drop=True)
    reserve_bc_display = work.loc[reserve_mask].copy().reset_index(drop=True)

    if not candidate_recipe_display_main.empty:
        candidate_recipe_display_main["Pick_Seq"] = (
            candidate_recipe_display_main.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1
        )

    if not reserve_bc_display.empty:
        reserve_bc_display["Pick_Seq"] = (
            reserve_bc_display.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1
        )

    return candidate_recipe_display_main, reserve_bc_display

def write_recipe_layout_sheet(
    ws,
    df_display,
    wb,
    recipe_date,
    choice,
    production_mode,
    header_stock,
    header_corr,
    hidden_mats=None,
    unused_mats_for_hide=None,
    apply_hidden_candidate_rules=True
):
    fmt_header_small = wb.add_format({"bold": False, "font_size": 9, "align": "left", "valign": "vcenter", "border": 1, "border_color": "#000000"})
    fmt_header = wb.add_format({"bold": True, "font_size": 14, "align": "left", "valign": "vcenter", "border": 1, "border_color": "#000000"})
    fmt_table_header = wb.add_format({"bold": True, "font_size": 14, "align": "center", "valign": "vcenter", "border": 1, "border_color": "#000000", "bg_color": "#DCE6F1"})
    fmt_section = wb.add_format({"border": 2, "border_color": "#000000", "bold": True, "font_size": 14, "align": "center", "valign": "vcenter"})
    fmt_center_light = wb.add_format({"border": 1, "border_color": "#D9D9D9", "font_size": 12, "align": "center", "valign": "vcenter"})
    fmt_packet_light = wb.add_format({"border": 1, "border_color": "#D9D9D9", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})
    fmt_bt_center = wb.add_format({"top": 2, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "center", "valign": "vcenter"})
    fmt_bm_center = wb.add_format({"top": 0, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "center", "valign": "vcenter"})
    fmt_bb_center = wb.add_format({"top": 0, "bottom": 2, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "center", "valign": "vcenter"})
    fmt_bt_packet = wb.add_format({"top": 2, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})
    fmt_bm_packet = wb.add_format({"top": 0, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})
    fmt_bb_packet = wb.add_format({"top": 0, "bottom": 2, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})

    ws.set_column(0, 0, 25)
    ws.set_column(1, 1, 12)
    ws.set_column(2, 2, None, fmt_center_light, {"hidden": True})
    ws.set_column(3, 3, None, fmt_center_light, {"hidden": True})
    ws.set_column(4, 4, 18, fmt_center_light)
    ws.set_column(5, 8, None, fmt_center_light, {"hidden": True})
    ws.set_column(9, 9, 90, fmt_packet_light)
    ws.set_column(10, 10, 10, fmt_center_light)

    ws.write(0, 0, header_stock, fmt_header_small)
    ws.write(1, 0, header_corr, fmt_header)

    table_header_row = 2
    cols = ["Material_Vis", "Big Bags", "Percentage", "Artikel", "Lagerplatz", "AV_LB%", "AV_Lightness", "Quantity", "Quality", "Charge/ Paket", "Pick_Seq"]
    for j, col_ in enumerate(cols):
        hdr = f"{col_} ({recipe_date.isoformat()}, {choice}, {production_mode})" if col_ == "Charge/ Paket" else col_
        ws.write(table_header_row, j, hdr, fmt_table_header)

    if df_display is None or df_display.empty:
        ws.write(table_header_row + 1, 0, "No rows available for this sheet.", fmt_header)
        return

    df_merge = df_display.copy().reset_index(drop=True)

    for c in cols:
        if c not in df_merge.columns:
            df_merge[c] = 0.0 if c in ("Big Bags", "AV_LB%", "AV_Lightness", "Quantity", "Pick_Seq", "Percentage") else ""

    merges = []
    curr_mat, start, end, prev_bb, prev_perc = None, None, None, None, None
    for i, row in df_merge.iterrows():
        mat = row["Material_Vis"]
        r = table_header_row + 1 + i
        if mat != curr_mat:
            if curr_mat is not None:
                merges.append((start, end, curr_mat, prev_bb, prev_perc))
            curr_mat = mat
            start = r
            prev_bb = row["Big Bags"]
            prev_perc = row["Percentage"]
        end = r
    if curr_mat is not None:
        merges.append((start, end, curr_mat, prev_bb, prev_perc))

    mat_has_non_overflow = {}
    if "LB_is_overflow" in df_merge.columns and not df_merge.empty:
        mat_has_non_overflow = df_merge.groupby("Material_Vis", observed=False)["LB_is_overflow"].apply(lambda s: (~s).any()).to_dict()

    for s, e, mat, bb, perc in merges:
        if s < e:
            ws.merge_range(s, 0, e, 0, mat, fmt_section)
            ws.merge_range(s, 1, e, 1, bb, fmt_section)
            ws.merge_range(s, 2, e, 2, perc, fmt_section)
        else:
            ws.write(s, 0, mat, fmt_section)
            ws.write(s, 1, bb, fmt_section)
            ws.write(s, 2, perc, fmt_section)

    for i, row in df_merge.iterrows():
        r = table_header_row + 1 + i
        mat = row["Material_Vis"]
        s, e, _, _, _ = next((b for b in merges if b[0] <= r <= b[1]), (r, r, mat, row.get("Big Bags", None), row.get("Percentage", None)))
        if s == e:
            f4 = f8 = f9 = fmt_section
        else:
            if r == s:
                f4, f8, f9 = fmt_bt_center, fmt_bt_packet, fmt_bt_center
            elif r == e:
                f4, f8, f9 = fmt_bb_center, fmt_bb_packet, fmt_bb_center
            else:
                f4, f8, f9 = fmt_bm_center, fmt_bm_packet, fmt_bm_center

        ws.write(r, 3, row["Artikel"], fmt_center_light)
        ws.write(r, 4, row["Lagerplatz"], f4)
        ws.write(r, 5, row["AV_LB%"], fmt_center_light)
        ws.write(r, 6, row["AV_Lightness"], fmt_center_light)
        ws.write(r, 7, row["Quantity"], fmt_center_light)
        ws.write(r, 8, row["Quality"], fmt_center_light)
        ws.write(r, 9, row["Charge/ Paket"], f8)
        ws.write(r, 10, int(row["Pick_Seq"]) if pd.notna(row["Pick_Seq"]) else "", f9)

        if apply_hidden_candidate_rules:
            if bool(row.get("LB_is_overflow", False)) and mat_has_non_overflow.get(mat, False):
                ws.set_row(r, None, None, {"hidden": True})

    if apply_hidden_candidate_rules:
        mats_to_hide = set(hidden_mats or set()) | set(unused_mats_for_hide or set())
        for s, e, mat, _, _ in merges:
            if str(mat) in mats_to_hide:
                for rr in range(s, e + 1):
                    ws.set_row(rr, None, None, {"hidden": True})

# =========================================================
# MAIN PIPELINE
# =========================================================
def generate_recipe():
    global solver_bins, groups_present, group_lb_means, group_bd_means, group_lt_means
    global stock_caps, stock_shares_no_hl_all
    global upper_cap_local, effective_min_total, PRODUCTION_MODE
    global target_lb, LB_TOL, lt_target, LT_TOL
    global recipe_date, product, candidate_recipe

    apply_active_specs_from_widgets()

    choice = str(product_dd.value)
    PRODUCTION_MODE = str(mode_dd.value)
    recipe_date = datetime.strptime(str(date_picker.value), "%Y-%m-%d").date()

    specs = get_recipe_specs(choice, PRODUCTION_MODE)
    target_lb = float(specs["lb_target"])
    LB_TOL    = float(specs["lb_tol"])
    lt_target = float(specs["lt_target"])
    LT_TOL    = float(specs["lt_tol"])

    print(f"Selected date: {recipe_date.isoformat()}")
    print(f"Selected product: {choice} → LB {target_lb:.1f}±{LB_TOL:.1f}")
    print(f"Selected mode: {PRODUCTION_MODE} → LT {lt_target:.3f}±{LT_TOL:.3f}  (from L* {specs['L_range'][0]}–{specs['L_range'][1]})")

    uploaded = None
    try:
        uploaded = files.upload()
    except TypeError:
        print("File upload cancelled or failed.")
        return

    if not uploaded:
        print("No file uploaded; aborting.")
        return

    file_name = next(iter(uploaded))
    raw = pd.read_excel(file_name, sheet_name=0)

    col_map = {c: c for c in raw.columns}
    known_map = {
        "Paket/Chargennummer": "Charge/ Paket",
        "Menge [to}": "Menge",
        "Schüttdichte 1": "Schüttgewicht",
        "PET lightblue (hellblau)": "Lightblue",
        "PET Blue_Dark": "Darkblue",
        "Helligkeit": "Helligkeit",
        "Lagerplatz": "Lagerplatz",
        "Artikel": "Artikel",
        "Artikel ": "Artikel",
        "Quality": "Quality",
        "Quality ": "Quality",
        "Beschreibung": "Beschreibung",
        "Beschreibung ": "Beschreibung",
        "Qualität": "Quality",
        "BlueDark": "Darkblue",
        "Schüttgewicht": "Schüttgewicht",
        "Charge/ Paket": "Charge/ Paket",
        "Menge": "Menge",
        "Lightblue": "Lightblue",
        "Darkblue": "Darkblue",
    }
    for c in list(raw.columns):
        if c in known_map:
            col_map[c] = known_map[c]
    raw = raw.rename(columns=col_map)

    for req in ['Artikel', 'Beschreibung', 'Charge/ Paket', 'Menge', 'Lagerplatz', 'Lightblue', 'Darkblue']:
        if req not in raw.columns:
            raw[req] = None

    raw['Artikel_clean'] = pd.to_numeric(raw['Artikel'], errors='coerce')
    if raw['Artikel_clean'].isna().any():
        mask = raw['Artikel_clean'].isna()
        raw.loc[mask, 'Artikel_clean'] = np.arange(1000000, 1000000 + mask.sum())
    raw['Artikel'] = raw['Artikel_clean'].astype(int)
    raw.drop(columns=['Artikel_clean'], inplace=True)

    raw = raw.reset_index(drop=True).copy()
    raw["Row_ID"] = np.arange(1, len(raw) + 1, dtype=int)

    raw = raw[~raw["Artikel"].isin(EXCLUDE_ARTIKELS)].copy()

    def _has_fa_packet(cell):
        if cell is None or (isinstance(cell, float) and np.isnan(cell)):
            return False
        if isinstance(cell, (list, tuple, set)):
            items = cell
        elif isinstance(cell, str):
            items = [x.strip() for x in cell.split(",")] if "," in cell else [cell.strip()]
        else:
            items = [str(cell).strip()]
        for x in items:
            t = str(x).strip().upper()
            if not t or t in ("NAN", "NONE"):
                continue
            if t.startswith("FA"):
                return True
        return False

    raw["__is_A1_by_FA"] = raw["Charge/ Paket"].apply(_has_fa_packet)

    mode_key = str(PRODUCTION_MODE)
    if mode_key == "A1_ONLY":
        if not A1_ALLOWED:
            raise ValueError("A1_ALLOWED is empty. Refusing to run.")
        raw = raw[(raw["Artikel"].isin(A1_ALLOWED)) | (raw["__is_A1_by_FA"])].copy()
    elif mode_key == "RECYCLASS":
        if not RECY_ALLOWED:
            raise ValueError("RECY_ALLOWED is empty. Refusing to run.")
        raw = raw[(raw["Artikel"].isin(RECY_ALLOWED)) | (raw["__is_A1_by_FA"])].copy()
    elif mode_key == "STANDARD":
        pass
    else:
        raise ValueError(f"Unknown PRODUCTION_MODE: {mode_key}")

    for c in ['Lightblue', 'Darkblue']:
        raw[c] = raw[c].astype(str).replace('[>]', '', regex=True).replace(',', '.', regex=True)
        raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0.0)

    if 'BlueDark' in raw.columns and 'Darkblue' in raw.columns:
        raw['Darkblue'] = raw['Darkblue'].where(raw['Darkblue'].notna(), raw['BlueDark'])
        raw.drop(columns=[c for c in ['BlueDark'] if c in raw.columns], inplace=True)

    lb = raw['Lightblue'].astype(float)
    db = raw['Darkblue'].astype(float)

    hl_mask   = lb >= 50.0
    base_mask = ~hl_mask

    LB_phys = lb.copy()
    LB_phys[base_mask] = lb[base_mask] + DARKBLUE_FACTOR_BASE * db[base_mask]
    LB_phys[hl_mask]   = lb[hl_mask]   + DARKBLUE_FACTOR_HL   * db[hl_mask]
    raw['LB_raw_physics'] = LB_phys

    raw['Lightblue_adjusted'] = np.ceil(raw['LB_raw_physics'])
    raw['Lightblue_adjusted_disp'] = raw['Lightblue_adjusted'].clip(upper=99)

    raw.dropna(subset=['Beschreibung', 'Menge', 'Lightblue_adjusted'], inplace=True)

    if 'Quality' not in raw.columns:
        def infer_q(s):
            t = str(s)
            if 'Q1' in t:
                return 'Q1'
            if 'Q2' in t:
                return 'Q2'
            return 'Q3'
        raw['Quality'] = raw['Beschreibung'].apply(infer_q)

    if 'Helligkeit' not in raw.columns:
        raw['Helligkeit'] = np.nan

    def parse_lightness_cell(v):
        if v is None:
            return None, np.nan
        if isinstance(v, float) and np.isnan(v):
            return None, np.nan

        s_raw = str(v).strip()
        if s_raw == "":
            return None, np.nan

        s = s_raw.lower().replace(",", ".")
        s = re.sub(r"\s+", " ", s).strip()

        if s in ("nan", "none", "-", "--"):
            return None, np.nan

        # text-first handling
        if "dunk" in s or "dark" in s:
            return "3.5", 3.5
        if "hell" in s:
            return "1.5", 1.5

        # explicit numeric forms like 01, 02, 03, 04, 05, 1, 2, 3, 4, 5, 1.5, 2.5 ...
        m = re.search(r'(?<!\d)(0?[1-5](?:\.[05])?)(?!\d)', s)
        if m:
            try:
                token = m.group(1)
                n = float(token)
                if 1.0 <= n <= 5.0:
                    if abs(n - round(n)) < 1e-9:
                        norm = str(int(round(n))).zfill(2)
                    elif abs(n * 2 - round(n * 2)) < 1e-9:
                        norm = f"{int(math.floor(n))}.5"
                    else:
                        norm = str(n)
                    return norm, n
            except Exception:
                pass

        # common L-forms like L3, L04
        m = re.search(r'\bl\s*0?([1-5])\b', s)
        if m:
            try:
                n = float(m.group(1))
                return str(int(n)).zfill(2), n
            except Exception:
                pass

        # "0" should not become a valid lightness class
        if s == "0":
            return None, np.nan

        return None, np.nan

    norms = raw['Helligkeit'].apply(parse_lightness_cell)
    raw['Helligkeit_norm'] = norms.apply(lambda x: x[0] if isinstance(x, tuple) else None)
    raw['Lightness_Num']   = norms.apply(lambda x: x[1] if isinstance(x, tuple) else np.nan)

    norms = raw['Helligkeit'].apply(parse_lightness_cell)
    raw['Helligkeit_norm'] = norms.apply(lambda x: x[0] if isinstance(x, tuple) else None)
    raw['Lightness_Num']   = norms.apply(lambda x: x[1] if isinstance(x, tuple) else np.nan)

    if 'Schüttgewicht' in raw.columns:
        raw['BulkDensity'] = raw['Schüttgewicht'].apply(parse_bulk_density)
    else:
        raw['BulkDensity'] = np.nan

    lb_disp = pd.to_numeric(raw["Lightblue_adjusted_disp"], errors="coerce").fillna(0.0).astype(float)

    def _lb_merge_guard(v):
        v = float(v)
        if v < 2.0:
            return "0 - 2"
        if v < 20.0:
            return "2 - 20"
        if v < 50.0:
            return "20 - 50"
        return "50 - 100"

    raw["Lightblue_Percentage"] = lb_disp.apply(_lb_merge_guard)
    raw["Color_Group"] = np.where(raw["Lightblue_Percentage"].eq("0 - 2"), "Clear", "Lightblue")

    def _material_name_from_bins(color_group, lb_pct_label):
        cg = str(color_group)
        lbv = str(lb_pct_label)
        if cg == "Clear" and lbv == "0 - 2":
            return "PET_Flakes_Clear_0_2"
        if lbv == "2 - 50":
            return "PET_Flakes_LB_2_50"
        return "PET_Flakes_LB_50_100"

    merged_rows = []
    grpcols = ["Color_Group", "Lightblue_Percentage", "Lagerplatz", "Artikel", "Beschreibung", "Helligkeit_norm"]
    for keys, grp in raw.groupby(grpcols, dropna=False):
        total = grp["Menge"].sum()
        if total <= 0:
            continue

        avg_lb_raw  = (grp["Menge"] * grp["LB_raw_physics"]).sum() / total
        avg_lb_disp = (grp["Menge"] * grp["Lightblue_adjusted_disp"]).sum() / total
        avg_db      = (grp["Menge"] * grp["Darkblue"]).sum() / total if "Darkblue" in grp.columns else 0.0

        bd_series = grp["BulkDensity"]
        lt_series = grp["Lightness_Num"]
        avg_bd = (grp["Menge"] * bd_series).sum() / total if not bd_series.isna().all() else np.nan
        avg_lt = (grp["Menge"] * lt_series).sum() / total if not lt_series.isna().all() else np.nan

        color_group = keys[0]
        lb_label    = keys[1]
        mat_name    = _material_name_from_bins(color_group, lb_label)

        merged_rows.append({
            "Color_Group": color_group,
            "Lightblue_Percentage": lb_label,
            "Material_Name": mat_name,
            "Lagerplatz": keys[2],
            "Artikel": int(keys[3]),
            "Beschreibung": keys[4],
            "Helligkeit_norm": keys[5],
            "Menge": total,
            "AV_LB_raw": avg_lb_raw,
            "AV_LB%": avg_lb_disp,
            "AV_DB%": avg_db,
            "AV_BulkDensity": avg_bd,
            "AV_Lightness": avg_lt,
            "Charge/ Paket": list(grp["Charge/ Paket"]),
            "Quality": grp["Quality"].mode().iat[0] if "Quality" in grp else None,
            "Mixing_Level": 1,
            "__is_A1_by_FA": bool(grp["__is_A1_by_FA"].any()) if "__is_A1_by_FA" in grp.columns else False,
        })

    combined_stock = pd.DataFrame(merged_rows)

    # =========================================================
    # 5-lane pipeline with legacy lane-local anchoring
    # =========================================================
    solver_rows = combined_stock.copy()

    solver_rows["bb"] = solver_rows["Charge/ Paket"].apply(
        lambda lst: len(lst) if isinstance(lst, (list, tuple, set)) else 0
    )

    solver_rows["lb_pct_raw"] = (
        solver_rows["AV_LB_raw"].astype(float)
        if "AV_LB_raw" in solver_rows.columns
        else solver_rows["AV_LB%"].astype(float)
    )
    solver_rows["lb_pct"] = solver_rows["lb_pct_raw"].astype(float)
    solver_rows["cluster"] = solver_rows["Lagerplatz"].astype(str) + "|" + solver_rows["Artikel"].astype(str)
    solver_rows["lt_num"] = pd.to_numeric(solver_rows.get("AV_Lightness", np.nan), errors="coerce")

    def _is_dark_row_any(row):
        lt = row.get("lt_num", np.nan)
        try:
            if lt is not None and not (isinstance(lt, float) and np.isnan(lt)):
                if float(lt) >= float(DARK_LT_THRESHOLD):
                    return True
        except Exception:
            pass

        h = row.get("Helligkeit_norm", None)
        h = "" if h is None else str(h).strip().lower()
        try:
            if h in set([str(x).strip().lower() for x in DARK_TEXT_TOKENS]):
                return True
        except Exception:
            pass

        if re.search(r"(^|[^0-9])0?[34]([^0-9]|$)", h):
            return True
        if re.search(r"\bl\s*0?[34]\b", h):
            return True

        txt = " ".join([
            "" if row.get("Beschreibung") is None else str(row.get("Beschreibung")),
            "" if row.get("Material_Name") is None else str(row.get("Material_Name")),
            "" if row.get("Material_Display") is None else str(row.get("Material_Display")),
        ]).lower()

        return any(tok in txt for tok in ("lightgrey", "lg", "grau", "dark", "dunkel", "l3", "l4"))

    solver_rows["__is_dark"] = solver_rows.apply(_is_dark_row_any, axis=1)

    def _lane_from_lb_and_dark(lb_val, is_dark):
        if lb_val is None or (isinstance(lb_val, float) and np.isnan(lb_val)):
            return None
        v = float(lb_val)
        if v >= 50.0:
            return "highlight_use"
        if bool(is_dark):
            return "lightgrey"
        if v < 2.0:
            return "clear"
        if v < 25.0:
            return "clb_low"
        return "clb_high"

    solver_rows["group"] = solver_rows.apply(
        lambda r: _lane_from_lb_and_dark(r.get("lb_pct", np.nan), r.get("__is_dark", False)),
        axis=1
    )
    solver_rows = solver_rows[solver_rows["group"].notna()].copy()

    try:
        bd_fix = pd.to_numeric(solver_rows.get("AV_BulkDensity", np.nan), errors="coerce")
    except Exception:
        bd_fix = pd.Series(np.nan, index=solver_rows.index)

    fa_series = solver_rows["__is_A1_by_FA"].fillna(False).astype(bool) if "__is_A1_by_FA" in solver_rows.columns else pd.Series(False, index=solver_rows.index)

    mask_fa_clear_lowbd = (
        solver_rows["group"].eq("clear") &
        fa_series &
        bd_fix.lt(250.0)
    )
    solver_rows.loc[mask_fa_clear_lowbd, "group"] = "clb_low"

    bad = solver_rows[(solver_rows["group"] == "clear") & (solver_rows["__is_dark"])]
    if not bad.empty:
        print("ERROR: dark rows leaked into clear. Showing top 10:")
        display(bad[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("Dark rows leaked into clear lane.")

    bad_hl = solver_rows[
        (solver_rows["group"] == "lightgrey") &
        (pd.to_numeric(solver_rows["lb_pct"], errors="coerce") >= 50.0)
    ]
    if not bad_hl.empty:
        print("ERROR: HL rows leaked into lightgrey. Showing top 10:")
        display(bad_hl[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("HL rows leaked into lightgrey lane.")

    lt_fix = pd.to_numeric(solver_rows.get("lt_num", np.nan), errors="coerce")
    mask_force_lg = lt_fix.notna() & (lt_fix.astype(float) >= float(DARK_LT_THRESHOLD))
    mask_force_lg = mask_force_lg & solver_rows["group"].isin(["clear", "clb_low", "clb_high"])
    solver_rows.loc[mask_force_lg, "group"] = "lightgrey"

    bad2 = solver_rows[
        (solver_rows["group"] == "clear") &
        (pd.to_numeric(solver_rows["lt_num"], errors="coerce") >= float(DARK_LT_THRESHOLD))
    ]
    if not bad2.empty:
        print("ERROR: LT-dark rows still in clear AFTER reroute. Showing top 10:")
        display(bad2[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("LT-dark rows still in clear after reroute.")

    bad_hl2 = solver_rows[
        (solver_rows["group"] == "lightgrey") &
        (pd.to_numeric(solver_rows["lb_pct"], errors="coerce") >= 50.0)
    ]
    if not bad_hl2.empty:
        print("ERROR: HL rows leaked into lightgrey AFTER reroute. Showing top 10:")
        display(bad_hl2[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("HL rows leaked into lightgrey after reroute.")

    low_anchor_primary = _choose_clb_low_anchor_window(solver_rows[solver_rows["group"].eq("clb_low")].copy())
    high_anchor_primary = _choose_clb_high_anchor_window(solver_rows[solver_rows["group"].eq("clb_high")].copy())
    lg_anchor_primary = _choose_lg_anchor_window(solver_rows[solver_rows["group"].eq("lightgrey")].copy())

    def _anchor_bb_summary(df_all, lane_name, anchor_dict, label_prefix):
        lane_df = df_all[df_all["group"].eq(lane_name)].copy()

        total_bb = float(pd.to_numeric(lane_df.get("bb", 0.0), errors="coerce").fillna(0.0).sum())

        if lane_df.empty or anchor_dict is None:
            print(f"{label_prefix} anchor USED=none | kept=0 BB | overflow=0 BB | total={int(round(total_bb))} BB")
            return

        lb_vals = pd.to_numeric(lane_df.get("lb_pct", np.nan), errors="coerce")
        bb_vals = pd.to_numeric(lane_df.get("bb", 0.0), errors="coerce").fillna(0.0).astype(float)

        start_v = float(anchor_dict["start"])
        end_v = float(anchor_dict["end"])

        keep_mask = lb_vals.ge(start_v) & lb_vals.lt(end_v)
        low_over_mask = lb_vals.lt(start_v)
        high_over_mask = lb_vals.ge(end_v)

        kept_bb = float(bb_vals[keep_mask].sum())
        low_over_bb = float(bb_vals[low_over_mask].sum())
        high_over_bb = float(bb_vals[high_over_mask].sum())
        overflow_bb = low_over_bb + high_over_bb

        start_txt = int(round(start_v))
        end_txt = int(round(end_v))

        def _fmt_bb(x):
            if abs(x - round(x)) < 1e-9:
                return str(int(round(x)))
            return f"{x:.1f}"

        print(
            f"{label_prefix} anchor USED={start_txt}–{end_txt} | "
            f"kept={_fmt_bb(kept_bb)} BB | "
            f"overflow={_fmt_bb(overflow_bb)} BB "
            f"(low={_fmt_bb(low_over_bb)}, high={_fmt_bb(high_over_bb)}) | "
            f"total={_fmt_bb(total_bb)} BB"
        )

    def _edge_fallback_anchor(lane_min, lane_max, width, side):
        if str(side).upper() == "LOW_EDGE":
            return {
                "start": float(lane_min),
                "end": float(lane_min + width),
                "massW": np.nan,
                "lt_err": np.nan,
                "fallback_side": "LOW_EDGE",
            }
        if str(side).upper() == "HIGH_EDGE":
            return {
                "start": float(lane_max - width),
                "end": float(lane_max),
                "massW": np.nan,
                "lt_err": np.nan,
                "fallback_side": "HIGH_EDGE",
            }
        return None

        low_share = low_over_bb / max(overflow_bb, EPS)
        high_share = high_over_bb / max(overflow_bb, EPS)

        if low_share >= float(dominance_frac):
            fb = {
                "start": float(lane_min),
                "end": float(lane_min + width),
                "massW": np.nan,
                "lt_err": np.nan,
                "fallback_side": "LOW_EDGE",
            }
        elif high_share >= float(dominance_frac):
            fb = {
                "start": float(lane_max - width),
                "end": float(lane_max),
                "massW": np.nan,
                "lt_err": np.nan,
                "fallback_side": "HIGH_EDGE",
            }
        else:
            return None

        if abs(float(fb["start"]) - start_v) < 1e-9 and abs(float(fb["end"]) - end_v) < 1e-9:
            return None

        return fb

    def _apply_anchor_plan(df_base, low_anchor_use, high_anchor_use, lg_anchor_use):
        work = df_base.copy()

        work["CLB_LOW_anchor_start"] = np.nan
        work["CLB_LOW_anchor_end"] = np.nan
        work["CLB_LOW_in_anchor"] = True

        work["CLB_HIGH_anchor_start"] = np.nan
        work["CLB_HIGH_anchor_end"] = np.nan
        work["CLB_HIGH_in_anchor"] = True

        work["LG_anchor_start"] = np.nan
        work["LG_anchor_end"] = np.nan
        work["LG_in_anchor"] = True

        mask_low_lane = work["group"].eq("clb_low")
        if low_anchor_use is not None:
            work.loc[mask_low_lane, "CLB_LOW_anchor_start"] = float(low_anchor_use["start"])
            work.loc[mask_low_lane, "CLB_LOW_anchor_end"] = float(low_anchor_use["end"])
            work.loc[mask_low_lane, "CLB_LOW_in_anchor"] = (
                pd.to_numeric(work.loc[mask_low_lane, "lb_pct"], errors="coerce").ge(float(low_anchor_use["start"])) &
                pd.to_numeric(work.loc[mask_low_lane, "lb_pct"], errors="coerce").lt(float(low_anchor_use["end"]))
            )

        mask_high_lane = work["group"].eq("clb_high")
        if high_anchor_use is not None:
            work.loc[mask_high_lane, "CLB_HIGH_anchor_start"] = float(high_anchor_use["start"])
            work.loc[mask_high_lane, "CLB_HIGH_anchor_end"] = float(high_anchor_use["end"])
            work.loc[mask_high_lane, "CLB_HIGH_in_anchor"] = (
                pd.to_numeric(work.loc[mask_high_lane, "lb_pct"], errors="coerce").ge(float(high_anchor_use["start"])) &
                pd.to_numeric(work.loc[mask_high_lane, "lb_pct"], errors="coerce").lt(float(high_anchor_use["end"]))
            )

        mask_lg_lane = work["group"].eq("lightgrey")
        if lg_anchor_use is not None:
            work.loc[mask_lg_lane, "LG_anchor_start"] = float(lg_anchor_use["start"])
            work.loc[mask_lg_lane, "LG_anchor_end"] = float(lg_anchor_use["end"])
            work.loc[mask_lg_lane, "LG_in_anchor"] = (
                pd.to_numeric(work.loc[mask_lg_lane, "lb_pct"], errors="coerce").ge(float(lg_anchor_use["start"])) &
                pd.to_numeric(work.loc[mask_lg_lane, "lb_pct"], errors="coerce").lt(float(lg_anchor_use["end"]))
            )

        work["SOLVER_EXCLUDE"] = False
        work["SOLVER_EXCLUDE_REASON"] = ""
        work["LB_is_overflow"] = False

        if low_anchor_use is not None:
            mask_low_out = mask_low_lane & (~work["CLB_LOW_in_anchor"].fillna(False))
            work.loc[mask_low_out, "SOLVER_EXCLUDE"] = True
            work.loc[mask_low_out, "SOLVER_EXCLUDE_REASON"] = _append_reason_col(
                work.loc[mask_low_out, "SOLVER_EXCLUDE_REASON"],
                "CLB_LOW_outside_anchor_window"
            )
            work.loc[mask_low_out, "LB_is_overflow"] = True

        if high_anchor_use is not None:
            mask_high_out = mask_high_lane & (~work["CLB_HIGH_in_anchor"].fillna(False))
            work.loc[mask_high_out, "SOLVER_EXCLUDE"] = True
            work.loc[mask_high_out, "SOLVER_EXCLUDE_REASON"] = _append_reason_col(
                work.loc[mask_high_out, "SOLVER_EXCLUDE_REASON"],
                "CLB_HIGH_outside_anchor_window"
            )
            work.loc[mask_high_out, "LB_is_overflow"] = True

        if lg_anchor_use is not None:
            mask_lg_out = mask_lg_lane & (~work["LG_in_anchor"].fillna(False))
            work.loc[mask_lg_out, "SOLVER_EXCLUDE"] = True
            work.loc[mask_lg_out, "SOLVER_EXCLUDE_REASON"] = _append_reason_col(
                work.loc[mask_lg_out, "SOLVER_EXCLUDE_REASON"],
                "LG_outside_anchor_window"
            )
            work.loc[mask_lg_out, "LB_is_overflow"] = True

        def compute_visible_idx(df_lane, bb_limit):
            if df_lane.empty:
                return set()
            df_sorted = df_lane.sort_values("lb_pct", ascending=True)
            keep_idx = []
            cum_bb = 0.0
            for idx_row, row in df_sorted.iterrows():
                bb_here = float(row.get("bb", 0.0))
                if cum_bb + bb_here <= bb_limit + 1e-9:
                    keep_idx.append(idx_row)
                    cum_bb += bb_here
                else:
                    break
            if cum_bb <= EPS:
                keep_idx = list(df_sorted.index)
            return set(keep_idx)

        hl_subset = work[work["group"].eq("highlight_use")].copy()
        highlight_visible_idx = compute_visible_idx(hl_subset, VISIBLE_BB_LIMIT)
        mask_hl_all = work["group"].eq("highlight_use")
        work.loc[mask_hl_all & (~work.index.isin(highlight_visible_idx)), "LB_is_overflow"] = True

        def _dyn_bin(row):
            g = row.get("group", "")

            if g == "clear":
                return "0 - 2"

            if g == "clb_low":
                s = row.get("CLB_LOW_anchor_start", np.nan)
                e = row.get("CLB_LOW_anchor_end", np.nan)
                if pd.notna(s) and pd.notna(e):
                    return f"{int(round(float(s)))} - {int(round(float(e)))}"
                return "2 - 25"

            if g == "clb_high":
                s = row.get("CLB_HIGH_anchor_start", np.nan)
                e = row.get("CLB_HIGH_anchor_end", np.nan)
                if pd.notna(s) and pd.notna(e):
                    return f"{int(round(float(s)))} - {int(round(float(e)))}"
                return "25 - 50"

            if g == "lightgrey":
                s = row.get("LG_anchor_start", np.nan)
                e = row.get("LG_anchor_end", np.nan)
                if pd.notna(s) and pd.notna(e):
                    return f"LG {int(round(float(s)))} - {int(round(float(e)))}"
                return "LG 2 - 25"

            if g == "highlight_use":
                return "50 - 100"

            return ""

        GROUP_TO_BASE = {
            "clear": "Clear Neutral",
            "clb_low": "Clearlightblue Low",
            "clb_high": "Clearlightblue High",
            "lightgrey": "Lightgrey",
            "highlight_use": "Lightblue"
        }
        work["Material"] = work["group"].map(GROUP_TO_BASE)
        work["Material_Display"] = work["Material"]

        bins = work.copy()
        bins["dyn_bin"] = bins.apply(_dyn_bin, axis=1)
        eff = bins[~bins["SOLVER_EXCLUDE"].fillna(False)].copy()

        return work, bins, eff

    def _refresh_solver_state_from_eff(eff_df):
        global solver_bins, groups_present, group_lb_means, group_bd_means, group_lt_means
        global stock_caps, stock_shares_no_hl_all, upper_cap_local, effective_min_total

        stock_caps_local = (
            eff_df.groupby("group", as_index=False)["bb"]
            .sum()
            .set_index("group")["bb"]
            .to_dict()
        )

        (
            groups_present_local,
            group_lb_means_local,
            stock_shares_no_hl_all_local,
            group_bd_means_local,
            group_lt_means_local
        ) = compute_group_stats_for_solver(eff_df)

        allowed_groups = {"clear", "clb_low", "clb_high", "lightgrey", "highlight_use"}
        groups_present_local = {k: v for k, v in groups_present_local.items() if k in allowed_groups}
        group_lb_means_local = {k: v for k, v in group_lb_means_local.items() if k in allowed_groups}
        group_bd_means_local = {k: v for k, v in group_bd_means_local.items() if k in allowed_groups}
        group_lt_means_local = {k: v for k, v in group_lt_means_local.items() if k in allowed_groups}
        stock_caps_local     = {k: v for k, v in stock_caps_local.items()     if k in allowed_groups}

        stock_shares_no_hl_all_local = {
            k: v for k, v in stock_shares_no_hl_all_local.items()
            if k in {"clear", "clb_low", "clb_high", "lightgrey"}
        }

        upper_cap_local_local = {
            "clear": UPPER_CAP.get("clear", 200.0),
            "clb_low": UPPER_CAP.get("clb_low", 200.0),
            "clb_high": UPPER_CAP.get("clb_high", 200.0),
            "lightgrey": UPPER_CAP.get("lightgrey", 200.0),
            "highlight_use": UPPER_CAP.get("highlight_use", 200.0),
        }
        effective_min_total_local = get_effective_min_total(stock_caps_local, demand_mixes_floor=MIN_MIXES)

        solver_bins = eff_df.copy()  # temporary guard against accidental empty global use
        groups_present = groups_present_local
        group_lb_means = group_lb_means_local
        group_bd_means = group_bd_means_local
        group_lt_means = group_lt_means_local
        stock_caps = stock_caps_local
        stock_shares_no_hl_all = stock_shares_no_hl_all_local
        upper_cap_local = upper_cap_local_local
        effective_min_total = effective_min_total_local

    primary_plan = {
        "clb_low": low_anchor_primary,
        "clb_high": high_anchor_primary,
        "lightgrey": lg_anchor_primary,
        "tag_suffix": "PRIMARY",
        "fallback_used": [],
    }

    solver_rows_primary, solver_bins_primary, solver_eff_primary = _apply_anchor_plan(
        solver_rows,
        primary_plan["clb_low"],
        primary_plan["clb_high"],
        primary_plan["lightgrey"]
    )

    _refresh_solver_state_from_eff(solver_eff_primary)
    solver_bins = solver_bins_primary.copy()
    lane_bb_tonnage = _lane_bb_tonnage_map(solver_eff_primary)

    scored = build_scored_candidates_with_priority()
    best = choose_best_candidate(scored)

    chosen_plan = primary_plan
    chosen_solver_rows = solver_rows_primary
    chosen_solver_bins = solver_bins_primary
    chosen_solver_eff = solver_eff_primary

    if best is None:
        low_anchor_fb_low = _edge_fallback_anchor(0.0, CLB_LOW_MAX, 12.0, "LOW_EDGE")
        low_anchor_fb_high = _edge_fallback_anchor(0.0, CLB_LOW_MAX, 12.0, "HIGH_EDGE")

        high_anchor_fb_low = _edge_fallback_anchor(CLB_HIGH_MIN, CLB_HIGH_MAX, 12.0, "LOW_EDGE")
        high_anchor_fb_high = _edge_fallback_anchor(CLB_HIGH_MIN, CLB_HIGH_MAX, 12.0, "HIGH_EDGE")

        lg_anchor_fb_low = _edge_fallback_anchor(LG_MIN, LG_MAX, ANCHOR_WIDTH_LG, "LOW_EDGE")
        lg_anchor_fb_high = _edge_fallback_anchor(LG_MIN, LG_MAX, ANCHOR_WIDTH_LG, "HIGH_EDGE")

        fallback_scenarios = []

        def _same_anchor(a, b):
            if a is None or b is None:
                return False
            return (
                abs(float(a["start"]) - float(b["start"])) < 1e-9 and
                abs(float(a["end"]) - float(b["end"])) < 1e-9
            )

        def _add_fallback_plan(low_a, high_a, lg_a, tag_suffix, flags):
            if (low_a is not None and low_anchor_primary is not None and _same_anchor(low_a, low_anchor_primary) and
                high_a is not None and high_anchor_primary is not None and _same_anchor(high_a, high_anchor_primary) and
                lg_a is not None and lg_anchor_primary is not None and _same_anchor(lg_a, lg_anchor_primary)):
                return

            fallback_scenarios.append({
                "clb_low": low_a,
                "clb_high": high_a,
                "lightgrey": lg_a,
                "tag_suffix": tag_suffix,
                "fallback_used": flags,
            })

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            if low_anchor_primary is None or not _same_anchor(a_low, low_anchor_primary):
                _add_fallback_plan(a_low, high_anchor_primary, lg_anchor_primary, f"FALLBACK_{lbl_low.replace('=', '_')}", [lbl_low])

        for a_high, lbl_high in [
            (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
            (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
        ]:
            if high_anchor_primary is None or not _same_anchor(a_high, high_anchor_primary):
                _add_fallback_plan(low_anchor_primary, a_high, lg_anchor_primary, f"FALLBACK_{lbl_high.replace('=', '_')}", [lbl_high])

        for a_lg, lbl_lg in [
            (lg_anchor_fb_low, "LG=LOW_EDGE"),
            (lg_anchor_fb_high, "LG=HIGH_EDGE"),
        ]:
            if lg_anchor_primary is None or not _same_anchor(a_lg, lg_anchor_primary):
                _add_fallback_plan(low_anchor_primary, high_anchor_primary, a_lg, f"FALLBACK_{lbl_lg.replace('=', '_')}", [lbl_lg])

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            for a_high, lbl_high in [
                (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
                (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
            ]:
                flags = [lbl_low, lbl_high]
                _add_fallback_plan(
                    a_low, a_high, lg_anchor_primary,
                    f"FALLBACK_{lbl_low.replace('=', '_')}_{lbl_high.replace('=', '_')}",
                    flags
                )

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            for a_lg, lbl_lg in [
                (lg_anchor_fb_low, "LG=LOW_EDGE"),
                (lg_anchor_fb_high, "LG=HIGH_EDGE"),
            ]:
                flags = [lbl_low, lbl_lg]
                _add_fallback_plan(
                    a_low, high_anchor_primary, a_lg,
                    f"FALLBACK_{lbl_low.replace('=', '_')}_{lbl_lg.replace('=', '_')}",
                    flags
                )

        for a_high, lbl_high in [
            (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
            (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
        ]:
            for a_lg, lbl_lg in [
                (lg_anchor_fb_low, "LG=LOW_EDGE"),
                (lg_anchor_fb_high, "LG=HIGH_EDGE"),
            ]:
                flags = [lbl_high, lbl_lg]
                _add_fallback_plan(
                    low_anchor_primary, a_high, a_lg,
                    f"FALLBACK_{lbl_high.replace('=', '_')}_{lbl_lg.replace('=', '_')}",
                    flags
                )

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            for a_high, lbl_high in [
                (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
                (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
            ]:
                for a_lg, lbl_lg in [
                    (lg_anchor_fb_low, "LG=LOW_EDGE"),
                    (lg_anchor_fb_high, "LG=HIGH_EDGE"),
                ]:
                    flags = [lbl_low, lbl_high, lbl_lg]
                    _add_fallback_plan(
                        a_low, a_high, a_lg,
                        f"FALLBACK_{lbl_low.replace('=', '_')}_{lbl_high.replace('=', '_')}_{lbl_lg.replace('=', '_')}",
                        flags
                    )

        if fallback_scenarios:
            print(f"ANCHOR FALLBACK scenarios queued={len(fallback_scenarios)}")
        else:
            print("ANCHOR FALLBACK scenarios queued=0")

        feasible_fallbacks = []

        for plan_try in fallback_scenarios:
            def _fmt_anchor(a):
                if a is None:
                    return "none"
                return f"{int(round(float(a['start'])))}–{int(round(float(a['end'])))}"

            print(
                "TRY FALLBACK:",
                f"tag={plan_try['tag_suffix']}",
                f"CLB_LOW={_fmt_anchor(plan_try['clb_low'])}",
                f"CLB_HIGH={_fmt_anchor(plan_try['clb_high'])}",
                f"LG={_fmt_anchor(plan_try['lightgrey'])}",
                f"flags={','.join(plan_try.get('fallback_used', [])) if plan_try.get('fallback_used') else 'none'}"
            )

            rows_try, bins_try, eff_try = _apply_anchor_plan(
                solver_rows,
                plan_try["clb_low"],
                plan_try["clb_high"],
                plan_try["lightgrey"]
            )

            _refresh_solver_state_from_eff(eff_try)
            solver_bins = bins_try.copy()
            lane_bb_tonnage = _lane_bb_tonnage_map(eff_try)

            scored_try = build_scored_candidates_with_priority()
            best_try = choose_best_candidate(scored_try)

            if best_try is not None:
                print(f"FALLBACK RESULT: feasible -> {plan_try['tag_suffix']}")
                feasible_fallbacks.append({
                    "best": best_try,
                    "plan": plan_try,
                    "rows": rows_try,
                    "bins": bins_try,
                    "eff": eff_try,
                })
            else:
                print(f"FALLBACK RESULT: no feasible -> {plan_try['tag_suffix']}")

        if feasible_fallbacks:
            def _fb_sort_key(rec):
                b = rec["best"]
                lb_err = abs(float(b.get("result_lb", np.nan)) - float(target_lb))
                lt_err = abs(float(b.get("result_lt", np.nan)) - float(lt_target))
                mixes = int(b.get("mixes", 0))
                total_bb = float(sum((b.get("picks") or {}).values()))
                fallback_count = len(rec["plan"].get("fallback_used", []))
                return (
                    fallback_count,   # prefer smaller change
                    lb_err,           # then closer LB
                    lt_err,           # then closer LT
                    -mixes,           # then more mixes
                    -total_bb         # then fuller batch
                )

            feasible_fallbacks = sorted(feasible_fallbacks, key=_fb_sort_key)
            chosen_fb = feasible_fallbacks[0]

            best = chosen_fb["best"]
            chosen_plan = chosen_fb["plan"]
            chosen_solver_rows = chosen_fb["rows"]
            chosen_solver_bins = chosen_fb["bins"]
            chosen_solver_eff = chosen_fb["eff"]

            print(
                "FALLBACK BEST CHOSEN:",
                f"tag={chosen_plan['tag_suffix']}",
                f"flags={','.join(chosen_plan.get('fallback_used', [])) if chosen_plan.get('fallback_used') else 'none'}",
                f"mixes={int(best.get('mixes', 0))}",
                f"LB={float(best.get('result_lb', np.nan)):.3f}",
                f"LT={float(best.get('result_lt', np.nan)):.3f}"
            )

    solver_rows = chosen_solver_rows.copy()
    solver_bins = chosen_solver_bins.copy()
    solver_eff = chosen_solver_eff.copy()

    _refresh_solver_state_from_eff(solver_eff)
    solver_bins = chosen_solver_bins.copy()
    lane_bb_tonnage = _lane_bb_tonnage_map(solver_eff)

    if chosen_plan.get("fallback_used"):
        print("ANCHOR FALLBACK USED:", ", ".join(chosen_plan["fallback_used"]))

    _anchor_bb_summary(solver_rows, "clb_low", chosen_plan["clb_low"], "CLB_LOW")
    _anchor_bb_summary(solver_rows, "clb_high", chosen_plan["clb_high"], "CLB_HIGH")
    _anchor_bb_summary(solver_rows, "lightgrey", chosen_plan["lightgrey"], "LG")

    solver_tag = f"SOLVER=DUAL_CLB_LANE_LOCAL_ANCHOR_{chosen_plan['tag_suffix']}"
    print(solver_tag)

    def compute_hidden_materials(df_full):
        tmp = df_full.copy()
        if "bb" in tmp.columns:
            tmp["_bb_count"] = pd.to_numeric(tmp["bb"], errors="coerce").fillna(0.0).astype(float)
        else:
            tmp["_bb_count"] = tmp["Charge/ Paket"].apply(lambda x: float(len(x)) if isinstance(x, (list, tuple)) else 0.0)

        tmp["MatForHide"] = tmp["Material_Display"].astype(str)
        tmp.loc[(tmp["group"] == "highlight_use") & (tmp["LB_is_overflow"]), "MatForHide"] = "Lightblue (overflow)"

        by_mat = (
            tmp.groupby("MatForHide", as_index=False)["_bb_count"]
               .sum()
               .rename(columns={"_bb_count": "BB_Count"})
        )
        total_bb = float(by_mat["BB_Count"].sum())

        def _thr(material):
            abs_thr = 5.0 if str(material).startswith("Lightblue") else 20.0
            pct_thr = 0.05 * total_bb
            return min(abs_thr, pct_thr)

        return set(by_mat[by_mat.apply(lambda r: float(r["BB_Count"]) < _thr(r["MatForHide"]), axis=1)]["MatForHide"])

    hidden_mats = compute_hidden_materials(solver_bins)

    def build_headers_from_solverbins(solver_bins, best, specs, hidden_mats=None, mix_stats=None, solver_tag=None):
        view = solver_bins.groupby("group", as_index=False).agg(bb=("bb", "sum"))
        total_bb = float(view["bb"].sum())
        sh = {r["group"]: float(r["bb"]) / max(total_bb, 1e-9) for _, r in view.iterrows()}

        Lmin, Lmax = specs["L_range"]
        bmin, bmax = specs["b_range"]

        order = ["clear", "clb_low", "clb_high", "lightgrey", "highlight_use"]
        label_map = {
            "clear": "clear",
            "clb_low": "clb_low",
            "clb_high": "clb_high",
            "lightgrey": "lightgrey",
            "highlight_use": "highlight",
        }
        tag_txt = f", {solver_tag}" if solver_tag else ""

        parts = [f"Stock: {int(round(total_bb))} BB total (Modus {PRODUCTION_MODE}{tag_txt})"]
        for g in order:
            bb_val = float(view.loc[view["group"] == g, "bb"].sum()) if (g in set(view["group"].values)) else 0.0
            if bb_val > 0:
                parts.append(f"{label_map[g]}: {int(round(bb_val))} BB ({sh.get(g, 0.0) * 100:.1f}%)")
        stock_line = " | ".join(parts)

        metrics_suffix = ""
        if mix_stats is not None:
            mix_lb = mix_stats.get("mix_lb", float("nan"))
            mix_lt = mix_stats.get("mix_lt", float("nan"))
            mix_bd = mix_stats.get("mix_bd", float("nan"))
            mix_mx = int(mix_stats.get("mix_mx", 0))

            def _fmt_num(v, fmt, dash="—"):
                try:
                    if v is None or (isinstance(v, float) and np.isnan(v)):
                        return dash
                    return fmt.format(float(v))
                except Exception:
                    return dash

            lb_txt = _fmt_num(mix_lb, "LB={:.3f}")
            lt_txt = _fmt_num(mix_lt, "LT={:.3f}")
            bd_txt = _fmt_num(mix_bd, "BD={:.0f}")
            mx_txt = f"mixes={mix_mx}" if mix_mx > 0 else "mixes=—"

            bd_flag = ""
            try:
                if mix_bd is not None and not (isinstance(mix_bd, float) and np.isnan(mix_bd)):
                    bd_val = float(mix_bd)
                    bd_range_txt = f"{BD_WARN_LOW:.0f}–{BD_WARN_HIGH:.0f}"
                    if bd_val < float(BD_WARN_LOW):
                        bd_flag = f" | BD warning: LOW (<{BD_WARN_LOW:.0f})"
                    elif bd_val > float(BD_WARN_HIGH):
                        bd_flag = f" | BD warning: HIGH (>{BD_WARN_HIGH:.0f})"
                    else:
                        bd_flag = f" | BD OK ({bd_range_txt})"
            except Exception:
                bd_flag = ""

            metrics_suffix = f"Mixture: {lb_txt} | {lt_txt} | {bd_txt} | {mx_txt}{bd_flag}"

        def _is_hidden(name):
            if not hidden_mats:
                return False
            name = str(name)
            if name in hidden_mats:
                return True
            if name == "Lightblue":
                return ("Lightblue" in hidden_mats) or ("Lightblue (overflow)" in hidden_mats)
            return False

        def line_for(label, best_dict):
            suffix_specs = f"Expected L*: {Lmin:.1f} – {Lmax:.1f} | Expected b*: {bmin:.1f} – {bmax:.1f}"
            if not best_dict:
                core = f"{label}: —"
            else:
                picks = best_dict.get("picks") or {}
                mixes = int(best_dict.get("mixes", 0))

                def disp_part(key, dispname):
                    v = float(picks.get(key, 0.0))
                    if v <= EPS:
                        return None
                    amt_txt = bb_to_display(v, key)
                    if amt_txt is None:
                        return None
                    suffix = " (hidden)" if _is_hidden(dispname) else ""
                    if key == "highlight_use":
                        return f"{amt_txt} Lightblue{suffix}"
                    return f"{amt_txt} {dispname}{suffix}"

                segs = [
                    disp_part("clear", "Clear Neutral"),
                    disp_part("clb_low", "Clearlightblue Low"),
                    disp_part("clb_high", "Clearlightblue High"),
                    disp_part("lightgrey", "Lightgrey"),
                    disp_part("highlight_use", "Lightblue"),
                ]
                segs = [s for s in segs if s]
                core = f"{label}: " + " | ".join(segs) + (f" → Max Mixes {mixes}" if mixes > 0 else "")

            out = core + " | " + suffix_specs
            if metrics_suffix:
                out = out + " | " + metrics_suffix
            return out

        corr = line_for("Recipe Blend", best)
        return stock_line, corr

    mix_stats = None
    if best is not None:
        mix_stats = {
            "mix_lb": best.get("result_lb", float("nan")),
            "mix_lt": best.get("result_lt", float("nan")),
            "mix_bd": best.get("result_bd", float("nan")),
            "mix_mx": int(best.get("mixes", 0)),
        }

    header_stock, header_corr = build_headers_from_solverbins(solver_bins, best, specs, hidden_mats, mix_stats, solver_tag)
    print(header_stock)
    print(header_corr)

    if best is None:
        return

    # =========================================================
    # Candidate Recipe table + Excel
    # =========================================================
    combined_stock_solver = solver_bins.copy()
    total_stock = float(combined_stock_solver["Menge"].sum())
    required_tons = total_stock

    if total_stock > 0:
        combined_stock_solver["Dynamic_Percentage"] = combined_stock_solver["Menge"] / total_stock * 100.0
        combined_stock_solver["Tonnage_Allocated"] = combined_stock_solver["Menge"] / total_stock * required_tons
    else:
        combined_stock_solver["Dynamic_Percentage"] = 0.0
        combined_stock_solver["Tonnage_Allocated"] = 0.0

    valid_cands = [r for r in combined_stock_solver.to_dict("records") if float(r.get("Tonnage_Allocated", 0.0)) > 0.0]
    best_candidate = tuple(valid_cands) if valid_cands else None

    def aggregate_candidate_recipe_all(required):
        df0 = combined_stock_solver.copy()

        stats = df0.groupby(
            ["Material_Name", "group", "dyn_bin", "Material_Display", "LB_is_overflow"],
            as_index=False
        ).agg({
            "Menge": "sum",
            "AV_LB%": lambda s: (
                float(np.average(pd.to_numeric(s, errors="coerce").fillna(0.0), weights=df0.loc[s.index, "Menge"]))
                if not pd.to_numeric(s, errors="coerce").isna().all() else np.nan
            ),
            "AV_BulkDensity": lambda s: (
                float(np.average(pd.to_numeric(df0.loc[s.index, "AV_BulkDensity"], errors="coerce").fillna(0.0),
                                 weights=df0.loc[s.index, "Menge"]))
                if not df0.loc[s.index, "AV_BulkDensity"].isna().all() else np.nan
            ),
            "AV_Lightness": lambda s: (
                float(np.average(pd.to_numeric(df0.loc[s.index, "AV_Lightness"], errors="coerce").fillna(0.0),
                                 weights=df0.loc[s.index, "Menge"]))
                if not df0.loc[s.index, "AV_Lightness"].isna().all() else np.nan
            ),
            "Tonnage_Allocated": "sum",
            "Dynamic_Percentage": "sum" if "Dynamic_Percentage" in df0.columns else "size",
        }).rename(columns={
            "Menge": "Group_Menge",
            "AV_LB%": "Group_AV_LB%",
            "AV_BulkDensity": "Group_AV_BD",
            "AV_Lightness": "Group_AV_LT",
            "Tonnage_Allocated": "Group_TA",
            "Dynamic_Percentage": "Group_DP"
        })

        def weight_factor(lb):
            if lb is None or (isinstance(lb, float) and np.isnan(lb)):
                return 1.0
            return 1.0 / (1.0 + abs(float(lb) - float(target_lb)) / max(float(target_lb), 1e-9))

        stats["Weight_Factor"] = stats["Group_AV_LB%"].apply(weight_factor)
        stats["Modified_Weight"] = stats["Group_TA"] * stats["Weight_Factor"]
        total_mod = float(stats["Modified_Weight"].sum())
        stats["Adj_TA"] = (stats["Modified_Weight"] / total_mod * required) if total_mod else stats["Group_TA"]
        stats["Percentage"] = stats["Adj_TA"] / required * 100.0 if required > 0 else 0.0

        batches = required / 5.5 if required > 0 else 1.0

        def _bb_round(row):
            per_batch_tons = float(row["Adj_TA"]) / batches if batches > 0 else 0.0
            grp = str(row["group"])
            if grp == "highlight_use":
                return round(per_batch_tons / STEP_HL) * STEP_HL
            tons_per_bb = float(lane_bb_tonnage.get(grp, NORMAL_BB_TONNAGE))
            bb_needed = per_batch_tons / max(tons_per_bb, 1e-9)
            return round(bb_needed * 2.0) / 2.0

        stats["Big Bags"] = stats.apply(_bb_round, axis=1)
        stats["BB_Min"] = (stats["Big Bags"] * 2 - 1).clip(lower=0).round() / 2
        stats["BB_Max"] = (stats["Big Bags"] * 2 + 1).round() / 2

        out = df0.merge(
            stats[[
                "Material_Name", "group", "dyn_bin", "Material_Display", "LB_is_overflow",
                "Percentage", "Big Bags", "BB_Min", "BB_Max"
            ]],
            on=["Material_Name", "group", "dyn_bin", "Material_Display", "LB_is_overflow"],
            how="left"
        )
        out["Material"] = out["Material_Display"]
        return out

    candidate_recipe = aggregate_candidate_recipe_all(required_tons) if best_candidate else pd.DataFrame()

    def format_packets(packets):
        if isinstance(packets, str):
            packets = packets.split(", ")
        return ", ".join(find_ranges(packets))

    unused_mats_for_hide = set()

    if not candidate_recipe.empty:
        candidate_recipe = candidate_recipe.copy()
        expr = r"(?:st[-\s]?f|recyclass)"
        candidate_recipe["NamePriority"] = candidate_recipe["Beschreibung"].astype(str).str.contains(expr, case=False, na=False, regex=True).astype(int)
        candidate_recipe["Material_Vis"] = candidate_recipe["Material"].astype(str)

        keep_mats = ["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"]
        candidate_recipe = candidate_recipe[candidate_recipe["Material_Vis"].isin(keep_mats)].copy()
        candidate_recipe["Material_Vis"] = pd.Categorical(candidate_recipe["Material_Vis"], categories=keep_mats, ordered=True)
        candidate_recipe["AV_LB%"] = pd.to_numeric(candidate_recipe["AV_LB%"], errors="coerce")

        def _as_pkt_list(x):
            if isinstance(x, (list, tuple, set)):
                return [str(p).strip() for p in x if str(p).strip() not in ("", "nan", "NaN", "None")]
            if x is None or (isinstance(x, float) and np.isnan(x)):
                return []
            s = str(x).strip()
            if s in ("", "nan", "NaN", "None"):
                return []
            return [s]

        candidate_recipe["Raw_Charge"] = candidate_recipe["Charge/ Paket"].apply(_as_pkt_list)
        if "dyn_bin" not in candidate_recipe.columns:
            candidate_recipe["dyn_bin"] = ""

        merge_keys = ["Material_Vis", "Lagerplatz", "dyn_bin", "LB_is_overflow"]
        merged_rows = []

        for keys, grp in candidate_recipe.groupby(merge_keys, dropna=False, sort=False, observed=False):
            matv, lager, dbin, is_overflow = keys

            pkts = []
            for lst in grp["Raw_Charge"].tolist():
                if isinstance(lst, (list, tuple, set)):
                    pkts.extend(lst)
            pkts = list(dict.fromkeys([str(p).strip() for p in pkts if str(p).strip() not in ("", "nan", "NaN", "None")]))

            w = pd.to_numeric(grp.get("Menge", grp.get("Quantity", 0.0)), errors="coerce").fillna(0.0).astype(float)
            total_w = float(w.sum())
            if total_w <= EPS:
                continue

            def _wavg(col):
                if col not in grp.columns:
                    return np.nan
                x = pd.to_numeric(grp[col], errors="coerce")
                if x.isna().all():
                    return np.nan
                m = ~x.isna()
                if not m.any():
                    return np.nan
                ww = w[m].astype(float)
                if float(ww.sum()) <= EPS:
                    return float(np.nanmean(x))
                return float((ww * x[m].astype(float)).sum() / max(float(ww.sum()), 1e-9))

            arts = [a for a in grp.get("Artikel", pd.Series([], dtype=object)).tolist() if pd.notna(a)]
            arts_u = sorted(set(int(a) for a in arts)) if arts else []
            if len(arts_u) == 1:
                art_out = arts_u[0]
                desc_out = str(grp["Beschreibung"].iloc[0]) if "Beschreibung" in grp.columns else ""
            else:
                art_out = 0
                desc_out = "MIXED (" + ", ".join(map(str, arts_u[:10])) + (", ..." if len(arts_u) > 10 else "") + ")"

            qty_sum = float(pd.to_numeric(grp["Quantity"], errors="coerce").fillna(0.0).sum()) if "Quantity" in grp.columns else total_w
            ta_sum = float(pd.to_numeric(grp["Tonnage_Allocated"], errors="coerce").fillna(0.0).sum()) if "Tonnage_Allocated" in grp.columns else float("nan")
            dp_sum = float(pd.to_numeric(grp["Dynamic_Percentage"], errors="coerce").fillna(0.0).sum()) if "Dynamic_Percentage" in grp.columns else float("nan")

            if "Quality" in grp.columns and not grp["Quality"].isna().all():
                q_out = grp["Quality"].mode().iat[0]
            else:
                q_out = None

            merged_rows.append({
                "Material_Vis": str(matv),
                "Lagerplatz": lager,
                "dyn_bin": str(dbin),
                "LB_is_overflow": bool(is_overflow),
                "Artikel": art_out,
                "Beschreibung": desc_out,
                "Quality": q_out,
                "Menge": total_w,
                "Quantity": qty_sum,
                "Tonnage_Allocated": ta_sum,
                "Dynamic_Percentage": dp_sum,
                "AV_LB%": _wavg("AV_LB%"),
                "AV_Lightness": _wavg("AV_Lightness"),
                "AV_BulkDensity": _wavg("AV_BulkDensity"),
                "Raw_Charge": pkts,
                "NamePriority": int(grp["NamePriority"].max()) if "NamePriority" in grp.columns else 0,
            })

        candidate_recipe = pd.DataFrame(merged_rows).reset_index(drop=True)
        candidate_recipe["Charge/ Paket"] = candidate_recipe["Raw_Charge"].apply(format_packets)

        sort_parts = []
        for _, row in candidate_recipe.iterrows():
            matv = str(row["Material_Vis"])
            avlb = float(row["AV_LB%"]) if pd.notna(row["AV_LB%"]) else 0.0
            if matv == "Clearlightblue High":
                sort_parts.append(-avlb)
            else:
                sort_parts.append(avlb)
        candidate_recipe["__mat_specific_sort"] = sort_parts

        candidate_recipe = candidate_recipe.sort_values(
            ["Material_Vis", "NamePriority", "__mat_specific_sort", "Lagerplatz", "AV_LB%"],
            kind="stable", ignore_index=True
        )
        candidate_recipe = candidate_recipe.drop(columns=["__mat_specific_sort"])
        candidate_recipe["Pick_Seq"] = candidate_recipe.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1

        pick_dict = (best or {}).get("picks", {})
        group_to_vis = {
            "clear": "Clear Neutral",
            "clb_low": "Clearlightblue Low",
            "clb_high": "Clearlightblue High",
            "lightgrey": "Lightgrey",
            "highlight_use": "Lightblue"
        }
        vis_to_group = {
            "Clear Neutral": "clear",
            "Clearlightblue Low": "clb_low",
            "Clearlightblue High": "clb_high",
            "Lightgrey": "lightgrey",
            "Lightblue": "highlight_use"
        }

        def _pkt_is_fa(pkt):
            if pkt is None:
                return False
            s = str(pkt).strip().upper()
            return s.startswith("FA")

        def _lane_effective_bb_mass(mat_vis):
            grp_key = vis_to_group.get(str(mat_vis))
            if grp_key is None:
                return 1.0
            lane_df = solver_eff[solver_eff["group"].eq(grp_key)].copy()
            if lane_df.empty:
                return 1.0

            fa_bb = 0.0
            tot_bb = 0.0
            for _, rr in lane_df.iterrows():
                bb_here = float(pd.to_numeric(rr.get("bb", 0.0), errors="coerce"))
                if not np.isfinite(bb_here):
                    bb_here = 0.0
                if bb_here <= EPS:
                    continue
                pkts = rr.get("Charge/ Paket", [])
                if not isinstance(pkts, (list, tuple, set)):
                    pkts = [pkts] if pd.notna(pkts) else []
                pkt_list = [str(x).strip() for x in pkts if str(x).strip() not in ("", "nan", "NaN", "None")]
                if len(pkt_list) == 0:
                    continue
                fa_count = sum(1 for p in pkt_list if _pkt_is_fa(p))
                row_fa_frac = float(fa_count) / float(len(pkt_list))
                tot_bb += bb_here
                fa_bb += bb_here * row_fa_frac

            if tot_bb <= EPS:
                return 1.0
            fa_share = fa_bb / tot_bb
            if fa_share >= 0.50:
                return 1.0 - 0.25 * fa_share
            return 1.0

        bb_plan = {}
        for g_key, bb_val in pick_dict.items():
            vis = group_to_vis.get(g_key)
            if vis is None:
                continue
            bb_val = float(bb_val)
            lane_mass_factor = float(_lane_effective_bb_mass(vis))
            if lane_mass_factor <= EPS:
                lane_mass_factor = 1.0
            corrected_bb = bb_val / lane_mass_factor
            if g_key == "highlight_use":
                rounded = round(corrected_bb / STEP_HL) * STEP_HL
            else:
                rounded = round(corrected_bb * 2.0) / 2.0
            bb_plan[vis] = bb_plan.get(vis, 0.0) + rounded

        for m in keep_mats:
            bb_plan.setdefault(m, 0.0)

        total_bb_plan = float(sum(v for v in bb_plan.values() if v is not None))
        candidate_recipe["Big Bags"] = candidate_recipe["Material_Vis"].astype(str).map(bb_plan).astype(float).fillna(0.0)

        def _lane_pct(matv):
            if total_bb_plan <= 0:
                return 0.0
            return (float(bb_plan.get(str(matv), 0.0)) / total_bb_plan) * 100.0

        candidate_recipe["Percentage"] = candidate_recipe["Material_Vis"].apply(_lane_pct)
        unused_mats_for_hide = {m for m, v in bb_plan.items() if (v is None or float(v) <= 1e-9)}

    def generate_supply_chain_exact(df):
        if df is None or df.empty:
            return pd.DataFrame()

        order_vis = ["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"]
        supply, required, flexible, pkt_row = {}, {}, {}, {}

        for _, r in df.iterrows():
            matv = str(r["Material_Vis"])
            pkts = r.get("Raw_Charge", [])
            pkts = list(pkts) if isinstance(pkts, (list, tuple)) else ([pkts] if pkts else [])
            if not pkts:
                continue

            supply.setdefault(matv, [])
            rb = float(r.get("Big Bags", 0.0))
            required[matv] = max(int(math.floor(rb)), 0)
            flexible[matv] = (matv in ["Lightblue"])

            for pkt in pkts:
                supply[matv].append(pkt)
                pkt_row[pkt] = {
                    "Artikel": r.get("Artikel"),
                    "Beschreibung": r.get("Beschreibung"),
                    "Lagerplatz": r.get("Lagerplatz"),
                    "AV_LB%": r.get("AV_LB%"),
                    "Material_Vis": matv
                }

        max_full = []
        for m in order_vis:
            if m in supply and (not flexible.get(m, False)) and required.get(m, 0) > 0:
                max_full.append(len(supply[m]) // required[m])
        max_mix = min(max_full) if max_full else 0

        rows = []
        for mix in range(max_mix):
            for matv in order_vis:
                req = required.get(matv, 0)
                if req <= 0 or matv not in supply:
                    continue
                subset = supply[matv][mix * req:(mix + 1) * req]
                for pkt in subset:
                    r0 = pkt_row.get(pkt, {})
                    rows.append({
                        "Artikel": r0.get("Artikel"),
                        "Beschreibung": r0.get("Beschreibung"),
                        "Lagerplatz": r0.get("Lagerplatz"),
                        "AV_LB%": r0.get("AV_LB%"),
                        "Material": matv,
                        "Packet": pkt,
                        "Mixture": mix + 1
                    })

        sc = pd.DataFrame(rows)
        if not sc.empty:
            sc["PacketIndex"] = sc.groupby(["Mixture", "Material"]).cumcount()
        return sc

    supply_chain_df = generate_supply_chain_exact(candidate_recipe)

    qc = (
        solver_bins.groupby(["Material_Display", "dyn_bin"], as_index=False)
        .agg({
            "Menge": "sum",
            "AV_LB%": lambda s: np.average(s, weights=solver_bins.loc[s.index, "Menge"]),
            "AV_BulkDensity": lambda s: (
                lambda vals, wts: float(np.average(vals, weights=wts)) if len(vals) > 0 else np.nan
            )(
                vals=solver_bins.loc[s.index, "AV_BulkDensity"][~solver_bins.loc[s.index, "AV_BulkDensity"].isna()].astype(float),
                wts=solver_bins.loc[s.index, "Menge"][~solver_bins.loc[s.index, "AV_BulkDensity"].isna()].astype(float),
            ),
            "AV_Lightness": lambda s: (
                lambda vals, wts: float(np.average(vals, weights=wts)) if len(vals) > 0 else np.nan
            )(
                vals=solver_bins.loc[s.index, "AV_Lightness"][~solver_bins.loc[s.index, "AV_Lightness"].isna()].astype(float),
                wts=solver_bins.loc[s.index, "Menge"][~solver_bins.loc[s.index, "AV_Lightness"].isna()].astype(float),
            ),
            "Charge/ Paket": lambda lists: sum(len(l) for l in lists)
        })
        .rename(columns={
            "Material_Display": "Material",
            "Menge": "Total_Stock",
            "AV_LB%": "Weighted_LB%",
            "AV_BulkDensity": "Weighted_BulkDensity",
            "AV_Lightness": "Weighted_Lightness",
            "Charge/ Paket": "BB_Count"
        })
    )

    group_order_map = {
        "Clear Neutral": 0,
        "Clearlightblue Low": 1,
        "Clearlightblue High": 2,
        "Lightgrey": 3,
        "Lightblue": 4
    }
    qc["GroupOrder"] = qc["Material"].map(group_order_map).fillna(99).astype(int)
    qc["DynOrder"] = qc["dyn_bin"].astype(str).str.extract(r"(\d+)")[0].astype(float).fillna(999).astype(int)
    qc = qc.sort_values(["GroupOrder", "DynOrder", "Material", "dyn_bin"]).reset_index(drop=True)

    tot = float(qc["Total_Stock"].sum())
    wlb = float((qc["Total_Stock"] * qc["Weighted_LB%"]).sum() / tot) if tot else 0.0
    BBs = int(qc["BB_Count"].sum())
    wbd_tot = float((qc["Total_Stock"] * qc["Weighted_BulkDensity"].fillna(0)).sum() / tot) if tot and not qc["Weighted_BulkDensity"].isna().all() else np.nan
    wlt_tot = float((qc["Total_Stock"] * qc["Weighted_Lightness"].fillna(0)).sum() / tot) if tot and not qc["Weighted_Lightness"].isna().all() else np.nan

    total_row = pd.DataFrame({
        "Material": ["Total"], "dyn_bin": [""], "Total_Stock": [tot],
        "Weighted_LB%": [wlb], "Weighted_BulkDensity": [wbd_tot], "Weighted_Lightness": [wlt_tot],
        "BB_Count": [BBs], "GroupOrder": [99], "DynOrder": [999]
    })
    qc2 = pd.concat([qc, total_row], ignore_index=True)

    mm = solver_bins.copy()
    def _fmt_ranges(p):
        return ", ".join(find_ranges(p)) if isinstance(p, (list, tuple)) else ""

    mm["Charge/ Paket"] = mm["Charge/ Paket"].apply(_fmt_ranges)
    mm["Material"] = mm["Material_Display"]
    mm["GroupOrder"] = mm["Material"].map(group_order_map).fillna(99).astype(int)
    mm["DynOrder"] = mm["dyn_bin"].astype(str).str.extract(r"(\d+)")[0].astype(float).fillna(999).astype(int)
    mm = mm.sort_values(["GroupOrder", "DynOrder", "Material", "dyn_bin", "AV_LB%"], ignore_index=True)
    mm = mm[[
        "Material", "dyn_bin", "Beschreibung", "Artikel", "Lagerplatz", "AV_LB%",
        "AV_BulkDensity", "AV_Lightness", "Menge", "Quality", "Charge/ Paket",
        "CLB_LOW_anchor_start", "CLB_LOW_anchor_end", "CLB_LOW_in_anchor",
        "CLB_HIGH_anchor_start", "CLB_HIGH_anchor_end", "CLB_HIGH_in_anchor",
        "LG_anchor_start", "LG_anchor_end", "LG_in_anchor",
        "SOLVER_EXCLUDE", "SOLVER_EXCLUDE_REASON", "LB_is_overflow"
    ]]

    def round_for_display(df):
        for col in ["AV_LB%", "Weighted_LB%"]:
            if col in df.columns:
                df[col] = df[col].clip(upper=99).round(1)
        for col in ["AV_Lightness", "Weighted_Lightness"]:
            if col in df.columns:
                df[col] = df[col].round(1)
        for col in ["AV_BulkDensity", "Weighted_BulkDensity"]:
            if col in df.columns:
                df[col] = df[col].round(0)
        return df

    qc2 = round_for_display(qc2)
    mm = round_for_display(mm)
    candidate_recipe_display = round_for_display(candidate_recipe.copy()) if not candidate_recipe.empty else pd.DataFrame()

    # =========================================================
    # Late-stage split based ONLY on final shown recipe-table rows
    # =========================================================
    final_recipe_display_full = prepare_final_recipe_display(candidate_recipe_display)
    candidate_recipe_display_main, reserve_bc_display = split_final_display_for_reserve_bc(
        final_recipe_display_full,
        shown_amount_threshold=5
    )

    output_file = f"{recipe_date.isoformat()}_{choice}_Rezept_{PRODUCTION_MODE}.xlsx"
    with pd.ExcelWriter(output_file, engine="xlsxwriter", engine_kwargs={"options": {"nan_inf_to_errors": True}}) as writer:
        wb = writer.book

        ws1 = wb.add_worksheet("Candidate_Recipe")
        writer.sheets["Candidate_Recipe"] = ws1

        write_recipe_layout_sheet(
            ws=ws1,
            df_display=candidate_recipe_display_main,
            wb=wb,
            recipe_date=recipe_date,
            choice=choice,
            production_mode=PRODUCTION_MODE,
            header_stock=header_stock,
            header_corr=header_corr,
            hidden_mats=hidden_mats,
            unused_mats_for_hide=unused_mats_for_hide,
            apply_hidden_candidate_rules=True
        )

        ws2 = wb.add_worksheet("Stock Summary")
        writer.sheets["Stock Summary"] = ws2
        ws2.write(0, 0, f"QC Approved Stock Summary (Post-Move Bins) – Modus {PRODUCTION_MODE}", wb.add_format({"bold": True, "font_size": 14}))

        qc2.to_excel(writer, sheet_name="Stock Summary", startrow=1,
                     columns=["Material", "dyn_bin", "Total_Stock", "Weighted_LB%", "Weighted_BulkDensity", "Weighted_Lightness", "BB_Count"],
                     index=False)

        start = len(qc2) + 4
        ws2.write(start - 1, 0, "Detailed Material Stock (Post-Move Bins)", wb.add_format({"bold": True, "font_size": 14}))
        mm.to_excel(writer, sheet_name="Stock Summary", startrow=start, index=False)

        ws3 = wb.add_worksheet("Supply_Chain")
        writer.sheets["Supply_Chain"] = ws3
        if supply_chain_df is not None and (not supply_chain_df.empty):
            supp = supply_chain_df.copy()
            order_vis = ["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"]
            supp["MatOrder"] = supp["Material"].map({m: i for i, m in enumerate(order_vis)})
            supp.sort_values(["Mixture", "MatOrder", "PacketIndex"], inplace=True)
            supp.drop(columns=["MatOrder", "PacketIndex"], errors="ignore", inplace=True)
            supp.to_excel(writer, sheet_name="Supply_Chain", startrow=0, index=False)
            rows_sc, cols_sc = supp.shape
            ws3.add_table(0, 0, rows_sc, cols_sc - 1, {"columns": [{"header": h} for h in supp.columns], "name": "SupplyChainTable"})

        ws4 = wb.add_worksheet(RESERVE_BC_SHEET_NAME)
        writer.sheets[RESERVE_BC_SHEET_NAME] = ws4
        write_recipe_layout_sheet(
            ws=ws4,
            df_display=reserve_bc_display,
            wb=wb,
            recipe_date=recipe_date,
            choice=choice,
            production_mode=PRODUCTION_MODE,
            header_stock=header_stock,
            header_corr=header_corr,
            hidden_mats=None,
            unused_mats_for_hide=None,
            apply_hidden_candidate_rules=False
        )

        # Only Candidate_Recipe visible
        ws2.hide()
        ws3.hide()
        ws4.hide()

    files.download(output_file)
    print("Done.")

# =========================================================
# UI: button + wiring (ONLY Date, Product, Mode)
# =========================================================
generate_button = widgets.Button(
    description="Generate recipe",
    button_style="success",
    tooltip="Generate recipe with current settings"
)

def on_generate_clicked(b):
    clear_output()
    display(date_picker, product_dd, mode_dd, generate_button)
    generate_recipe()

generate_button.on_click(on_generate_clicked)

display(date_picker, product_dd, mode_dd, generate_button)

Text(value='2026-03-13', description='Recipe Date:', style=DescriptionStyle(description_width='initial'))

Dropdown(description='Product:', options=(('C100', 'C100'), ('C102', 'C102'), ('C103', 'C103')), style=Descrip…

Dropdown(description='Production Mode:', options=(('Standard', 'STANDARD'), ('RecyClass', 'RECYCLASS'), ('A1 o…

Button(button_style='success', description='Generate recipe', style=ButtonStyle(), tooltip='Generate recipe wi…

Selected date: 2026-03-13
Selected product: C100 → LB 10.0±3.0
Selected mode: STANDARD → LT 2.050±0.250  (from L* 68.0–72.0)


Saving 2026-03-12 Lager.xlsx to 2026-03-12 Lager.xlsx
DEBUG build_scored: base_total=32764 base_no_clear=7438 base_with_clear=25326
DEBUG build_scored: feasible_total=0 feasible_no_clear=0 feasible_with_clear=0
ANCHOR FALLBACK scenarios queued=25
TRY FALLBACK: tag=FALLBACK_CLB_LOW_LOW_EDGE CLB_LOW=0–12 CLB_HIGH=31–41 LG=0–14 flags=CLB_LOW=LOW_EDGE
DEBUG build_scored: base_total=32764 base_no_clear=7438 base_with_clear=25326
DEBUG build_scored: feasible_total=21 feasible_no_clear=21 feasible_with_clear=0
DEBUG choose_best_candidate: candidates=21 no_clear=21 with_clear=0
FALLBACK RESULT: feasible -> FALLBACK_CLB_LOW_LOW_EDGE
TRY FALLBACK: tag=FALLBACK_CLB_LOW_HIGH_EDGE CLB_LOW=13–25 CLB_HIGH=31–41 LG=0–14 flags=CLB_LOW=HIGH_EDGE
DEBUG build_scored: base_total=32764 base_no_clear=7438 base_with_clear=25326
DEBUG build_scored: feasible_total=0 feasible_no_clear=0 feasible_with_clear=0
FALLBACK RESULT: no feasible -> FALLBACK_CLB_LOW_HIGH_EDGE
TRY FALLBACK: tag=FALLBACK_CLB_HIGH_LOW_EDGE C

/tmp/ipykernel_379/1073944055.py:910: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(cr[c]):
/tmp/ipykernel_379/1073944055.py:930: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_reorder_block)
/tmp/ipykernel_379/1073944055.py:936: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(df_merge["Material_Vis"]):


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.


In [8]:
# @title rPET__Recipe v3.0 (Standard / RecyClass / A1-only)_Global weight

from datetime import datetime
import itertools, math, re, io

import numpy as np
import pandas as pd

from IPython.display import display, clear_output
from google.colab import files
import ipywidgets as widgets

# --- Excel writer dependency (robust in Colab) ---
try:
    import xlsxwriter  # noqa: F401
except Exception:
    !pip install -q xlsxwriter
    import xlsxwriter  # noqa: F401


# =========================================================
# UI widgets (ONLY Date, Product, Mode)
# =========================================================
date_picker = widgets.Text(
    value=datetime.today().date().isoformat(),
    description="Recipe Date:",
    style={"description_width": "initial"},
)

product_dd = widgets.Dropdown(
    options=[("C100", "C100"), ("C102", "C102"), ("C103", "C103")],
    value="C100",
    description="Product:",
    style={"description_width": "initial"},
)

mode_dd = widgets.Dropdown(
    options=[("Standard", "STANDARD"), ("RecyClass", "RECYCLASS"), ("A1 only", "A1_ONLY")],
    value="STANDARD",
    description="Production Mode:",
    style={"description_width": "initial"},
)


# =========================================================
# Product + Mode specs (single source of truth)
# =========================================================
PRODUCT_SPECS = {
    # product: (LB_target, LB_tol, b*_min, b*_max)
    "C100": (10.0, 3.0, -3.0,  0.0),
    "C103": ( 5.0, 3.0,  0.0,  1.0),
    "C102": (15.0, 3.0, -5.0, -3.0),
}

MODE_LSTAR_RANGE = {
    "STANDARD":  (68.0, 72.0),
    "RECYCLASS": (67.0, 70.0),
    "A1_ONLY":   (66.0, 69.0),
}

# L* -> LT mapping anchor:
# L*=72 -> LT=1.80, L*=68 -> LT=2.30 (linear)
def lt_from_Lstar(L):
    L = float(L)
    return 1.8 + (72.0 - L) * 0.125

def get_recipe_specs(product, mode):
    lb_target, lb_tol, bmin, bmax = PRODUCT_SPECS[product]

    Lmin, Lmax = MODE_LSTAR_RANGE[mode]
    lt_min = lt_from_Lstar(Lmax)
    lt_max = lt_from_Lstar(Lmin)
    lt_target = 0.5 * (lt_min + lt_max)
    lt_tol    = 0.5 * (lt_max - lt_min)

    return {
        "lb_target": float(lb_target),
        "lb_tol": float(lb_tol),
        "b_range": (float(bmin), float(bmax)),
        "L_range": (float(Lmin), float(Lmax)),
        "lt_target": float(lt_target),
        "lt_tol": float(lt_tol),
    }

# =========================================================
# Global constants / tunables
# =========================================================
MIN_TOTAL_SOFT = 4
MIN_TOTAL_HARD = 4
MAX_TOTAL      = 6.7

MIN_MIXES = 50
MIXES_PREF = 40
MAX_NONZERO_BINS = 5
EPS = 1e-9

# Solver lanes:
#   clear
#   clb_low        (2 .. <25)
#   clb_high       (25 .. <50)
#   lightgrey
#   highlight_use
STEP_CLEAR    = 0.5
STEP_CLB_LOW  = 0.5
STEP_CLB_HIGH = 0.5
STEP_LG       = 0.5
STEP_HL       = 0.050

VISIBLE_BB_LIMIT = 200.0

BD_WARN_LOW  = 220.0
BD_WARN_HIGH = 270.0

# ======= Lightblue / Darkblue physics tunables =======
DARKBLUE_FACTOR_BASE = 2.5
DARKBLUE_FACTOR_HL   = 2.5

# ======= Real lane average mass fallback =======
DEFAULT_TONS_PER_BB = 1.0

# Caps for candidate enumeration (BB units)
UPPER_CAP = {
    "clear":          200.0,
    "clb_low":        200.0,
    "clb_high":       200.0,
    "lightgrey":      200.0,
    "highlight_use":  200.0,
}

# Artikels to exclude early
EXCLUDE_ARTIKELS = {
    1100138, 1101019, 1000047, 1000048,
    1000057, 1100031, 1001049
}

# Solver search grid bound
MAX_GUESS_PER_GROUP = 20

# ======= LG (Lightgrey) / darkness tunables =======
DARK_LT_THRESHOLD = 3.0
DARK_TEXT_TOKENS  = {"03", "04", "dunkel", "dark", "lg", "lightgrey", "grau"}

# ======= Legacy-style anchor tunables =======
ANCHOR_WIDTH_CLB_LOW  = 10.0
ANCHOR_WIDTH_CLB_HIGH = 10.0
ANCHOR_WIDTH_LG       = 14.0

CLB_LOW_MIN   = 2.0
CLB_LOW_MAX   = 25.0
CLB_LOW_STEP  = 2.0

CLB_HIGH_MIN  = 25.0
CLB_HIGH_MAX  = 50.0
CLB_HIGH_STEP = 2.0

LG_MIN   = 0.0
LG_MAX   = 25.0
LG_STEP  = 2.0

# =========================================================
# RecyClass / A1 allowlists (single source of truth)
# =========================================================
RECYCLASS_REF = pd.DataFrame([
    {"Artikel": 1000018, "Recyclass_label": "Ohne"},
    {"Artikel": 1000020, "Recyclass_label": "Ohne"},
    {"Artikel": 1000049, "Recyclass_label": "Ohne"},
    {"Artikel": 1000127, "Recyclass_label": "RecyClass"},
    {"Artikel": 1000126, "Recyclass_label": "RecyClass"},
    {"Artikel": 1000125, "Recyclass_label": "RecyClass"},
    {"Artikel": 1000118, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1000119, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1000120, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1001049, "Recyclass_label": "Ohne"},
    {"Artikel": 1000047, "Recyclass_label": "Ohne"},
    {"Artikel": 1000021, "Recyclass_label": "Ohne"},
    {"Artikel": 1000022, "Recyclass_label": "Ohne"},
    {"Artikel": 1000130, "Recyclass_label": "Ohne"},
    {"Artikel": 1000132, "Recyclass_label": "Ohne"},
    {"Artikel": 1000122, "Recyclass_label": "Ohne"},
    {"Artikel": 1100101, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1100094, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1100106, "Recyclass_label": "RecyClass A1"},
    {"Artikel": 1101003, "Recyclass_label": "Ohne"},
])

RECYCLASS_REF = RECYCLASS_REF.copy()
RECYCLASS_REF["Artikel"] = pd.to_numeric(RECYCLASS_REF["Artikel"], errors="coerce").astype("Int64")
RECYCLASS_REF["Recyclass_label"] = RECYCLASS_REF["Recyclass_label"].astype(str).str.strip()

A1_ALLOWED = set(
    RECYCLASS_REF.loc[RECYCLASS_REF["Recyclass_label"].eq("RecyClass A1"), "Artikel"]
    .dropna().astype(int).tolist()
)

RECY_ALLOWED = set(
    RECYCLASS_REF.loc[RECYCLASS_REF["Recyclass_label"].isin(["RecyClass", "RecyClass A1"]), "Artikel"]
    .dropna().astype(int).tolist()
)

# =========================================================
# Runtime active specs
# =========================================================
PRODUCTION_MODE = "STANDARD"
PRODUCT_CODE    = "C100"
ACTIVE_SPECS    = get_recipe_specs(PRODUCT_CODE, PRODUCTION_MODE)

target_lb = float(ACTIVE_SPECS["lb_target"])
LB_TOL    = float(ACTIVE_SPECS["lb_tol"])
lt_target = float(ACTIVE_SPECS["lt_target"])
LT_TOL    = float(ACTIVE_SPECS["lt_tol"])

def apply_active_specs_from_widgets():
    global PRODUCTION_MODE, PRODUCT_CODE, ACTIVE_SPECS
    global target_lb, LB_TOL, lt_target, LT_TOL

    PRODUCT_CODE    = str(product_dd.value)
    PRODUCTION_MODE = str(mode_dd.value)

    ACTIVE_SPECS = get_recipe_specs(PRODUCT_CODE, PRODUCTION_MODE)

    target_lb = float(ACTIVE_SPECS["lb_target"])
    LB_TOL    = float(ACTIVE_SPECS["lb_tol"])
    lt_target = float(ACTIVE_SPECS["lt_target"])
    LT_TOL    = float(ACTIVE_SPECS["lt_tol"])

# =========================================================
# Utilities
# =========================================================
def find_ranges(packets):
    out = []
    grouped = {}
    pat = re.compile(r"^(.*-)(\d+)$")

    for pkt in packets:
        s = str(pkt)
        m = pat.match(s)
        if not m:
            grouped.setdefault(s, []).append(None)
            continue
        p, n = m.groups()
        grouped.setdefault(p, []).append(int(n))

    for p, nums in grouped.items():
        if nums == [None]:
            out.append(p.rstrip('-'))
            continue

        numeric_suffixes = [x for x in nums if isinstance(x, int)]
        if not numeric_suffixes:
            out.append(p.rstrip('-'))
            continue

        numeric_suffixes = sorted(set(numeric_suffixes))
        prefix_base = p.rstrip('-')
        start = end = numeric_suffixes[0]

        for x in numeric_suffixes[1:]:
            if x == end + 1:
                end = x
            else:
                if start == end:
                    out.append(f"{p}{start}")
                else:
                    out.append(f"{prefix_base} ({start} bis {end})")
                start = end = x

        if start == end:
            out.append(f"{p}{start}")
        else:
            out.append(f"{prefix_base} ({start} bis {end})")

    return out

def _to_numeric(series):
    s = series.astype(str).str.replace(',', '.', regex=False)
    s = s.str.replace(r'[^\d\.\-]+', '', regex=True)
    return pd.to_numeric(s, errors='coerce')

def compute_result_lb(picks, lb_means):
    tot = sum(picks.values())
    if tot <= EPS:
        return float('nan')
    return sum(picks.get(g, 0.0) * lb_means.get(g, 0.0) for g in picks) / tot

def compute_result_lt(picks, lt_means):
    tot = sum(picks.values())
    if tot <= EPS:
        return float('nan')

    acc, w = 0.0, 0.0
    for g, v in picks.items():
        lt = lt_means.get(g, None)
        if lt is None or (isinstance(lt, float) and np.isnan(lt)):
            continue
        acc += v * lt
        w += v
    return acc / w if w > 0 else float('nan')

def parse_bulk_density(x):
    if x is None:
        return np.nan

    s = str(x).strip()
    s = s.replace(",", ".")
    s = re.sub(r"[^0-9.\-eE+]", "", s)

    try:
        v = float(s)
    except Exception:
        return np.nan

    if np.isnan(v):
        return np.nan

    if v < 5.0:
        return v * 1000.0
    return v

def compute_result_bd(picks, bd_means):
    tot = sum(picks.values())
    if tot <= EPS:
        return float("nan")

    acc, w = 0.0, 0.0
    for g, v in picks.items():
        bd = bd_means.get(g, None)
        if bd is None or (isinstance(bd, float) and np.isnan(bd)):
            continue
        acc += v * bd
        w += v
    return acc / w if w > 0 else float("nan")

def compute_full_mixes(picks, stock_caps):
    limits = []
    for g, v in picks.items():
        if v <= EPS:
            continue
        avail = stock_caps.get(g, 0.0)
        if avail <= EPS:
            return 0
        limits.append(math.floor(avail / v))
    return int(min(limits)) if limits else 0

def bb_to_display(amount_bb, group_key):
    if amount_bb is None or amount_bb <= EPS:
        return None

    if group_key == 'highlight_use':
        kg = round(amount_bb * 1000)
        return f"{kg} kg"

    x2 = amount_bb * 2.0
    if abs(x2 - round(x2)) < 1e-9:
        if abs(amount_bb - round(amount_bb)) < 1e-9:
            if int(round(amount_bb)) == 1:
                return "1 BB"
            return f"{int(round(amount_bb))} BB"
        return f"{amount_bb:.1f} BB"

    kg = round(amount_bb * 1000)
    return f"{kg} kg"

# =========================================================
# Legacy anchor helpers
# =========================================================
def _build_legacy_anchor_weights(df_lane):
    if df_lane.empty:
        return df_lane.copy()

    tmp = df_lane.copy()

    tmp["__w_mass"] = pd.to_numeric(tmp["bb"], errors="coerce").fillna(0.0).astype(float)
    if float(tmp["__w_mass"].sum()) <= EPS:
        tmp["__w_mass"] = pd.to_numeric(tmp["Menge"], errors="coerce").fillna(0.0).astype(float)

    tmp["__w_lt"] = pd.to_numeric(tmp["Menge"], errors="coerce").fillna(0.0).astype(float)
    if float(tmp["__w_lt"].sum()) <= EPS:
        tmp["__w_lt"] = tmp["__w_mass"]

    tmp["lb_pct"] = pd.to_numeric(tmp["lb_pct"], errors="coerce")
    tmp["lt_num"] = pd.to_numeric(tmp.get("lt_num", np.nan), errors="coerce")
    return tmp

def _scan_legacy_anchor_window(df_lane, start_min, end_max, width, step):
    if df_lane.empty:
        return None

    tmp = _build_legacy_anchor_weights(df_lane)
    best = None

    starts = []
    s = float(start_min)
    while s + float(width) <= float(end_max) + 1e-9:
        starts.append(round(s, 6))
        s += float(step)

    for start in starts:
        end = round(start + float(width), 6)
        mask = tmp["lb_pct"].ge(float(start)) & tmp["lb_pct"].lt(float(end))
        cand = tmp.loc[mask].copy()
        if cand.empty:
            continue

        massW = float(pd.to_numeric(cand["__w_mass"], errors="coerce").fillna(0.0).sum())
        if massW <= EPS:
            continue

        lt_vals = pd.to_numeric(cand["lt_num"], errors="coerce")
        lt_w = pd.to_numeric(cand["__w_lt"], errors="coerce").fillna(0.0).astype(float)
        lt_mask = lt_vals.notna() & lt_w.gt(EPS)

        if bool(lt_mask.any()):
            lt_mean = float((lt_vals[lt_mask].astype(float) * lt_w[lt_mask]).sum() / max(float(lt_w[lt_mask].sum()), EPS))
            lt_err = abs(float(lt_mean) - float(lt_target))
        else:
            lt_err = float("inf")

        cand_key = (massW, -lt_err, -start, -end)
        if best is None or cand_key > best["cmp"]:
            best = {
                "start": float(start),
                "end": float(end),
                "massW": float(massW),
                "lt_err": float(lt_err),
                "cmp": cand_key,
            }

    return None if best is None else {k: v for k, v in best.items() if k != "cmp"}

def _choose_clb_low_anchor_window(df_clb_low):
    return _scan_legacy_anchor_window(
        df_clb_low,
        start_min=CLB_LOW_MIN,
        end_max=CLB_LOW_MAX,
        width=ANCHOR_WIDTH_CLB_LOW,
        step=CLB_LOW_STEP,
    )

def _choose_clb_high_anchor_window(df_clb_high):
    return _scan_legacy_anchor_window(
        df_clb_high,
        start_min=CLB_HIGH_MIN,
        end_max=CLB_HIGH_MAX,
        width=ANCHOR_WIDTH_CLB_HIGH,
        step=CLB_HIGH_STEP,
    )

def _choose_lg_anchor_window(df_lg):
    return _scan_legacy_anchor_window(
        df_lg,
        start_min=LG_MIN,
        end_max=LG_MAX,
        width=ANCHOR_WIDTH_LG,
        step=LG_STEP,
    )

def _append_reason_col(series, reason):
    s = series.fillna("").astype(str)
    return np.where(s.eq(""), str(reason), s + "|" + str(reason))

# =========================================================
# Stock stats
# =========================================================
def compute_group_stats_for_solver(df):
    grp = df.groupby("group", as_index=False).agg(
        bb=("bb", "sum"),
        wlb=("lb_pct", lambda s: (df.loc[s.index, "bb"] * s).sum() / max(df.loc[s.index, "bb"].sum(), 1e-9)),
        wbd=("AV_BulkDensity",
             lambda s: (df.loc[s.index, "Menge"] * df.loc[s.index, "AV_BulkDensity"]).sum() /
                       max(df.loc[s.index, "Menge"].sum(), 1e-9)
             if not df.loc[s.index, "AV_BulkDensity"].isna().all() else np.nan),
        wlt=("AV_Lightness",
             lambda s: (df.loc[s.index, "Menge"] * df.loc[s.index, "AV_Lightness"]).sum() /
                       max(df.loc[s.index, "Menge"].sum(), 1e-9)
             if not df.loc[s.index, "AV_Lightness"].isna().all() else np.nan),
    )

    present = {r["group"]: float(r["bb"]) for _, r in grp.iterrows() if r["bb"] > EPS}
    lb_means = {r["group"]: float(r["wlb"]) for _, r in grp.iterrows()}
    bd_means = {r["group"]: (None if pd.isna(r["wbd"]) else float(r["wbd"])) for _, r in grp.iterrows()}
    lt_means = {r["group"]: (None if pd.isna(r["wlt"]) else float(r["wlt"])) for _, r in grp.iterrows()}

    base_no_hl = sum(v for g, v in present.items() if g != "highlight_use")
    shares_no_hl = {g: (present[g] / base_no_hl if base_no_hl > 0 else 0.0)
                    for g in present if g != "highlight_use"}

    return (present, lb_means, shares_no_hl, bd_means, lt_means)

def _lane_bb_tonnage_map(df_eff):
    out = {}
    for g in ["clear", "clb_low", "clb_high", "lightgrey", "highlight_use"]:
        lane = df_eff[df_eff["group"].eq(g)].copy()
        if lane.empty:
            out[g] = DEFAULT_TONS_PER_BB
            continue

        tons_total = float(pd.to_numeric(lane.get("Menge", 0.0), errors="coerce").fillna(0.0).sum())
        bb_total = float(pd.to_numeric(lane.get("bb", 0.0), errors="coerce").fillna(0.0).sum())

        if tons_total > EPS and bb_total > EPS:
            out[g] = tons_total / bb_total
        else:
            out[g] = DEFAULT_TONS_PER_BB

    return out

# =========================================================
# Dynamic stock assessment & min batch size logic
# =========================================================
def get_effective_min_total(stock_caps, demand_mixes_floor=10):
    return MIN_TOTAL_HARD

# =========================================================
# Candidate enumeration helpers
# =========================================================
def generate_grid_values(group_name, step, cap_limit, cap_stock, max_guess):
    vals = [0.0]
    k = 1
    while True:
        v = round(k * step, 6)
        if v > cap_limit + 1e-9:
            break
        if v > cap_stock + 1e-9:
            break
        vals.append(v)
        k += 1
        if len(vals) >= max_guess + 1:
            break
    return vals

def enumerate_candidates_base(groups_present, uppers, max_guess_per_group, min_total_dynamic):
    keys_core = [
        g for g in ["clear", "clb_low", "clb_high", "lightgrey", "highlight_use"]
        if groups_present.get(g, 0.0) > 0.0
    ]
    if not keys_core:
        return []

    step_map = {
        "clear": STEP_CLEAR,
        "clb_low": STEP_CLB_LOW,
        "clb_high": STEP_CLB_HIGH,
        "lightgrey": STEP_LG,
        "highlight_use": STEP_HL,
    }
    grids = {
        g: generate_grid_values(
            g,
            step_map[g],
            uppers.get(g, float("inf")),
            groups_present.get(g, 0.0),
            max_guess_per_group
        )
        for g in keys_core
    }

    out = []
    for tpl in itertools.product(*[grids[g] for g in keys_core]):
        pick = dict(zip(keys_core, tpl))
        total_bb = sum(pick.values())

        if total_bb < min_total_dynamic - 1e-9 or total_bb > MAX_TOTAL + 1e-9:
            continue

        nz_nonhl = sum(1 for gg, vv in pick.items() if gg != "highlight_use" and vv > EPS)
        if nz_nonhl > MAX_NONZERO_BINS:
            continue

        out.append(pick)

    return out

# =========================================================
# Highlight tuning / feasibility
# =========================================================
def nudge_highlight_for_lb(target_lb_local, pick, lb_means, caps, max_iter=200):
    cur = dict(pick)
    cur.setdefault("highlight_use", 0.0)

    for _ in range(max_iter):
        lb_now = compute_result_lb(cur, lb_means)
        if np.isnan(lb_now):
            break

        diff = lb_now - float(target_lb_local)
        if abs(diff) <= float(LB_TOL) + 1e-9:
            break

        if diff > 0:
            if cur["highlight_use"] > EPS:
                cur["highlight_use"] = round(max(0.0, cur["highlight_use"] - STEP_HL), 6)
            else:
                break
        else:
            new_hl = cur["highlight_use"] + STEP_HL
            if new_hl <= caps.get("highlight_use", float("inf")) + 1e-9:
                cur["highlight_use"] = round(new_hl, 6)
            else:
                break

        total_now = sum(cur.values())
        if total_now > MAX_TOTAL + 1e-9:
            for g_try in ["clear", "clb_low", "clb_high", "lightgrey"]:
                if cur.get(g_try, 0.0) >= 0.5 - 1e-9:
                    cur[g_try] = round(cur[g_try] - 0.5, 6)
                    if cur[g_try] < EPS:
                        cur[g_try] = 0.0
                    break

    return cur

def feasible_sequence(base_pick):
    tuned = nudge_highlight_for_lb(target_lb, base_pick, group_lb_means, stock_caps)

    total_bb = sum(tuned.values())
    if total_bb < effective_min_total - 1e-9 or total_bb > MAX_TOTAL + 1e-9:
        return None

    nz_nonhl = sum(1 for gg, vv in tuned.items() if gg != "highlight_use" and vv > EPS)
    if nz_nonhl > MAX_NONZERO_BINS:
        return None

    lb_final = compute_result_lb(tuned, group_lb_means)
    if np.isnan(lb_final) or abs(lb_final - float(target_lb)) > float(LB_TOL) + 1e-9:
        return None

    lt_final = compute_result_lt(tuned, group_lt_means)
    if lt_final is None or (isinstance(lt_final, float) and np.isnan(lt_final)):
        return None
    if abs(float(lt_final) - float(lt_target)) > float(LT_TOL) + 1e-9:
        return None

    bd_final = compute_result_bd(tuned, group_bd_means)

    mixes_final = compute_full_mixes(tuned, stock_caps)
    if int(mixes_final) < int(MIN_MIXES):
        return None

    return {
        "picks": tuned,
        "mixes": mixes_final,
        "lb": lb_final,
        "bd": bd_final,
        "lt": float(lt_final),
        "total_bb": total_bb
    }

def build_scored_candidates_with_priority():
    base_list = enumerate_candidates_base(groups_present, upper_cap_local, MAX_GUESS_PER_GROUP, effective_min_total)

    try:
        base_no_clear = sum(1 for p in base_list if float(p.get("clear", 0.0)) <= EPS)
        base_with_clear = sum(1 for p in base_list if float(p.get("clear", 0.0)) > EPS)
        print(
            "DEBUG build_scored:",
            f"base_total={len(base_list)}",
            f"base_no_clear={base_no_clear}",
            f"base_with_clear={base_with_clear}"
        )
    except Exception:
        pass

    scored = []
    for base in base_list:
        cand = feasible_sequence(base)
        if cand is not None:
            scored.append(cand)

    try:
        scored_no_clear = sum(1 for r in scored if float(r.get("picks", {}).get("clear", 0.0)) <= EPS)
        scored_with_clear = sum(1 for r in scored if float(r.get("picks", {}).get("clear", 0.0)) > EPS)
        print(
            "DEBUG build_scored:",
            f"feasible_total={len(scored)}",
            f"feasible_no_clear={scored_no_clear}",
            f"feasible_with_clear={scored_with_clear}"
        )
    except Exception:
        pass

    return scored

# =========================================================
# Choose best candidate
# =========================================================
def choose_best_candidate(scored):
    if not scored:
        return None

    df = pd.DataFrame(scored).copy()
    df["lb_err"] = (df["lb"] - float(target_lb)).abs()
    df["lt_err"] = (df["lt"] - float(lt_target)).abs()

    df = df[df["lb_err"] <= float(LB_TOL) + 1e-9].copy()
    if df.empty:
        return None

    df = df[df["lt"].notna()].copy()
    if df.empty:
        return None
    df = df[df["lt_err"] <= float(LT_TOL) + 1e-9].copy()
    if df.empty:
        return None

    df = df[df["mixes"].astype(int) >= int(MIN_MIXES)].copy()
    if df.empty:
        return None

    df["total_bb"] = df["total_bb"].astype(float)
    df = df[(df["total_bb"] >= MIN_TOTAL_SOFT - 1e-9) & (df["total_bb"] <= MAX_TOTAL + 1e-9)].copy()
    if df.empty:
        return None
    try:
        df["uses_clear"] = df["picks"].apply(lambda p: float(p.get("clear", 0.0)) > EPS)
        print(
            "DEBUG choose_best_candidate:",
            f"candidates={len(df)}",
            f"no_clear={int((~df['uses_clear']).sum())}",
            f"with_clear={int(df['uses_clear'].sum())}"
        )
    except Exception:
        pass

    NONHL_LANES = ["clear", "clb_low", "clb_high", "lightgrey"]

    df_gate_base = df.copy()
    df = df_gate_base

    def _is_integer_bb(v):
        v = float(v)
        if v <= EPS:
            return True
        return abs(v - round(v)) < 1e-9

    def _all_nonhl_integer(picks):
        for g in NONHL_LANES:
            if not _is_integer_bb(picks.get(g, 0.0)):
                return False
        return True

    def _half_count_nonhl(picks):
        c = 0
        for g in NONHL_LANES:
            v = float(picks.get(g, 0.0))
            if v <= EPS:
                continue
            if abs(v - round(v)) < 1e-9:
                continue
            if abs(v * 2.0 - round(v * 2.0)) < 1e-9:
                c += 1
            else:
                c += 10
        return c

    df["all_integer"] = df["picks"].apply(_all_nonhl_integer)
    df["half_count"] = df["picks"].apply(_half_count_nonhl)

    stock_clear = float(stock_shares_no_hl_all.get("clear", 0.0))
    stock_clb_low = float(stock_shares_no_hl_all.get("clb_low", 0.0))
    stock_clb_high = float(stock_shares_no_hl_all.get("clb_high", 0.0))
    stock_lg = float(stock_shares_no_hl_all.get("lightgrey", 0.0))

    def stock_balance_penalty(picks):
        non_hl_total = float(
            picks.get("clear", 0.0) +
            picks.get("clb_low", 0.0) +
            picks.get("clb_high", 0.0) +
            picks.get("lightgrey", 0.0)
        )
        if non_hl_total <= EPS:
            return 1e6
        cand_clear = float(picks.get("clear", 0.0)) / non_hl_total
        cand_clb_low = float(picks.get("clb_low", 0.0)) / non_hl_total
        cand_clb_high = float(picks.get("clb_high", 0.0)) / non_hl_total
        cand_lg = float(picks.get("lightgrey", 0.0)) / non_hl_total
        return (
            abs(cand_clear - stock_clear) +
            abs(cand_clb_low - stock_clb_low) +
            abs(cand_clb_high - stock_clb_high) +
            abs(cand_lg - stock_lg)
        )

    def bottleneck_penalty(mixes_val):
        m = float(mixes_val)
        if m >= float(MIXES_PREF):
            return 0.0
        return ((float(MIXES_PREF) - m) / max(float(MIXES_PREF), 1.0)) ** 2

    def _is_half_step(v):
        return (v > EPS) and (abs(v * 2.0 - round(v * 2.0)) < 1e-9) and (abs(v - round(v)) > 1e-9)

    def fractional_penalty(picks):
        if PRODUCTION_MODE == "A1_ONLY":
            return 0.0

        half_count = 0
        half_in_lg = 0

        for g in ["clear", "clb_low", "clb_high", "lightgrey"]:
            v = float(picks.get(g, 0.0))
            if v <= EPS:
                continue

            if abs(v - round(v)) < 1e-9:
                continue

            if _is_half_step(v):
                half_count += 1
                if g == "lightgrey":
                    half_in_lg += 1
            else:
                return 1e6

        base = 500.0 * max(0, half_count - 1) + 50.0 * half_count
        base += 800.0 * half_in_lg
        return base

    df["fractional_penalty"] = df["picks"].apply(fractional_penalty)
    df["stock_balance_penalty"] = df["picks"].apply(stock_balance_penalty)
    df["bottleneck_penalty"] = df["mixes"].apply(bottleneck_penalty)

    df_sorted = df.sort_values(
        [
            "fractional_penalty",
            "stock_balance_penalty",
            "lb_err",
            "lt_err",
            "all_integer",
            "half_count",
            "total_bb",
            "mixes",
            "bottleneck_penalty",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            True,
            False,
            False,
            True,
        ],
        kind="stable"
    )
    if df_sorted.empty:
        return None

    pick = df_sorted.iloc[0]
    return {
        "picks": dict(pick["picks"]),
        "result_lb": float(pick["lb"]),
        "result_bd": (float(pick["bd"]) if not (pd.isna(pick["bd"])) else float("nan")),
        "result_lt": float(pick["lt"]),
        "mixes": int(pick["mixes"]),
        "single_lane": True,
        "lt_target": float(lt_target),
        "lt_tol": float(LT_TOL),
    }

# =========================================================
# MAIN PIPELINE
# =========================================================
def generate_recipe():
    global solver_bins, groups_present, group_lb_means, group_bd_means, group_lt_means
    global stock_caps, stock_shares_no_hl_all
    global upper_cap_local, effective_min_total, PRODUCTION_MODE
    global target_lb, LB_TOL, lt_target, LT_TOL
    global recipe_date, product, candidate_recipe

    apply_active_specs_from_widgets()

    choice = str(product_dd.value)
    PRODUCTION_MODE = str(mode_dd.value)
    recipe_date = datetime.strptime(str(date_picker.value), "%Y-%m-%d").date()

    specs = get_recipe_specs(choice, PRODUCTION_MODE)
    target_lb = float(specs["lb_target"])
    LB_TOL    = float(specs["lb_tol"])
    lt_target = float(specs["lt_target"])
    LT_TOL    = float(specs["lt_tol"])

    print(f"Selected date: {recipe_date.isoformat()}")
    print(f"Selected product: {choice} → LB {target_lb:.1f}±{LB_TOL:.1f}")
    print(f"Selected mode: {PRODUCTION_MODE} → LT {lt_target:.3f}±{LT_TOL:.3f}  (from L* {specs['L_range'][0]}–{specs['L_range'][1]})")

    uploaded = None
    try:
        uploaded = files.upload()
    except TypeError:
        print("File upload cancelled or failed.")
        return

    if not uploaded:
        print("No file uploaded; aborting.")
        return

    file_name = next(iter(uploaded))
    raw = pd.read_excel(file_name, sheet_name=0)

    col_map = {c: c for c in raw.columns}
    known_map = {
        "Paket/Chargennummer": "Charge/ Paket",
        "Menge [to}": "Menge",
        "Schüttdichte 1": "Schüttgewicht",
        "PET lightblue (hellblau)": "Lightblue",
        "PET Blue_Dark": "Darkblue",
        "Helligkeit": "Helligkeit",
        "Lagerplatz": "Lagerplatz",
        "Artikel": "Artikel",
        "Artikel ": "Artikel",
        "Quality": "Quality",
        "Quality ": "Quality",
        "Beschreibung": "Beschreibung",
        "Beschreibung ": "Beschreibung",
        "Qualität": "Quality",
        "BlueDark": "Darkblue",
        "Schüttgewicht": "Schüttgewicht",
        "Charge/ Paket": "Charge/ Paket",
        "Menge": "Menge",
        "Lightblue": "Lightblue",
        "Darkblue": "Darkblue",
    }
    for c in list(raw.columns):
        if c in known_map:
            col_map[c] = known_map[c]
    raw = raw.rename(columns=col_map)

    for req in ['Artikel', 'Beschreibung', 'Charge/ Paket', 'Menge', 'Lagerplatz', 'Lightblue', 'Darkblue']:
        if req not in raw.columns:
            raw[req] = None

    raw['Artikel_clean'] = pd.to_numeric(raw['Artikel'], errors='coerce')
    if raw['Artikel_clean'].isna().any():
        mask = raw['Artikel_clean'].isna()
        raw.loc[mask, 'Artikel_clean'] = np.arange(1000000, 1000000 + mask.sum())
    raw['Artikel'] = raw['Artikel_clean'].astype(int)
    raw.drop(columns=['Artikel_clean'], inplace=True)

    raw = raw.reset_index(drop=True).copy()
    raw["Row_ID"] = np.arange(1, len(raw) + 1, dtype=int)

    raw = raw[~raw["Artikel"].isin(EXCLUDE_ARTIKELS)].copy()

    def _has_fa_packet(cell):
        if cell is None or (isinstance(cell, float) and np.isnan(cell)):
            return False
        if isinstance(cell, (list, tuple, set)):
            items = cell
        elif isinstance(cell, str):
            items = [x.strip() for x in cell.split(",")] if "," in cell else [cell.strip()]
        else:
            items = [str(cell).strip()]
        for x in items:
            t = str(x).strip().upper()
            if not t or t in ("NAN", "NONE"):
                continue
            if t.startswith("FA"):
                return True
        return False

    raw["__is_A1_by_FA"] = raw["Charge/ Paket"].apply(_has_fa_packet)

    mode_key = str(PRODUCTION_MODE)
    if mode_key == "A1_ONLY":
        if not A1_ALLOWED:
            raise ValueError("A1_ALLOWED is empty. Refusing to run.")
        raw = raw[(raw["Artikel"].isin(A1_ALLOWED)) | (raw["__is_A1_by_FA"])].copy()
    elif mode_key == "RECYCLASS":
        if not RECY_ALLOWED:
            raise ValueError("RECY_ALLOWED is empty. Refusing to run.")
        raw = raw[(raw["Artikel"].isin(RECY_ALLOWED)) | (raw["__is_A1_by_FA"])].copy()
    elif mode_key == "STANDARD":
        pass
    else:
        raise ValueError(f"Unknown PRODUCTION_MODE: {mode_key}")

    for c in ['Lightblue', 'Darkblue']:
        raw[c] = raw[c].astype(str).replace('[>]', '', regex=True).replace(',', '.', regex=True)
        raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0.0)

    if 'BlueDark' in raw.columns and 'Darkblue' in raw.columns:
        raw['Darkblue'] = raw['Darkblue'].where(raw['Darkblue'].notna(), raw['BlueDark'])
        raw.drop(columns=[c for c in ['BlueDark'] if c in raw.columns], inplace=True)

    lb = raw['Lightblue'].astype(float)
    db = raw['Darkblue'].astype(float)

    hl_mask   = lb >= 50.0
    base_mask = ~hl_mask

    LB_phys = lb.copy()
    LB_phys[base_mask] = lb[base_mask] + DARKBLUE_FACTOR_BASE * db[base_mask]
    LB_phys[hl_mask]   = lb[hl_mask]   + DARKBLUE_FACTOR_HL   * db[hl_mask]
    raw['LB_raw_physics'] = LB_phys

    raw['Lightblue_adjusted'] = np.ceil(raw['LB_raw_physics'])
    raw['Lightblue_adjusted_disp'] = raw['Lightblue_adjusted'].clip(upper=99)

    raw.dropna(subset=['Beschreibung', 'Menge', 'Lightblue_adjusted'], inplace=True)

    if 'Quality' not in raw.columns:
        def infer_q(s):
            t = str(s)
            if 'Q1' in t:
                return 'Q1'
            if 'Q2' in t:
                return 'Q2'
            return 'Q3'
        raw['Quality'] = raw['Beschreibung'].apply(infer_q)

    if 'Helligkeit' not in raw.columns:
        raw['Helligkeit'] = np.nan

    def parse_lightness_cell(v):
        if v is None:
            return None, np.nan
        if isinstance(v, float) and np.isnan(v):
            return None, np.nan

        s_raw = str(v).strip()
        if s_raw == "":
            return None, np.nan

        s = s_raw.lower().replace(",", ".")
        s = re.sub(r"\s+", " ", s).strip()

        if s in ("nan", "none", "-", "--"):
            return None, np.nan

        # text-first handling
        if "dunk" in s or "dark" in s:
            return "3.5", 3.5
        if "hell" in s:
            return "1.5", 1.5

        # explicit numeric forms like 01, 02, 03, 04, 05, 1, 2, 3, 4, 5, 1.5, 2.5 ...
        m = re.search(r'(?<!\d)(0?[1-5](?:\.[05])?)(?!\d)', s)
        if m:
            try:
                token = m.group(1)
                n = float(token)
                if 1.0 <= n <= 5.0:
                    if abs(n - round(n)) < 1e-9:
                        norm = str(int(round(n))).zfill(2)
                    elif abs(n * 2 - round(n * 2)) < 1e-9:
                        norm = f"{int(math.floor(n))}.5"
                    else:
                        norm = str(n)
                    return norm, n
            except Exception:
                pass

        # common L-forms like L3, L04
        m = re.search(r'\bl\s*0?([1-5])\b', s)
        if m:
            try:
                n = float(m.group(1))
                return str(int(n)).zfill(2), n
            except Exception:
                pass

        # "0" should not become a valid lightness class
        if s == "0":
            return None, np.nan

        return None, np.nan

    norms = raw['Helligkeit'].apply(parse_lightness_cell)
    raw['Helligkeit_norm'] = norms.apply(lambda x: x[0] if isinstance(x, tuple) else None)
    raw['Lightness_Num']   = norms.apply(lambda x: x[1] if isinstance(x, tuple) else np.nan)
    norms = raw['Helligkeit'].apply(parse_lightness_cell)
    raw['Helligkeit_norm'] = norms.apply(lambda x: x[0] if isinstance(x, tuple) else None)
    raw['Lightness_Num']   = norms.apply(lambda x: x[1] if isinstance(x, tuple) else np.nan)

    if 'Schüttgewicht' in raw.columns:
        raw['BulkDensity'] = raw['Schüttgewicht'].apply(parse_bulk_density)
    else:
        raw['BulkDensity'] = np.nan

    lb_disp = pd.to_numeric(raw["Lightblue_adjusted_disp"], errors="coerce").fillna(0.0).astype(float)

    def _lb_merge_guard(v):
        v = float(v)
        if v < 2.0:
            return "0 - 2"
        if v < 20.0:
            return "2 - 20"
        if v < 50.0:
            return "20 - 50"
        return "50 - 100"

    raw["Lightblue_Percentage"] = lb_disp.apply(_lb_merge_guard)
    raw["Color_Group"] = np.where(raw["Lightblue_Percentage"].eq("0 - 2"), "Clear", "Lightblue")

    def _material_name_from_bins(color_group, lb_pct_label):
        cg = str(color_group)
        lbv = str(lb_pct_label)
        if cg == "Clear" and lbv == "0 - 2":
            return "PET_Flakes_Clear_0_2"
        if lbv == "2 - 50":
            return "PET_Flakes_LB_2_50"
        return "PET_Flakes_LB_50_100"

    merged_rows = []
    grpcols = ["Color_Group", "Lightblue_Percentage", "Lagerplatz", "Artikel", "Beschreibung", "Helligkeit_norm"]
    for keys, grp in raw.groupby(grpcols, dropna=False):
        total = grp["Menge"].sum()
        if total <= 0:
            continue

        avg_lb_raw  = (grp["Menge"] * grp["LB_raw_physics"]).sum() / total
        avg_lb_disp = (grp["Menge"] * grp["Lightblue_adjusted_disp"]).sum() / total
        avg_db      = (grp["Menge"] * grp["Darkblue"]).sum() / total if "Darkblue" in grp.columns else 0.0

        bd_series = grp["BulkDensity"]
        lt_series = grp["Lightness_Num"]
        avg_bd = (grp["Menge"] * bd_series).sum() / total if not bd_series.isna().all() else np.nan
        avg_lt = (grp["Menge"] * lt_series).sum() / total if not lt_series.isna().all() else np.nan

        color_group = keys[0]
        lb_label    = keys[1]
        mat_name    = _material_name_from_bins(color_group, lb_label)

        merged_rows.append({
            "Color_Group": color_group,
            "Lightblue_Percentage": lb_label,
            "Material_Name": mat_name,
            "Lagerplatz": keys[2],
            "Artikel": int(keys[3]),
            "Beschreibung": keys[4],
            "Helligkeit_norm": keys[5],
            "Menge": total,
            "AV_LB_raw": avg_lb_raw,
            "AV_LB%": avg_lb_disp,
            "AV_DB%": avg_db,
            "AV_BulkDensity": avg_bd,
            "AV_Lightness": avg_lt,
            "Charge/ Paket": list(grp["Charge/ Paket"]),
            "Quality": grp["Quality"].mode().iat[0] if "Quality" in grp else None,
            "Mixing_Level": 1,
            "__is_A1_by_FA": bool(grp["__is_A1_by_FA"].any()) if "__is_A1_by_FA" in grp.columns else False,
        })

    combined_stock = pd.DataFrame(merged_rows)

    # =========================================================
    # 5-lane pipeline with legacy lane-local anchoring
    # =========================================================
    solver_rows = combined_stock.copy()

    solver_rows["bb"] = solver_rows["Charge/ Paket"].apply(
        lambda lst: len(lst) if isinstance(lst, (list, tuple, set)) else 0
    )

    solver_rows["lb_pct_raw"] = (
        solver_rows["AV_LB_raw"].astype(float)
        if "AV_LB_raw" in solver_rows.columns
        else solver_rows["AV_LB%"].astype(float)
    )
    solver_rows["lb_pct"] = solver_rows["lb_pct_raw"].astype(float)
    solver_rows["cluster"] = solver_rows["Lagerplatz"].astype(str) + "|" + solver_rows["Artikel"].astype(str)
    solver_rows["lt_num"] = pd.to_numeric(solver_rows.get("AV_Lightness", np.nan), errors="coerce")

    def _is_dark_row_any(row):
        lt = row.get("lt_num", np.nan)
        try:
            if lt is not None and not (isinstance(lt, float) and np.isnan(lt)):
                if float(lt) >= float(DARK_LT_THRESHOLD):
                    return True
        except Exception:
            pass

        h = row.get("Helligkeit_norm", None)
        h = "" if h is None else str(h).strip().lower()
        try:
            if h in set([str(x).strip().lower() for x in DARK_TEXT_TOKENS]):
                return True
        except Exception:
            pass

        if re.search(r"(^|[^0-9])0?[34]([^0-9]|$)", h):
            return True
        if re.search(r"\bl\s*0?[34]\b", h):
            return True

        txt = " ".join([
            "" if row.get("Beschreibung") is None else str(row.get("Beschreibung")),
            "" if row.get("Material_Name") is None else str(row.get("Material_Name")),
            "" if row.get("Material_Display") is None else str(row.get("Material_Display")),
        ]).lower()

        return any(tok in txt for tok in ("lightgrey", "lg", "grau", "dark", "dunkel", "l3", "l4"))

    solver_rows["__is_dark"] = solver_rows.apply(_is_dark_row_any, axis=1)

    def _lane_from_lb_and_dark(lb_val, is_dark):
        if lb_val is None or (isinstance(lb_val, float) and np.isnan(lb_val)):
            return None
        v = float(lb_val)
        if v >= 50.0:
            return "highlight_use"
        if bool(is_dark):
            return "lightgrey"
        if v < 2.0:
            return "clear"
        if v < 25.0:
            return "clb_low"
        return "clb_high"

    solver_rows["group"] = solver_rows.apply(
        lambda r: _lane_from_lb_and_dark(r.get("lb_pct", np.nan), r.get("__is_dark", False)),
        axis=1
    )
    solver_rows = solver_rows[solver_rows["group"].notna()].copy()

    try:
        bd_fix = pd.to_numeric(solver_rows.get("AV_BulkDensity", np.nan), errors="coerce")
    except Exception:
        bd_fix = pd.Series(np.nan, index=solver_rows.index)

    fa_series = solver_rows["__is_A1_by_FA"].fillna(False).astype(bool) if "__is_A1_by_FA" in solver_rows.columns else pd.Series(False, index=solver_rows.index)

    mask_fa_clear_lowbd = (
        solver_rows["group"].eq("clear") &
        fa_series &
        bd_fix.lt(250.0)
    )
    solver_rows.loc[mask_fa_clear_lowbd, "group"] = "clb_low"

    bad = solver_rows[(solver_rows["group"] == "clear") & (solver_rows["__is_dark"])]
    if not bad.empty:
        print("ERROR: dark rows leaked into clear. Showing top 10:")
        display(bad[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("Dark rows leaked into clear lane.")

    bad_hl = solver_rows[
        (solver_rows["group"] == "lightgrey") &
        (pd.to_numeric(solver_rows["lb_pct"], errors="coerce") >= 50.0)
    ]
    if not bad_hl.empty:
        print("ERROR: HL rows leaked into lightgrey. Showing top 10:")
        display(bad_hl[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("HL rows leaked into lightgrey lane.")

    lt_fix = pd.to_numeric(solver_rows.get("lt_num", np.nan), errors="coerce")
    mask_force_lg = lt_fix.notna() & (lt_fix.astype(float) >= float(DARK_LT_THRESHOLD))
    mask_force_lg = mask_force_lg & solver_rows["group"].isin(["clear", "clb_low", "clb_high"])
    solver_rows.loc[mask_force_lg, "group"] = "lightgrey"

    bad2 = solver_rows[
        (solver_rows["group"] == "clear") &
        (pd.to_numeric(solver_rows["lt_num"], errors="coerce") >= float(DARK_LT_THRESHOLD))
    ]
    if not bad2.empty:
        print("ERROR: LT-dark rows still in clear AFTER reroute. Showing top 10:")
        display(bad2[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("LT-dark rows still in clear after reroute.")

    bad_hl2 = solver_rows[
        (solver_rows["group"] == "lightgrey") &
        (pd.to_numeric(solver_rows["lb_pct"], errors="coerce") >= 50.0)
    ]
    if not bad_hl2.empty:
        print("ERROR: HL rows leaked into lightgrey AFTER reroute. Showing top 10:")
        display(bad_hl2[["Beschreibung", "Lagerplatz", "lb_pct", "AV_Lightness", "lt_num", "Helligkeit_norm", "__is_dark"]].head(10))
        raise ValueError("HL rows leaked into lightgrey after reroute.")

    low_anchor_primary = _choose_clb_low_anchor_window(solver_rows[solver_rows["group"].eq("clb_low")].copy())
    high_anchor_primary = _choose_clb_high_anchor_window(solver_rows[solver_rows["group"].eq("clb_high")].copy())
    lg_anchor_primary = _choose_lg_anchor_window(solver_rows[solver_rows["group"].eq("lightgrey")].copy())

    def _anchor_bb_summary(df_all, lane_name, anchor_dict, label_prefix):
        lane_df = df_all[df_all["group"].eq(lane_name)].copy()

        total_bb = float(pd.to_numeric(lane_df.get("bb", 0.0), errors="coerce").fillna(0.0).sum())

        if lane_df.empty or anchor_dict is None:
            print(f"{label_prefix} anchor USED=none | kept=0 BB | overflow=0 BB | total={int(round(total_bb))} BB")
            return

        lb_vals = pd.to_numeric(lane_df.get("lb_pct", np.nan), errors="coerce")
        bb_vals = pd.to_numeric(lane_df.get("bb", 0.0), errors="coerce").fillna(0.0).astype(float)

        start_v = float(anchor_dict["start"])
        end_v = float(anchor_dict["end"])

        keep_mask = lb_vals.ge(start_v) & lb_vals.lt(end_v)
        low_over_mask = lb_vals.lt(start_v)
        high_over_mask = lb_vals.ge(end_v)

        kept_bb = float(bb_vals[keep_mask].sum())
        low_over_bb = float(bb_vals[low_over_mask].sum())
        high_over_bb = float(bb_vals[high_over_mask].sum())
        overflow_bb = low_over_bb + high_over_bb

        start_txt = int(round(start_v))
        end_txt = int(round(end_v))

        def _fmt_bb(x):
            if abs(x - round(x)) < 1e-9:
                return str(int(round(x)))
            return f"{x:.1f}"

        print(
            f"{label_prefix} anchor USED={start_txt}–{end_txt} | "
            f"kept={_fmt_bb(kept_bb)} BB | "
            f"overflow={_fmt_bb(overflow_bb)} BB "
            f"(low={_fmt_bb(low_over_bb)}, high={_fmt_bb(high_over_bb)}) | "
            f"total={_fmt_bb(total_bb)} BB"
        )

    def _edge_fallback_anchor(lane_min, lane_max, width, side):
        if str(side).upper() == "LOW_EDGE":
            return {
                "start": float(lane_min),
                "end": float(lane_min + width),
                "massW": np.nan,
                "lt_err": np.nan,
                "fallback_side": "LOW_EDGE",
            }
        if str(side).upper() == "HIGH_EDGE":
            return {
                "start": float(lane_max - width),
                "end": float(lane_max),
                "massW": np.nan,
                "lt_err": np.nan,
                "fallback_side": "HIGH_EDGE",
            }
        return None

    def _apply_anchor_plan(df_base, low_anchor_use, high_anchor_use, lg_anchor_use):
        work = df_base.copy()

        work["CLB_LOW_anchor_start"] = np.nan
        work["CLB_LOW_anchor_end"] = np.nan
        work["CLB_LOW_in_anchor"] = True

        work["CLB_HIGH_anchor_start"] = np.nan
        work["CLB_HIGH_anchor_end"] = np.nan
        work["CLB_HIGH_in_anchor"] = True

        work["LG_anchor_start"] = np.nan
        work["LG_anchor_end"] = np.nan
        work["LG_in_anchor"] = True

        mask_low_lane = work["group"].eq("clb_low")
        if low_anchor_use is not None:
            work.loc[mask_low_lane, "CLB_LOW_anchor_start"] = float(low_anchor_use["start"])
            work.loc[mask_low_lane, "CLB_LOW_anchor_end"] = float(low_anchor_use["end"])
            work.loc[mask_low_lane, "CLB_LOW_in_anchor"] = (
                pd.to_numeric(work.loc[mask_low_lane, "lb_pct"], errors="coerce").ge(float(low_anchor_use["start"])) &
                pd.to_numeric(work.loc[mask_low_lane, "lb_pct"], errors="coerce").lt(float(low_anchor_use["end"]))
            )

        mask_high_lane = work["group"].eq("clb_high")
        if high_anchor_use is not None:
            work.loc[mask_high_lane, "CLB_HIGH_anchor_start"] = float(high_anchor_use["start"])
            work.loc[mask_high_lane, "CLB_HIGH_anchor_end"] = float(high_anchor_use["end"])
            work.loc[mask_high_lane, "CLB_HIGH_in_anchor"] = (
                pd.to_numeric(work.loc[mask_high_lane, "lb_pct"], errors="coerce").ge(float(high_anchor_use["start"])) &
                pd.to_numeric(work.loc[mask_high_lane, "lb_pct"], errors="coerce").lt(float(high_anchor_use["end"]))
            )

        mask_lg_lane = work["group"].eq("lightgrey")
        if lg_anchor_use is not None:
            work.loc[mask_lg_lane, "LG_anchor_start"] = float(lg_anchor_use["start"])
            work.loc[mask_lg_lane, "LG_anchor_end"] = float(lg_anchor_use["end"])
            work.loc[mask_lg_lane, "LG_in_anchor"] = (
                pd.to_numeric(work.loc[mask_lg_lane, "lb_pct"], errors="coerce").ge(float(lg_anchor_use["start"])) &
                pd.to_numeric(work.loc[mask_lg_lane, "lb_pct"], errors="coerce").lt(float(lg_anchor_use["end"]))
            )

        work["SOLVER_EXCLUDE"] = False
        work["SOLVER_EXCLUDE_REASON"] = ""
        work["LB_is_overflow"] = False

        if low_anchor_use is not None:
            mask_low_out = mask_low_lane & (~work["CLB_LOW_in_anchor"].fillna(False))
            work.loc[mask_low_out, "SOLVER_EXCLUDE"] = True
            work.loc[mask_low_out, "SOLVER_EXCLUDE_REASON"] = _append_reason_col(
                work.loc[mask_low_out, "SOLVER_EXCLUDE_REASON"],
                "CLB_LOW_outside_anchor_window"
            )
            work.loc[mask_low_out, "LB_is_overflow"] = True

        if high_anchor_use is not None:
            mask_high_out = mask_high_lane & (~work["CLB_HIGH_in_anchor"].fillna(False))
            work.loc[mask_high_out, "SOLVER_EXCLUDE"] = True
            work.loc[mask_high_out, "SOLVER_EXCLUDE_REASON"] = _append_reason_col(
                work.loc[mask_high_out, "SOLVER_EXCLUDE_REASON"],
                "CLB_HIGH_outside_anchor_window"
            )
            work.loc[mask_high_out, "LB_is_overflow"] = True

        if lg_anchor_use is not None:
            mask_lg_out = mask_lg_lane & (~work["LG_in_anchor"].fillna(False))
            work.loc[mask_lg_out, "SOLVER_EXCLUDE"] = True
            work.loc[mask_lg_out, "SOLVER_EXCLUDE_REASON"] = _append_reason_col(
                work.loc[mask_lg_out, "SOLVER_EXCLUDE_REASON"],
                "LG_outside_anchor_window"
            )
            work.loc[mask_lg_out, "LB_is_overflow"] = True

        def compute_visible_idx(df_lane, bb_limit):
            if df_lane.empty:
                return set()
            df_sorted = df_lane.sort_values("lb_pct", ascending=True)
            keep_idx = []
            cum_bb = 0.0
            for idx_row, row in df_sorted.iterrows():
                bb_here = float(row.get("bb", 0.0))
                if cum_bb + bb_here <= bb_limit + 1e-9:
                    keep_idx.append(idx_row)
                    cum_bb += bb_here
                else:
                    break
            if cum_bb <= EPS:
                keep_idx = list(df_sorted.index)
            return set(keep_idx)

        hl_subset = work[work["group"].eq("highlight_use")].copy()
        highlight_visible_idx = compute_visible_idx(hl_subset, VISIBLE_BB_LIMIT)
        mask_hl_all = work["group"].eq("highlight_use")
        work.loc[mask_hl_all & (~work.index.isin(highlight_visible_idx)), "LB_is_overflow"] = True

        def _dyn_bin(row):
            g = row.get("group", "")

            if g == "clear":
                return "0 - 2"

            if g == "clb_low":
                s = row.get("CLB_LOW_anchor_start", np.nan)
                e = row.get("CLB_LOW_anchor_end", np.nan)
                if pd.notna(s) and pd.notna(e):
                    return f"{int(round(float(s)))} - {int(round(float(e)))}"
                return "2 - 25"

            if g == "clb_high":
                s = row.get("CLB_HIGH_anchor_start", np.nan)
                e = row.get("CLB_HIGH_anchor_end", np.nan)
                if pd.notna(s) and pd.notna(e):
                    return f"{int(round(float(s)))} - {int(round(float(e)))}"
                return "25 - 50"

            if g == "lightgrey":
                s = row.get("LG_anchor_start", np.nan)
                e = row.get("LG_anchor_end", np.nan)
                if pd.notna(s) and pd.notna(e):
                    return f"LG {int(round(float(s)))} - {int(round(float(e)))}"
                return "LG 0 - 25"

            if g == "highlight_use":
                return "50 - 100"

            return ""

        GROUP_TO_BASE = {
            "clear": "Clear Neutral",
            "clb_low": "Clearlightblue Low",
            "clb_high": "Clearlightblue High",
            "lightgrey": "Lightgrey",
            "highlight_use": "Lightblue"
        }
        work["Material"] = work["group"].map(GROUP_TO_BASE)
        work["Material_Display"] = work["Material"]

        bins = work.copy()
        bins["dyn_bin"] = bins.apply(_dyn_bin, axis=1)
        eff = bins[~bins["SOLVER_EXCLUDE"].fillna(False)].copy()

        return work, bins, eff

    def _refresh_solver_state_from_eff(eff_df):
        global solver_bins, groups_present, group_lb_means, group_bd_means, group_lt_means
        global stock_caps, stock_shares_no_hl_all, upper_cap_local, effective_min_total

        stock_caps_local = (
            eff_df.groupby("group", as_index=False)["bb"]
            .sum()
            .set_index("group")["bb"]
            .to_dict()
        )

        (
            groups_present_local,
            group_lb_means_local,
            stock_shares_no_hl_all_local,
            group_bd_means_local,
            group_lt_means_local
        ) = compute_group_stats_for_solver(eff_df)

        allowed_groups = {"clear", "clb_low", "clb_high", "lightgrey", "highlight_use"}
        groups_present_local = {k: v for k, v in groups_present_local.items() if k in allowed_groups}
        group_lb_means_local = {k: v for k, v in group_lb_means_local.items() if k in allowed_groups}
        group_bd_means_local = {k: v for k, v in group_bd_means_local.items() if k in allowed_groups}
        group_lt_means_local = {k: v for k, v in group_lt_means_local.items() if k in allowed_groups}
        stock_caps_local     = {k: v for k, v in stock_caps_local.items()     if k in allowed_groups}

        stock_shares_no_hl_all_local = {
            k: v for k, v in stock_shares_no_hl_all_local.items()
            if k in {"clear", "clb_low", "clb_high", "lightgrey"}
        }

        upper_cap_local_local = {
            "clear": UPPER_CAP.get("clear", 200.0),
            "clb_low": UPPER_CAP.get("clb_low", 200.0),
            "clb_high": UPPER_CAP.get("clb_high", 200.0),
            "lightgrey": UPPER_CAP.get("lightgrey", 200.0),
            "highlight_use": UPPER_CAP.get("highlight_use", 200.0),
        }
        effective_min_total_local = get_effective_min_total(stock_caps_local, demand_mixes_floor=MIN_MIXES)

        solver_bins = eff_df.copy()
        groups_present = groups_present_local
        group_lb_means = group_lb_means_local
        group_bd_means = group_bd_means_local
        group_lt_means = group_lt_means_local
        stock_caps = stock_caps_local
        stock_shares_no_hl_all = stock_shares_no_hl_all_local
        upper_cap_local = upper_cap_local_local
        effective_min_total = effective_min_total_local

    primary_plan = {
        "clb_low": low_anchor_primary,
        "clb_high": high_anchor_primary,
        "lightgrey": lg_anchor_primary,
        "tag_suffix": "PRIMARY",
        "fallback_used": [],
    }

    solver_rows_primary, solver_bins_primary, solver_eff_primary = _apply_anchor_plan(
        solver_rows,
        primary_plan["clb_low"],
        primary_plan["clb_high"],
        primary_plan["lightgrey"]
    )

    _refresh_solver_state_from_eff(solver_eff_primary)
    solver_bins = solver_bins_primary.copy()
    lane_bb_tonnage = _lane_bb_tonnage_map(solver_eff_primary)

    scored = build_scored_candidates_with_priority()
    best = choose_best_candidate(scored)

    chosen_plan = primary_plan
    chosen_solver_rows = solver_rows_primary
    chosen_solver_bins = solver_bins_primary
    chosen_solver_eff = solver_eff_primary

    if best is None:
        low_anchor_fb_low = _edge_fallback_anchor(0.0, CLB_LOW_MAX, 12.0, "LOW_EDGE")
        low_anchor_fb_high = _edge_fallback_anchor(0.0, CLB_LOW_MAX, 12.0, "HIGH_EDGE")

        high_anchor_fb_low = _edge_fallback_anchor(CLB_HIGH_MIN, CLB_HIGH_MAX, 12.0, "LOW_EDGE")
        high_anchor_fb_high = _edge_fallback_anchor(CLB_HIGH_MIN, CLB_HIGH_MAX, 12.0, "HIGH_EDGE")

        lg_anchor_fb_low = _edge_fallback_anchor(LG_MIN, LG_MAX, ANCHOR_WIDTH_LG, "LOW_EDGE")
        lg_anchor_fb_high = _edge_fallback_anchor(LG_MIN, LG_MAX, ANCHOR_WIDTH_LG, "HIGH_EDGE")

        fallback_scenarios = []

        def _same_anchor(a, b):
            if a is None or b is None:
                return False
            return (
                abs(float(a["start"]) - float(b["start"])) < 1e-9 and
                abs(float(a["end"]) - float(b["end"])) < 1e-9
            )

        def _add_fallback_plan(low_a, high_a, lg_a, tag_suffix, flags):
            if (low_a is not None and low_anchor_primary is not None and _same_anchor(low_a, low_anchor_primary) and
                high_a is not None and high_anchor_primary is not None and _same_anchor(high_a, high_anchor_primary) and
                lg_a is not None and lg_anchor_primary is not None and _same_anchor(lg_a, lg_anchor_primary)):
                return

            fallback_scenarios.append({
                "clb_low": low_a,
                "clb_high": high_a,
                "lightgrey": lg_a,
                "tag_suffix": tag_suffix,
                "fallback_used": flags,
            })

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            if low_anchor_primary is None or not _same_anchor(a_low, low_anchor_primary):
                _add_fallback_plan(a_low, high_anchor_primary, lg_anchor_primary, f"FALLBACK_{lbl_low.replace('=', '_')}", [lbl_low])

        for a_high, lbl_high in [
            (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
            (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
        ]:
            if high_anchor_primary is None or not _same_anchor(a_high, high_anchor_primary):
                _add_fallback_plan(low_anchor_primary, a_high, lg_anchor_primary, f"FALLBACK_{lbl_high.replace('=', '_')}", [lbl_high])

        for a_lg, lbl_lg in [
            (lg_anchor_fb_low, "LG=LOW_EDGE"),
            (lg_anchor_fb_high, "LG=HIGH_EDGE"),
        ]:
            if lg_anchor_primary is None or not _same_anchor(a_lg, lg_anchor_primary):
                _add_fallback_plan(low_anchor_primary, high_anchor_primary, a_lg, f"FALLBACK_{lbl_lg.replace('=', '_')}", [lbl_lg])

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            for a_high, lbl_high in [
                (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
                (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
            ]:
                flags = [lbl_low, lbl_high]
                _add_fallback_plan(
                    a_low, a_high, lg_anchor_primary,
                    f"FALLBACK_{lbl_low.replace('=', '_')}_{lbl_high.replace('=', '_')}",
                    flags
                )

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            for a_lg, lbl_lg in [
                (lg_anchor_fb_low, "LG=LOW_EDGE"),
                (lg_anchor_fb_high, "LG=HIGH_EDGE"),
            ]:
                flags = [lbl_low, lbl_lg]
                _add_fallback_plan(
                    a_low, high_anchor_primary, a_lg,
                    f"FALLBACK_{lbl_low.replace('=', '_')}_{lbl_lg.replace('=', '_')}",
                    flags
                )

        for a_high, lbl_high in [
            (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
            (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
        ]:
            for a_lg, lbl_lg in [
                (lg_anchor_fb_low, "LG=LOW_EDGE"),
                (lg_anchor_fb_high, "LG=HIGH_EDGE"),
            ]:
                flags = [lbl_high, lbl_lg]
                _add_fallback_plan(
                    low_anchor_primary, a_high, a_lg,
                    f"FALLBACK_{lbl_high.replace('=', '_')}_{lbl_lg.replace('=', '_')}",
                    flags
                )

        for a_low, lbl_low in [
            (low_anchor_fb_low, "CLB_LOW=LOW_EDGE"),
            (low_anchor_fb_high, "CLB_LOW=HIGH_EDGE"),
        ]:
            for a_high, lbl_high in [
                (high_anchor_fb_low, "CLB_HIGH=LOW_EDGE"),
                (high_anchor_fb_high, "CLB_HIGH=HIGH_EDGE"),
            ]:
                for a_lg, lbl_lg in [
                    (lg_anchor_fb_low, "LG=LOW_EDGE"),
                    (lg_anchor_fb_high, "LG=HIGH_EDGE"),
                ]:
                    flags = [lbl_low, lbl_high, lbl_lg]
                    _add_fallback_plan(
                        a_low, a_high, a_lg,
                        f"FALLBACK_{lbl_low.replace('=', '_')}_{lbl_high.replace('=', '_')}_{lbl_lg.replace('=', '_')}",
                        flags
                    )

        if fallback_scenarios:
            print(f"ANCHOR FALLBACK scenarios queued={len(fallback_scenarios)}")
        else:
            print("ANCHOR FALLBACK scenarios queued=0")

        feasible_fallbacks = []

        for plan_try in fallback_scenarios:
            def _fmt_anchor(a):
                if a is None:
                    return "none"
                return f"{int(round(float(a['start'])))}–{int(round(float(a['end'])))}"

            print(
                "TRY FALLBACK:",
                f"tag={plan_try['tag_suffix']}",
                f"CLB_LOW={_fmt_anchor(plan_try['clb_low'])}",
                f"CLB_HIGH={_fmt_anchor(plan_try['clb_high'])}",
                f"LG={_fmt_anchor(plan_try['lightgrey'])}",
                f"flags={','.join(plan_try.get('fallback_used', [])) if plan_try.get('fallback_used') else 'none'}"
            )

            rows_try, bins_try, eff_try = _apply_anchor_plan(
                solver_rows,
                plan_try["clb_low"],
                plan_try["clb_high"],
                plan_try["lightgrey"]
            )

            _refresh_solver_state_from_eff(eff_try)
            solver_bins = bins_try.copy()
            lane_bb_tonnage = _lane_bb_tonnage_map(eff_try)

            scored_try = build_scored_candidates_with_priority()
            best_try = choose_best_candidate(scored_try)

            if best_try is not None:
                print(f"FALLBACK RESULT: feasible -> {plan_try['tag_suffix']}")
                feasible_fallbacks.append({
                    "best": best_try,
                    "plan": plan_try,
                    "rows": rows_try,
                    "bins": bins_try,
                    "eff": eff_try,
                })
            else:
                print(f"FALLBACK RESULT: no feasible -> {plan_try['tag_suffix']}")

        if feasible_fallbacks:
            def _fb_sort_key(rec):
                b = rec["best"]
                lb_err = abs(float(b.get("result_lb", np.nan)) - float(target_lb))
                lt_err = abs(float(b.get("result_lt", np.nan)) - float(lt_target))
                mixes = int(b.get("mixes", 0))
                total_bb = float(sum((b.get("picks") or {}).values()))
                fallback_count = len(rec["plan"].get("fallback_used", []))
                return (
                    fallback_count,
                    lb_err,
                    lt_err,
                    -mixes,
                    -total_bb
                )

            feasible_fallbacks = sorted(feasible_fallbacks, key=_fb_sort_key)
            chosen_fb = feasible_fallbacks[0]

            best = chosen_fb["best"]
            chosen_plan = chosen_fb["plan"]
            chosen_solver_rows = chosen_fb["rows"]
            chosen_solver_bins = chosen_fb["bins"]
            chosen_solver_eff = chosen_fb["eff"]

            print(
                "FALLBACK BEST CHOSEN:",
                f"tag={chosen_plan['tag_suffix']}",
                f"flags={','.join(chosen_plan.get('fallback_used', [])) if chosen_plan.get('fallback_used') else 'none'}",
                f"mixes={int(best.get('mixes', 0))}",
                f"LB={float(best.get('result_lb', np.nan)):.3f}",
                f"LT={float(best.get('result_lt', np.nan)):.3f}"
            )

    solver_rows = chosen_solver_rows.copy()
    solver_bins = chosen_solver_bins.copy()
    solver_eff = chosen_solver_eff.copy()

    _refresh_solver_state_from_eff(solver_eff)
    solver_bins = chosen_solver_bins.copy()
    lane_bb_tonnage = _lane_bb_tonnage_map(solver_eff)

    if chosen_plan.get("fallback_used"):
        print("ANCHOR FALLBACK USED:", ", ".join(chosen_plan["fallback_used"]))

    _anchor_bb_summary(solver_rows, "clb_low", chosen_plan["clb_low"], "CLB_LOW")
    _anchor_bb_summary(solver_rows, "clb_high", chosen_plan["clb_high"], "CLB_HIGH")
    _anchor_bb_summary(solver_rows, "lightgrey", chosen_plan["lightgrey"], "LG")

    solver_tag = f"SOLVER=DUAL_CLB_LANE_LOCAL_ANCHOR_{chosen_plan['tag_suffix']}"
    print(solver_tag)

    def compute_hidden_materials(df_full):
        tmp = df_full.copy()
        if "bb" in tmp.columns:
            tmp["_bb_count"] = pd.to_numeric(tmp["bb"], errors="coerce").fillna(0.0).astype(float)
        else:
            tmp["_bb_count"] = tmp["Charge/ Paket"].apply(lambda x: float(len(x)) if isinstance(x, (list, tuple)) else 0.0)

        tmp["MatForHide"] = tmp["Material_Display"].astype(str)
        tmp.loc[(tmp["group"] == "highlight_use") & (tmp["LB_is_overflow"]), "MatForHide"] = "Lightblue (overflow)"

        by_mat = (
            tmp.groupby("MatForHide", as_index=False)["_bb_count"]
               .sum()
               .rename(columns={"_bb_count": "BB_Count"})
        )
        total_bb = float(by_mat["BB_Count"].sum())

        def _thr(material):
            abs_thr = 5.0 if str(material).startswith("Lightblue") else 20.0
            pct_thr = 0.05 * total_bb
            return min(abs_thr, pct_thr)

        return set(by_mat[by_mat.apply(lambda r: float(r["BB_Count"]) < _thr(r["MatForHide"]), axis=1)]["MatForHide"])

    hidden_mats = compute_hidden_materials(solver_bins)

    def build_headers_from_solverbins(solver_bins, best, specs, hidden_mats=None, mix_stats=None, solver_tag=None):
        view = solver_bins.groupby("group", as_index=False).agg(bb=("bb", "sum"))
        total_bb = float(view["bb"].sum())
        sh = {r["group"]: float(r["bb"]) / max(total_bb, 1e-9) for _, r in view.iterrows()}

        Lmin, Lmax = specs["L_range"]
        bmin, bmax = specs["b_range"]

        order = ["clear", "clb_low", "clb_high", "lightgrey", "highlight_use"]
        label_map = {
            "clear": "clear",
            "clb_low": "clb_low",
            "clb_high": "clb_high",
            "lightgrey": "lightgrey",
            "highlight_use": "highlight",
        }
        tag_txt = f", {solver_tag}" if solver_tag else ""

        parts = [f"Stock: {int(round(total_bb))} BB total (Modus {PRODUCTION_MODE}{tag_txt})"]
        for g in order:
            bb_val = float(view.loc[view["group"] == g, "bb"].sum()) if (g in set(view["group"].values)) else 0.0
            if bb_val > 0:
                parts.append(f"{label_map[g]}: {int(round(bb_val))} BB ({sh.get(g, 0.0) * 100:.1f}%)")
        stock_line = " | ".join(parts)

        metrics_suffix = ""
        if mix_stats is not None:
            mix_lb = mix_stats.get("mix_lb", float("nan"))
            mix_lt = mix_stats.get("mix_lt", float("nan"))
            mix_bd = mix_stats.get("mix_bd", float("nan"))
            mix_mx = int(mix_stats.get("mix_mx", 0))

            def _fmt_num(v, fmt, dash="—"):
                try:
                    if v is None or (isinstance(v, float) and np.isnan(v)):
                        return dash
                    return fmt.format(float(v))
                except Exception:
                    return dash

            lb_txt = _fmt_num(mix_lb, "LB={:.3f}")
            lt_txt = _fmt_num(mix_lt, "LT={:.3f}")
            bd_txt = _fmt_num(mix_bd, "BD={:.0f}")
            mx_txt = f"mixes={mix_mx}" if mix_mx > 0 else "mixes=—"

            bd_flag = ""
            try:
                if mix_bd is not None and not (isinstance(mix_bd, float) and np.isnan(mix_bd)):
                    bd_val = float(mix_bd)
                    bd_range_txt = f"{BD_WARN_LOW:.0f}–{BD_WARN_HIGH:.0f}"
                    if bd_val < float(BD_WARN_LOW):
                        bd_flag = f" | BD warning: LOW (<{BD_WARN_LOW:.0f})"
                    elif bd_val > float(BD_WARN_HIGH):
                        bd_flag = f" | BD warning: HIGH (>{BD_WARN_HIGH:.0f})"
                    else:
                        bd_flag = f" | BD OK ({bd_range_txt})"
            except Exception:
                bd_flag = ""

            metrics_suffix = f"Mixture: {lb_txt} | {lt_txt} | {bd_txt} | {mx_txt}{bd_flag}"

        def _is_hidden(name):
            if not hidden_mats:
                return False
            name = str(name)
            if name in hidden_mats:
                return True
            if name == "Lightblue":
                return ("Lightblue" in hidden_mats) or ("Lightblue (overflow)" in hidden_mats)
            return False

        def line_for(label, best_dict):
            suffix_specs = f"Expected L*: {Lmin:.1f} – {Lmax:.1f} | Expected b*: {bmin:.1f} – {bmax:.1f}"
            if not best_dict:
                core = f"{label}: —"
            else:
                picks = best_dict.get("picks") or {}
                mixes = int(best_dict.get("mixes", 0))

                def disp_part(key, dispname):
                    v = float(picks.get(key, 0.0))
                    if v <= EPS:
                        return None
                    amt_txt = bb_to_display(v, key)
                    if amt_txt is None:
                        return None
                    suffix = " (hidden)" if _is_hidden(dispname) else ""
                    if key == "highlight_use":
                        return f"{amt_txt} Lightblue{suffix}"
                    return f"{amt_txt} {dispname}{suffix}"

                segs = [
                    disp_part("clear", "Clear Neutral"),
                    disp_part("clb_low", "Clearlightblue Low"),
                    disp_part("clb_high", "Clearlightblue High"),
                    disp_part("lightgrey", "Lightgrey"),
                    disp_part("highlight_use", "Lightblue"),
                ]
                segs = [s for s in segs if s]
                core = f"{label}: " + " | ".join(segs) + (f" → Max Mixes {mixes}" if mixes > 0 else "")

            out = core + " | " + suffix_specs
            if metrics_suffix:
                out = out + " | " + metrics_suffix
            return out

        corr = line_for("Recipe Blend", best)
        return stock_line, corr

    mix_stats = None
    if best is not None:
        mix_stats = {
            "mix_lb": best.get("result_lb", float("nan")),
            "mix_lt": best.get("result_lt", float("nan")),
            "mix_bd": best.get("result_bd", float("nan")),
            "mix_mx": int(best.get("mixes", 0)),
        }

    header_stock, header_corr = build_headers_from_solverbins(solver_bins, best, specs, hidden_mats, mix_stats, solver_tag)
    print(header_stock)
    print(header_corr)

    if best is None:
        return

    # =========================================================
    # Candidate Recipe table + Excel
    # =========================================================
    combined_stock_solver = solver_bins.copy()
    total_stock = float(combined_stock_solver["Menge"].sum())
    required_tons = total_stock

    if total_stock > 0:
        combined_stock_solver["Dynamic_Percentage"] = combined_stock_solver["Menge"] / total_stock * 100.0
        combined_stock_solver["Tonnage_Allocated"] = combined_stock_solver["Menge"] / total_stock * required_tons
    else:
        combined_stock_solver["Dynamic_Percentage"] = 0.0
        combined_stock_solver["Tonnage_Allocated"] = 0.0

    valid_cands = [r for r in combined_stock_solver.to_dict("records") if float(r.get("Tonnage_Allocated", 0.0)) > 0.0]
    best_candidate = tuple(valid_cands) if valid_cands else None

    def aggregate_candidate_recipe_all(required):
        df0 = combined_stock_solver.copy()

        stats = df0.groupby(
            ["Material_Name", "group", "dyn_bin", "Material_Display", "LB_is_overflow"],
            as_index=False
        ).agg({
            "Menge": "sum",
            "AV_LB%": lambda s: (
                float(np.average(pd.to_numeric(s, errors="coerce").fillna(0.0), weights=df0.loc[s.index, "Menge"]))
                if not pd.to_numeric(s, errors="coerce").isna().all() else np.nan
            ),
            "AV_BulkDensity": lambda s: (
                float(np.average(pd.to_numeric(df0.loc[s.index, "AV_BulkDensity"], errors="coerce").fillna(0.0),
                                 weights=df0.loc[s.index, "Menge"]))
                if not df0.loc[s.index, "AV_BulkDensity"].isna().all() else np.nan
            ),
            "AV_Lightness": lambda s: (
                float(np.average(pd.to_numeric(df0.loc[s.index, "AV_Lightness"], errors="coerce").fillna(0.0),
                                 weights=df0.loc[s.index, "Menge"]))
                if not df0.loc[s.index, "AV_Lightness"].isna().all() else np.nan
            ),
            "Tonnage_Allocated": "sum",
            "Dynamic_Percentage": "sum" if "Dynamic_Percentage" in df0.columns else "size",
        }).rename(columns={
            "Menge": "Group_Menge",
            "AV_LB%": "Group_AV_LB%",
            "AV_BulkDensity": "Group_AV_BD",
            "AV_Lightness": "Group_AV_LT",
            "Tonnage_Allocated": "Group_TA",
            "Dynamic_Percentage": "Group_DP"
        })

        def weight_factor(lb):
            if lb is None or (isinstance(lb, float) and np.isnan(lb)):
                return 1.0
            return 1.0 / (1.0 + abs(float(lb) - float(target_lb)) / max(float(target_lb), 1e-9))

        stats["Weight_Factor"] = stats["Group_AV_LB%"].apply(weight_factor)
        stats["Modified_Weight"] = stats["Group_TA"] * stats["Weight_Factor"]
        total_mod = float(stats["Modified_Weight"].sum())
        stats["Adj_TA"] = (stats["Modified_Weight"] / total_mod * required) if total_mod else stats["Group_TA"]
        stats["Percentage"] = stats["Adj_TA"] / required * 100.0 if required > 0 else 0.0

        batches = required / 5.5 if required > 0 else 1.0

        def _bb_round(row):
            per_batch_tons = float(row["Adj_TA"]) / batches if batches > 0 else 0.0
            grp = str(row["group"])
            tons_per_bb = float(lane_bb_tonnage.get(grp, DEFAULT_TONS_PER_BB))
            bb_needed = per_batch_tons / max(tons_per_bb, 1e-9)

            if grp == "highlight_use":
                return round(bb_needed / STEP_HL) * STEP_HL
            return round(bb_needed * 2.0) / 2.0

        stats["Big Bags"] = stats.apply(_bb_round, axis=1)
        stats["BB_Min"] = (stats["Big Bags"] * 2 - 1).clip(lower=0).round() / 2
        stats["BB_Max"] = (stats["Big Bags"] * 2 + 1).round() / 2

        out = df0.merge(
            stats[[
                "Material_Name", "group", "dyn_bin", "Material_Display", "LB_is_overflow",
                "Percentage", "Big Bags", "BB_Min", "BB_Max"
            ]],
            on=["Material_Name", "group", "dyn_bin", "Material_Display", "LB_is_overflow"],
            how="left"
        )
        out["Material"] = out["Material_Display"]
        return out

    candidate_recipe = aggregate_candidate_recipe_all(required_tons) if best_candidate else pd.DataFrame()

    def format_packets(packets):
        if isinstance(packets, str):
            packets = packets.split(", ")
        return ", ".join(find_ranges(packets))

    unused_mats_for_hide = set()

    if not candidate_recipe.empty:
        candidate_recipe = candidate_recipe.copy()
        expr = r"(?:st[-\s]?f|recyclass)"
        candidate_recipe["NamePriority"] = candidate_recipe["Beschreibung"].astype(str).str.contains(expr, case=False, na=False, regex=True).astype(int)
        candidate_recipe["Material_Vis"] = candidate_recipe["Material"].astype(str)

        keep_mats = ["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"]
        candidate_recipe = candidate_recipe[candidate_recipe["Material_Vis"].isin(keep_mats)].copy()
        candidate_recipe["Material_Vis"] = pd.Categorical(candidate_recipe["Material_Vis"], categories=keep_mats, ordered=True)
        candidate_recipe["AV_LB%"] = pd.to_numeric(candidate_recipe["AV_LB%"], errors="coerce")

        def _as_pkt_list(x):
            if isinstance(x, (list, tuple, set)):
                return [str(p).strip() for p in x if str(p).strip() not in ("", "nan", "NaN", "None")]
            if x is None or (isinstance(x, float) and np.isnan(x)):
                return []
            s = str(x).strip()
            if s in ("", "nan", "NaN", "None"):
                return []
            return [s]

        candidate_recipe["Raw_Charge"] = candidate_recipe["Charge/ Paket"].apply(_as_pkt_list)
        if "dyn_bin" not in candidate_recipe.columns:
            candidate_recipe["dyn_bin"] = ""

        merge_keys = ["Material_Vis", "Lagerplatz", "dyn_bin", "LB_is_overflow"]
        merged_rows = []

        for keys, grp in candidate_recipe.groupby(merge_keys, dropna=False, sort=False, observed=False):
            matv, lager, dbin, is_overflow = keys

            pkts = []
            for lst in grp["Raw_Charge"].tolist():
                if isinstance(lst, (list, tuple, set)):
                    pkts.extend(lst)
            pkts = list(dict.fromkeys([str(p).strip() for p in pkts if str(p).strip() not in ("", "nan", "NaN", "None")]))

            w = pd.to_numeric(grp.get("Menge", grp.get("Quantity", 0.0)), errors="coerce").fillna(0.0).astype(float)
            total_w = float(w.sum())
            if total_w <= EPS:
                continue

            def _wavg(col):
                if col not in grp.columns:
                    return np.nan
                x = pd.to_numeric(grp[col], errors="coerce")
                if x.isna().all():
                    return np.nan
                m = ~x.isna()
                if not m.any():
                    return np.nan
                ww = w[m].astype(float)
                if float(ww.sum()) <= EPS:
                    return float(np.nanmean(x))
                return float((ww * x[m].astype(float)).sum() / max(float(ww.sum()), 1e-9))

            arts = [a for a in grp.get("Artikel", pd.Series([], dtype=object)).tolist() if pd.notna(a)]
            arts_u = sorted(set(int(a) for a in arts)) if arts else []
            if len(arts_u) == 1:
                art_out = arts_u[0]
                desc_out = str(grp["Beschreibung"].iloc[0]) if "Beschreibung" in grp.columns else ""
            else:
                art_out = 0
                desc_out = "MIXED (" + ", ".join(map(str, arts_u[:10])) + (", ..." if len(arts_u) > 10 else "") + ")"

            qty_sum = float(pd.to_numeric(grp["Quantity"], errors="coerce").fillna(0.0).sum()) if "Quantity" in grp.columns else total_w
            ta_sum = float(pd.to_numeric(grp["Tonnage_Allocated"], errors="coerce").fillna(0.0).sum()) if "Tonnage_Allocated" in grp.columns else float("nan")
            dp_sum = float(pd.to_numeric(grp["Dynamic_Percentage"], errors="coerce").fillna(0.0).sum()) if "Dynamic_Percentage" in grp.columns else float("nan")

            if "Quality" in grp.columns and not grp["Quality"].isna().all():
                q_out = grp["Quality"].mode().iat[0]
            else:
                q_out = None

            merged_rows.append({
                "Material_Vis": str(matv),
                "Lagerplatz": lager,
                "dyn_bin": str(dbin),
                "LB_is_overflow": bool(is_overflow),
                "Artikel": art_out,
                "Beschreibung": desc_out,
                "Quality": q_out,
                "Menge": total_w,
                "Quantity": qty_sum,
                "Tonnage_Allocated": ta_sum,
                "Dynamic_Percentage": dp_sum,
                "AV_LB%": _wavg("AV_LB%"),
                "AV_Lightness": _wavg("AV_Lightness"),
                "AV_BulkDensity": _wavg("AV_BulkDensity"),
                "Raw_Charge": pkts,
                "NamePriority": int(grp["NamePriority"].max()) if "NamePriority" in grp.columns else 0,
            })

        candidate_recipe = pd.DataFrame(merged_rows).reset_index(drop=True)
        candidate_recipe["Charge/ Paket"] = candidate_recipe["Raw_Charge"].apply(format_packets)

        sort_parts = []
        for _, row in candidate_recipe.iterrows():
            matv = str(row["Material_Vis"])
            avlb = float(row["AV_LB%"]) if pd.notna(row["AV_LB%"]) else 0.0
            if matv == "Clearlightblue High":
                sort_parts.append(-avlb)
            else:
                sort_parts.append(avlb)
        candidate_recipe["__mat_specific_sort"] = sort_parts

        candidate_recipe = candidate_recipe.sort_values(
            ["Material_Vis", "NamePriority", "__mat_specific_sort", "Lagerplatz", "AV_LB%"],
            kind="stable", ignore_index=True
        )
        candidate_recipe = candidate_recipe.drop(columns=["__mat_specific_sort"])
        candidate_recipe["Pick_Seq"] = candidate_recipe.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1

        pick_dict = (best or {}).get("picks", {})
        group_to_vis = {
            "clear": "Clear Neutral",
            "clb_low": "Clearlightblue Low",
            "clb_high": "Clearlightblue High",
            "lightgrey": "Lightgrey",
            "highlight_use": "Lightblue"
        }

        bb_plan = {}
        for g_key, bb_val in pick_dict.items():
            vis = group_to_vis.get(g_key)
            if vis is None:
                continue

            bb_val = float(bb_val)
            tons_per_bb = float(lane_bb_tonnage.get(g_key, DEFAULT_TONS_PER_BB))
            if tons_per_bb <= EPS:
                tons_per_bb = DEFAULT_TONS_PER_BB

            target_tons = bb_val * tons_per_bb
            corrected_bb = target_tons / tons_per_bb

            if g_key == "highlight_use":
                rounded = round(corrected_bb / STEP_HL) * STEP_HL
            else:
                rounded = round(corrected_bb * 2.0) / 2.0

            bb_plan[vis] = bb_plan.get(vis, 0.0) + rounded

        for m in keep_mats:
            bb_plan.setdefault(m, 0.0)

        total_bb_plan = float(sum(v for v in bb_plan.values() if v is not None))
        candidate_recipe["Big Bags"] = candidate_recipe["Material_Vis"].astype(str).map(bb_plan).astype(float).fillna(0.0)

        def _lane_pct(matv):
            if total_bb_plan <= 0:
                return 0.0
            return (float(bb_plan.get(str(matv), 0.0)) / total_bb_plan) * 100.0

        candidate_recipe["Percentage"] = candidate_recipe["Material_Vis"].apply(_lane_pct)
        unused_mats_for_hide = {m for m, v in bb_plan.items() if (v is None or float(v) <= 1e-9)}

    def generate_supply_chain_exact(df):
        if df is None or df.empty:
            return pd.DataFrame()

        order_vis = ["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"]
        supply, required, flexible, pkt_row = {}, {}, {}, {}

        for _, r in df.iterrows():
            matv = str(r["Material_Vis"])
            pkts = r.get("Raw_Charge", [])
            pkts = list(pkts) if isinstance(pkts, (list, tuple)) else ([pkts] if pkts else [])
            if not pkts:
                continue

            supply.setdefault(matv, [])
            rb = float(r.get("Big Bags", 0.0))
            required[matv] = max(int(math.floor(rb)), 0)
            flexible[matv] = (matv in ["Lightblue"])

            for pkt in pkts:
                supply[matv].append(pkt)
                pkt_row[pkt] = {
                    "Artikel": r.get("Artikel"),
                    "Beschreibung": r.get("Beschreibung"),
                    "Lagerplatz": r.get("Lagerplatz"),
                    "AV_LB%": r.get("AV_LB%"),
                    "Material_Vis": matv
                }

        max_full = []
        for m in order_vis:
            if m in supply and (not flexible.get(m, False)) and required.get(m, 0) > 0:
                max_full.append(len(supply[m]) // required[m])
        max_mix = min(max_full) if max_full else 0

        rows = []
        for mix in range(max_mix):
            for matv in order_vis:
                req = required.get(matv, 0)
                if req <= 0 or matv not in supply:
                    continue
                subset = supply[matv][mix * req:(mix + 1) * req]
                for pkt in subset:
                    r0 = pkt_row.get(pkt, {})
                    rows.append({
                        "Artikel": r0.get("Artikel"),
                        "Beschreibung": r0.get("Beschreibung"),
                        "Lagerplatz": r0.get("Lagerplatz"),
                        "AV_LB%": r0.get("AV_LB%"),
                        "Material": matv,
                        "Packet": pkt,
                        "Mixture": mix + 1
                    })

        sc = pd.DataFrame(rows)
        if not sc.empty:
            sc["PacketIndex"] = sc.groupby(["Mixture", "Material"]).cumcount()
        return sc

    supply_chain_df = generate_supply_chain_exact(candidate_recipe)

    qc = (
        solver_bins.groupby(["Material_Display", "dyn_bin"], as_index=False)
        .agg({
            "Menge": "sum",
            "AV_LB%": lambda s: np.average(s, weights=solver_bins.loc[s.index, "Menge"]),
            "AV_BulkDensity": lambda s: (
                lambda vals, wts: float(np.average(vals, weights=wts)) if len(vals) > 0 else np.nan
            )(
                vals=solver_bins.loc[s.index, "AV_BulkDensity"][~solver_bins.loc[s.index, "AV_BulkDensity"].isna()].astype(float),
                wts=solver_bins.loc[s.index, "Menge"][~solver_bins.loc[s.index, "AV_BulkDensity"].isna()].astype(float),
            ),
            "AV_Lightness": lambda s: (
                lambda vals, wts: float(np.average(vals, weights=wts)) if len(vals) > 0 else np.nan
            )(
                vals=solver_bins.loc[s.index, "AV_Lightness"][~solver_bins.loc[s.index, "AV_Lightness"].isna()].astype(float),
                wts=solver_bins.loc[s.index, "Menge"][~solver_bins.loc[s.index, "AV_Lightness"].isna()].astype(float),
            ),
            "Charge/ Paket": lambda lists: sum(len(l) for l in lists)
        })
        .rename(columns={
            "Material_Display": "Material",
            "Menge": "Total_Stock",
            "AV_LB%": "Weighted_LB%",
            "AV_BulkDensity": "Weighted_BulkDensity",
            "AV_Lightness": "Weighted_Lightness",
            "Charge/ Paket": "BB_Count"
        })
    )

    group_order_map = {
        "Clear Neutral": 0,
        "Clearlightblue Low": 1,
        "Clearlightblue High": 2,
        "Lightgrey": 3,
        "Lightblue": 4
    }
    qc["GroupOrder"] = qc["Material"].map(group_order_map).fillna(99).astype(int)
    qc["DynOrder"] = qc["dyn_bin"].astype(str).str.extract(r"(\d+)")[0].astype(float).fillna(999).astype(int)
    qc = qc.sort_values(["GroupOrder", "DynOrder", "Material", "dyn_bin"]).reset_index(drop=True)

    tot = float(qc["Total_Stock"].sum())
    wlb = float((qc["Total_Stock"] * qc["Weighted_LB%"]).sum() / tot) if tot else 0.0
    BBs = int(qc["BB_Count"].sum())
    wbd_tot = float((qc["Total_Stock"] * qc["Weighted_BulkDensity"].fillna(0)).sum() / tot) if tot and not qc["Weighted_BulkDensity"].isna().all() else np.nan
    wlt_tot = float((qc["Total_Stock"] * qc["Weighted_Lightness"].fillna(0)).sum() / tot) if tot and not qc["Weighted_Lightness"].isna().all() else np.nan

    total_row = pd.DataFrame({
        "Material": ["Total"], "dyn_bin": [""], "Total_Stock": [tot],
        "Weighted_LB%": [wlb], "Weighted_BulkDensity": [wbd_tot], "Weighted_Lightness": [wlt_tot],
        "BB_Count": [BBs], "GroupOrder": [99], "DynOrder": [999]
    })
    qc2 = pd.concat([qc, total_row], ignore_index=True)

    mm = solver_bins.copy()
    def _fmt_ranges(p):
        return ", ".join(find_ranges(p)) if isinstance(p, (list, tuple)) else ""

    mm["Charge/ Paket"] = mm["Charge/ Paket"].apply(_fmt_ranges)
    mm["Material"] = mm["Material_Display"]
    mm["GroupOrder"] = mm["Material"].map(group_order_map).fillna(99).astype(int)
    mm["DynOrder"] = mm["dyn_bin"].astype(str).str.extract(r"(\d+)")[0].astype(float).fillna(999).astype(int)
    mm = mm.sort_values(["GroupOrder", "DynOrder", "Material", "dyn_bin", "AV_LB%"], ignore_index=True)
    mm = mm[[
        "Material", "dyn_bin", "Beschreibung", "Artikel", "Lagerplatz", "AV_LB%",
        "AV_BulkDensity", "AV_Lightness", "Menge", "Quality", "Charge/ Paket",
        "CLB_LOW_anchor_start", "CLB_LOW_anchor_end", "CLB_LOW_in_anchor",
        "CLB_HIGH_anchor_start", "CLB_HIGH_anchor_end", "CLB_HIGH_in_anchor",
        "LG_anchor_start", "LG_anchor_end", "LG_in_anchor",
        "SOLVER_EXCLUDE", "SOLVER_EXCLUDE_REASON", "LB_is_overflow"
    ]]

    def round_for_display(df):
        for col in ["AV_LB%", "Weighted_LB%"]:
            if col in df.columns:
                df[col] = df[col].clip(upper=99).round(1)
        for col in ["AV_Lightness", "Weighted_Lightness"]:
            if col in df.columns:
                df[col] = df[col].round(1)
        for col in ["AV_BulkDensity", "Weighted_BulkDensity"]:
            if col in df.columns:
                df[col] = df[col].round(0)
        return df

    qc2 = round_for_display(qc2)
    mm = round_for_display(mm)
    candidate_recipe_display = round_for_display(candidate_recipe.copy()) if not candidate_recipe.empty else pd.DataFrame()

    output_file = f"{recipe_date.isoformat()}_{choice}_Rezept_{PRODUCTION_MODE}.xlsx"
    with pd.ExcelWriter(output_file, engine="xlsxwriter", engine_kwargs={"options": {"nan_inf_to_errors": True}}) as writer:
        wb = writer.book

        fmt_header_small = wb.add_format({"bold": False, "font_size": 9, "align": "left", "valign": "vcenter", "border": 1, "border_color": "#000000"})
        fmt_header = wb.add_format({"bold": True, "font_size": 14, "align": "left", "valign": "vcenter", "border": 1, "border_color": "#000000"})
        fmt_table_header = wb.add_format({"bold": True, "font_size": 14, "align": "center", "valign": "vcenter", "border": 1, "border_color": "#000000", "bg_color": "#DCE6F1"})
        fmt_section = wb.add_format({"border": 2, "border_color": "#000000", "bold": True, "font_size": 14, "align": "center", "valign": "vcenter"})
        fmt_center_light = wb.add_format({"border": 1, "border_color": "#D9D9D9", "font_size": 12, "align": "center", "valign": "vcenter"})
        fmt_packet_light = wb.add_format({"border": 1, "border_color": "#D9D9D9", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})
        fmt_bt_center = wb.add_format({"top": 2, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "center", "valign": "vcenter"})
        fmt_bm_center = wb.add_format({"top": 0, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "center", "valign": "vcenter"})
        fmt_bb_center = wb.add_format({"top": 0, "bottom": 2, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "center", "valign": "vcenter"})
        fmt_bt_packet = wb.add_format({"top": 2, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})
        fmt_bm_packet = wb.add_format({"top": 0, "bottom": 0, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})
        fmt_bb_packet = wb.add_format({"top": 0, "bottom": 2, "left": 2, "right": 2, "border_color": "#000000", "font_size": 12, "align": "left", "valign": "vcenter", "text_wrap": True})

        ws1 = wb.add_worksheet("Candidate_Recipe")
        writer.sheets["Candidate_Recipe"] = ws1

        ws1.set_column(0, 0, 25)
        ws1.set_column(1, 1, 12)
        ws1.set_column(2, 2, None, fmt_center_light, {"hidden": True})
        ws1.set_column(3, 3, None, fmt_center_light, {"hidden": True})
        ws1.set_column(4, 4, 18, fmt_center_light)
        ws1.set_column(5, 8, None, fmt_center_light, {"hidden": True})
        ws1.set_column(9, 9, 90, fmt_packet_light)
        ws1.set_column(10, 10, 10, fmt_center_light)

        ws1.write(0, 0, header_stock, fmt_header_small)
        ws1.write(1, 0, header_corr, fmt_header)

        table_header_row = 2
        cols = ["Material_Vis", "Big Bags", "Percentage", "Artikel", "Lagerplatz", "AV_LB%", "AV_Lightness", "Quantity", "Quality", "Charge/ Paket", "Pick_Seq"]
        for j, col_ in enumerate(cols):
            hdr = f"{col_} ({recipe_date.isoformat()}, {choice}, {PRODUCTION_MODE})" if col_ == "Charge/ Paket" else col_
            ws1.write(table_header_row, j, hdr, fmt_table_header)

        if not candidate_recipe_display.empty:
            cr = candidate_recipe_display.copy()
            cr["Pick_Seq"] = cr.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1
            if "Quantity" not in cr.columns:
                cr["Quantity"] = cr.get("Tonnage_Allocated", cr.get("Menge", 0.0))

            for c in cols:
                if c not in cr.columns:
                    cr[c] = 0.0 if c in ("Big Bags", "AV_LB%", "AV_Lightness", "Quantity", "Pick_Seq", "Percentage") else ""
                else:
                    if pd.api.types.is_categorical_dtype(cr[c]):
                        cr[c] = cr[c].astype(str)

            df_merge = cr[cols].reset_index(drop=True).copy()
            df_merge["LB_is_overflow"] = cr["LB_is_overflow"].values if "LB_is_overflow" in cr.columns else False

            def _reorder_block(block):
                overflow = block[block["LB_is_overflow"]].copy()
                core = block[~block["LB_is_overflow"]].copy()
                mat_name = str(block["Material_Vis"].iloc[0]) if "Material_Vis" in block.columns and len(block) > 0 else ""
                desc_order = (mat_name == "Clearlightblue High")
                if core.empty:
                    return block.sort_values("AV_LB%", ascending=(not desc_order)) if "AV_LB%" in block.columns else block
                if "AV_LB%" in core.columns:
                    core = core.sort_values("AV_LB%", ascending=(not desc_order))
                return pd.concat([overflow, core], axis=0)

            df_merge = (
                df_merge.groupby("Material_Vis", group_keys=False, sort=False, observed=False)
                        .apply(_reorder_block)
                        .reset_index(drop=True)
            )

            if pd.api.types.is_categorical_dtype(df_merge["Material_Vis"]):
                df_merge["__mat_order"] = df_merge["Material_Vis"].cat.codes
            else:
                df_merge["__mat_order"] = df_merge["Material_Vis"].map({m: i for i, m in enumerate(["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"])}).fillna(999).astype(int)

            df_merge["__row_order"] = np.arange(len(df_merge), dtype=int)
            df_merge = (df_merge.sort_values(["__mat_order", "__row_order"], kind="stable")
                               .drop(columns=["__mat_order", "__row_order"])
                               .reset_index(drop=True))

            df_merge["Pick_Seq"] = df_merge.groupby("Material_Vis", sort=False, observed=False).cumcount() + 1

            merges = []
            curr_mat, start, end, prev_bb, prev_perc = None, None, None, None, None
            for i, row in df_merge.iterrows():
                mat = row["Material_Vis"]
                r = table_header_row + 1 + i
                if mat != curr_mat:
                    if curr_mat is not None:
                        merges.append((start, end, curr_mat, prev_bb, prev_perc))
                    curr_mat = mat
                    start = r
                    prev_bb = row["Big Bags"]
                    prev_perc = row["Percentage"]
                end = r
            if curr_mat is not None:
                merges.append((start, end, curr_mat, prev_bb, prev_perc))

            mat_has_non_overflow = df_merge.groupby("Material_Vis", observed=False)["LB_is_overflow"].apply(lambda s: (~s).any()).to_dict()

            for s, e, mat, bb, perc in merges:
                if s < e:
                    ws1.merge_range(s, 0, e, 0, mat, fmt_section)
                    ws1.merge_range(s, 1, e, 1, bb, fmt_section)
                    ws1.merge_range(s, 2, e, 2, perc, fmt_section)
                else:
                    ws1.write(s, 0, mat, fmt_section)
                    ws1.write(s, 1, bb, fmt_section)
                    ws1.write(s, 2, perc, fmt_section)

            for i, row in df_merge.iterrows():
                r = table_header_row + 1 + i
                mat = row["Material_Vis"]
                s, e, _, _, _ = next((b for b in merges if b[0] <= r <= b[1]), (r, r, mat, row.get("Big Bags", None), row.get("Percentage", None)))
                if s == e:
                    f4 = f8 = f9 = fmt_section
                else:
                    if r == s:
                        f4, f8, f9 = fmt_bt_center, fmt_bt_packet, fmt_bt_center
                    elif r == e:
                        f4, f8, f9 = fmt_bb_center, fmt_bb_packet, fmt_bb_center
                    else:
                        f4, f8, f9 = fmt_bm_center, fmt_bm_packet, fmt_bm_center

                ws1.write(r, 3, row["Artikel"], fmt_center_light)
                ws1.write(r, 4, row["Lagerplatz"], f4)
                ws1.write(r, 5, row["AV_LB%"], fmt_center_light)
                ws1.write(r, 6, row["AV_Lightness"], fmt_center_light)
                ws1.write(r, 7, row["Quantity"], fmt_center_light)
                ws1.write(r, 8, row["Quality"], fmt_center_light)
                ws1.write(r, 9, row["Charge/ Paket"], f8)
                ws1.write(r, 10, int(row["Pick_Seq"]) if pd.notna(row["Pick_Seq"]) else "", f9)

                if bool(row.get("LB_is_overflow", False)) and mat_has_non_overflow.get(mat, False):
                    ws1.set_row(r, None, None, {"hidden": True})

            mats_to_hide = set(hidden_mats) | set(unused_mats_for_hide)
            for s, e, mat, _, _ in merges:
                if str(mat) in mats_to_hide:
                    for rr in range(s, e + 1):
                        ws1.set_row(rr, None, None, {"hidden": True})
        else:
            ws1.write(table_header_row + 1, 0, "No valid candidate recipe could be generated.", fmt_header)

        ws2 = wb.add_worksheet("Stock Summary")
        writer.sheets["Stock Summary"] = ws2
        ws2.write(0, 0, f"QC Approved Stock Summary (Post-Move Bins) – Modus {PRODUCTION_MODE}", wb.add_format({"bold": True, "font_size": 14}))

        qc2.to_excel(writer, sheet_name="Stock Summary", startrow=1,
                     columns=["Material", "dyn_bin", "Total_Stock", "Weighted_LB%", "Weighted_BulkDensity", "Weighted_Lightness", "BB_Count"],
                     index=False)

        start = len(qc2) + 4
        ws2.write(start - 1, 0, "Detailed Material Stock (Post-Move Bins)", wb.add_format({"bold": True, "font_size": 14}))
        mm.to_excel(writer, sheet_name="Stock Summary", startrow=start, index=False)

        ws3 = wb.add_worksheet("Supply_Chain")
        writer.sheets["Supply_Chain"] = ws3
        if supply_chain_df is not None and (not supply_chain_df.empty):
            supp = supply_chain_df.copy()
            order_vis = ["Clear Neutral", "Clearlightblue Low", "Clearlightblue High", "Lightgrey", "Lightblue"]
            supp["MatOrder"] = supp["Material"].map({m: i for i, m in enumerate(order_vis)})
            supp.sort_values(["Mixture", "MatOrder", "PacketIndex"], inplace=True)
            supp.drop(columns=["MatOrder", "PacketIndex"], errors="ignore", inplace=True)
            supp.to_excel(writer, sheet_name="Supply_Chain", startrow=0, index=False)
            rows_sc, cols_sc = supp.shape
            ws3.add_table(0, 0, rows_sc, cols_sc - 1, {"columns": [{"header": h} for h in supp.columns], "name": "SupplyChainTable"})

    files.download(output_file)
    print("Done.")

# =========================================================
# UI: button + wiring (ONLY Date, Product, Mode)
# =========================================================
generate_button = widgets.Button(
    description="Generate recipe",
    button_style="success",
    tooltip="Generate recipe with current settings"
)

def on_generate_clicked(b):
    clear_output()
    display(date_picker, product_dd, mode_dd, generate_button)
    generate_recipe()

generate_button.on_click(on_generate_clicked)

display(date_picker, product_dd, mode_dd, generate_button)

Text(value='2026-03-12', description='Recipe Date:', style=DescriptionStyle(description_width='initial'))

Dropdown(description='Product:', options=(('C100', 'C100'), ('C102', 'C102'), ('C103', 'C103')), style=Descrip…

Dropdown(description='Production Mode:', options=(('Standard', 'STANDARD'), ('RecyClass', 'RECYCLASS'), ('A1 o…

Button(button_style='success', description='Generate recipe', style=ButtonStyle(), tooltip='Generate recipe wi…